## Environment Setup

In [ ]:
%%configure
{
"conf": {
"spark.dynamicAllocation.enabled" : "true",
"spark.dynamicAllocation.minExecutors" : 4,
"spark.dynamicAllocation.maxExecutors" : 9
}
}

In [ ]:
library(truveta.notebook.study)
library(arrow)
library(tidyverse)
library(glmnet)
library(mice)
library(survival)
library(ggsurvfit)
library(survminer)
library(PSweight)
library(ggplot2)
library(survey)
library(cobalt)
library(Matrix)
library(parallel)
library(data.table)
library(foreach)

In [ ]:
%%sparkr
con <- create_connection()
study <- get_study(con)

In [ ]:
# get file system reference
artifacts_path_local <- get_artifacts_path(con, study, fs = TRUE)
artifacts_images_dir = file.path(artifacts_path_local, "study_images")
# create directory if not exists
if (!dir.exists(artifacts_images_dir)) {
  message("Creating directory: ", artifacts_images_dir)
  dir.create(artifacts_images_dir)
}

## Install Packages

In [ ]:
packages <- c('glmnet')
install_packages(con, packages, quiet=TRUE)

## Upload saved files

In [ ]:
artifacts_path <- get_artifacts_path(con, study, fs=TRUE)
dir_path <- file.path(artifacts_path, "mice")
files <- list.files(dir_path, full.names=TRUE)
print(files)

In [ ]:
imputed_t2d <- readRDS(files[7])

t2d_outcome <- as.data.frame(
  load_artifacts_data(
    con,
    study,
    file.path("/aim_1b/Primary_Endpoint_t2d_primary_all_sex_population")
  )
)

outcome_cols <- c(
  "PersonId", "index_date",
  "Outcome", "Survival_Time", "Event_Time", "Event_Name",
  "Death_Time", "Event_or_Censor_Date",
  "Outcome_Text", "AgeGroup", "DcsiScoreGroup"
)

t2d_outcome_sub <- t2d_outcome %>%
  dplyr::select(dplyr::all_of(outcome_cols))

imputed_t2d_list <- vector("list", 5)

for (i in 1:5) {
  imputed_t2d_list[[i]] <- mice::complete(imputed_t2d, i) %>%
    dplyr::left_join(
      t2d_outcome_sub,
      by = c("PersonId", "FirstMedTime" = "index_date")
    )
}

lapply(imputed_t2d_list, dim)

In [ ]:
for (i in 1:5) {
  imputed_t2d_list[[i]] <- imputed_t2d_list[[i]] %>%
    filter(Outcome_Text != "Invalid")
}

lapply(imputed_t2d_list, dim)

# T2D - Secondary

## OVERALL

In [ ]:
setDTthreads(0)

prepare_overall_variables <- function(imp_data, covariates, treatment) {

  cat("\n=== Preparing Variables (Overall, First Imputation) ===\n")

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, treatment)
  dt <- dt[, ..needed_cols]
  dt[, Z := as.integer(get(treatment) == "glp")]

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, "Z")
  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  X_df <- dt[1:min(1000, nrow(dt)), ..covariates]
  X    <- sparse.model.matrix(~ . - 1, data = X_df)

  union_vars <- colnames(X)

  cat(sprintf("  Original covariates  : %d\n", length(covariates)))
  cat(sprintf("  After dummy expansion: %d variables\n", length(union_vars)))
  cat("  All main effects included (no subgroup interactions for overall)\n")

  return(list(union_vars = union_vars))
}

analyze_overall <- function(imp_data, imp_num,
                            covariates, treatment,
                            union_vars, tau = 1095) {

  cat(sprintf("\n=== Processing Imputation %d ===\n", imp_num))

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, "Z")
  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)
  sample_per_group   <- ceiling(target_sample_size / 2)

  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = Z]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Sample: %d obs\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample    <- sparse.model.matrix(~ . - 1, data = X_df_sample)

  selected_cols          <- intersect(union_vars, colnames(X_sample))
  design_selected_sample <- X_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d x %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model (ridge)...\n")
  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family      = "binomial",
    alpha       = 0,
    lambda      = 0.01,
    standardize = TRUE,
    thresh      = 1e-2,
    maxit       = 1000
  )

  cat("  Predicting on full cohort...\n")
  chunk_size <- 200000
  n_chunks   <- ceiling(n_total / chunk_size)
  ps_all     <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx   <- min(chunk_i * chunk_size, n_total)

    dt_chunk     <- dt[start_idx:end_idx]
    X_df_chunk   <- dt_chunk[, ..covariates]
    X_chunk      <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    design_chunk <- X_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, design_chunk); invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  # Overlap weights, normalized within treatment group
  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = Z]

  data <- dt[, .(Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data))
}

In [ ]:
# ==============================================================================
# Cox Regression (Unadjusted + OW-Adjusted)
# ==============================================================================
complete_analysis_overall <- function(data_list, imp_num, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  cat(sprintf("  Cox models (unadjusted + OW-adjusted)...\n"))

  # 1. Unadjusted
  cox_unadj <- coxph(
    Surv(Survival_Time, Outcome) ~ Z,
    data    = data,
    robust  = TRUE,
    method  = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )
  unadj_s <- summary(cox_unadj)
  unadj_effects <- list(
    treatment_coef   = unadj_s$coefficients["Z", "coef"],
    treatment_se     = unadj_s$coefficients["Z", "robust se"],
    treatment_hr     = unadj_s$coefficients["Z", "exp(coef)"],
    treatment_pvalue = unadj_s$coefficients["Z", "Pr(>|z|)"]
  )

  # 2. OW-Adjusted
  cox_adj <- coxph(
    Surv(Survival_Time, Outcome) ~ Z,
    data    = data,
    weights = w,
    robust  = TRUE,
    method  = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )
  adj_s <- summary(cox_adj)
  adj_effects <- list(
    treatment_coef   = adj_s$coefficients["Z", "coef"],
    treatment_se     = adj_s$coefficients["Z", "robust se"],
    treatment_hr     = adj_s$coefficients["Z", "exp(coef)"],
    treatment_pvalue = adj_s$coefficients["Z", "Pr(>|z|)"]
  )

  return(list(
    unadj_effects = unadj_effects,
    adj_effects   = adj_effects,
    data          = data
  ))
}

# ==============================================================================
# Rubin's Rules Pooling
# ==============================================================================
pool_results_overall <- function(all_results) {

  m <- length(all_results)

  pool_one <- function(coefs, ses) {
    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B     <- var(coefs)
    T_var <- U_bar + (1 + 1/m) * B
    se    <- sqrt(T_var)
    df    <- if(B < 1e-10) Inf else (m - 1) * (1 + U_bar / ((1 + 1/m) * B))^2
    list(
      coef     = Q_bar,
      se       = se,
      hr       = exp(Q_bar),
      ci_lower = exp(Q_bar - qt(0.975, df) * se),
      ci_upper = exp(Q_bar + qt(0.975, df) * se),
      pvalue   = 2 * pt(-abs(Q_bar / se), df = df),
      df       = df
    )
  }

  unadj <- pool_one(
    sapply(all_results, function(x) x$unadj_effects$treatment_coef),
    sapply(all_results, function(x) x$unadj_effects$treatment_se)
  )
  adj <- pool_one(
    sapply(all_results, function(x) x$adj_effects$treatment_coef),
    sapply(all_results, function(x) x$adj_effects$treatment_se)
  )

  return(list(unadjusted = unadj, adjusted = adj))
}

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders",
  "HasChronicHypertensionWithoutComplications","HasChronicHypertensionWithComplications",
  "HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism",
  "HasRenalFailure","HasLiverDisease","HasPepticUlcerDisease","HasHivAids",
  "HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30",
  "HasWeightLossAndMalnutrition","HasFluidAndElectrolyteDisorders",
  "HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking",
  "HasDementia","HasHyperlipidemia","HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke","HasOsteoarthritis","HasObstructiveSleepApnea",
  "HasNafld","HasPolycysticOvarianSyndrome","HasSchizophrenia",
  "HasTraumaticBrainInjury","HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin","HasOtherLLT","HasAntiplatelet","HasAnticoagulant",
  "HasInsulin","HasMetformin","HasAceorArb","HasFinerenone","HasOtherAntiHypertensive",
  "HasHospitalization","Sex","Ethnicity","Race","CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction",
  "HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory",
  "HealthManagementNonMotivationRiskCategory",
  "NumEncounters","NumHospitalization","NumOutpatientVisits","NumEmergencyVisits",
  "NumAntidiabeticMed","DcsiScore","ElixhauserComorbidityScore","Age","CalendarYear",
  "AltValue","AstValue","BmiValue","DiastolicBloodPressureValue","EGfrValue",
  "HemoglobinValue","LdlCholesterolValue","HdlCholesterolValue",
  "SerumCreatinineValue","SerumTriglyceridesValue","SystolicBloodPressureValue",
  "TotalCholesterolValue","WhiteBloodCellCountValue","HbA1cValue",
  "AddressChangeCountLast6Months","AddressChangeCountLast12Months",
  "HouseholdMotorizedPropertyRegistrationsCount","AddressChangeCountLast60Months",
  "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt","CurrAddrLenOfRes",
  "HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel",
  "AddressChangeCountLast3Months","AddressChangeCountLast1Month"
)
treatment <- "FirstMedGroup"

start_time <- Sys.time()

# PHASE 1: Variable Preparation
cat("\n", rep("=", 80), "\n")
cat("PHASE 1: Variable Preparation (All Main Effects)\n")
cat(rep("=", 80), "\n")

prep_result <- prepare_overall_variables(
  imp_data   = imputed_t2d_list[[1]],
  covariates = covariates,
  treatment  = treatment
)
union_vars <- prep_result$union_vars

cat(sprintf("\nTotal variables: %d\n", length(union_vars)))

# PHASE 2: Analysis across all imputations
cat("\n", rep("=", 80), "\n")
cat("PHASE 2: Full Analysis (All 5 Imputations)\n")
cat(rep("=", 80), "\n")

all_results <- list()
for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_overall(
    imp_data   = imputed_t2d_list[[i]],
    imp_num    = i,
    covariates = covariates,
    treatment  = treatment,
    union_vars = union_vars,
    tau        = 1095
  )

  cox_results <- complete_analysis_overall(data_list, imp_num = i, tau = 1095)

  all_results[[i]] <- cox_results
  invisible(gc())
}

# PHASE 3: Pooling
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Rubin's Rules Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results_overall(all_results)

end_time <- Sys.time()
elapsed  <- as.numeric(end_time - start_time, units = "mins")
cat(sprintf("\nTotal Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))

# Display Results
cat("\n", rep("=", 80), "\n")
cat("FINAL POOLED RESULTS - Overall Treatment Effect\n")
cat(rep("=", 80), "\n\n")

res_unadj <- pooled_results$unadjusted
res_adj   <- pooled_results$adjusted

cat("1. UNADJUSTED\n")
cat(rep("-", 80), "\n")
cat(sprintf("HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
            res_unadj$hr, res_unadj$ci_lower, res_unadj$ci_upper, res_unadj$pvalue))
if(res_unadj$df < Inf) cat(sprintf("df = %.1f\n", res_unadj$df))

cat("\n2. OW-ADJUSTED (Overlap Weighted Cox)\n")
cat(rep("-", 80), "\n")
cat(sprintf("HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
            res_adj$hr, res_adj$ci_lower, res_adj$ci_upper, res_adj$pvalue))
if(res_adj$df < Inf) cat(sprintf("df = %.1f\n", res_adj$df))

cat("\n3. COMPARISON\n")
cat(rep("-", 80), "\n")
cat(sprintf("Unadjusted HR : %.3f\n", res_unadj$hr))
cat(sprintf("Adjusted HR   : %.3f\n", res_adj$hr))
cat(sprintf("Change        : %.3f (%.1f%%)\n",
            res_adj$hr - res_unadj$hr,
            (res_adj$hr - res_unadj$hr) / res_unadj$hr * 100))

cat("\n", rep("=", 80), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 80), "\n")

## SEX

In [ ]:
setDTthreads(0)

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
perform_lasso_first_only <- function(imp_data, covariates, subgroup_var, treatment,
                                     sample_size = 50000) {

  cat("\n=== LASSO (First Imputation Only, Stratified Sampling) ===\n")

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment)
  dt <- dt[, ..needed_cols]

  dt[, Si := factor(get(subgroup_var), levels = c("1065405", "1065406"))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(sample_size / n_groups)

  cat(sprintf("  Total groups (Si × Z): %d\n", n_groups))
  cat(sprintf("  Target per group: %d\n", sample_per_group))

  # Stratified sampling by Si × Z
  set.seed(12345)
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Actual sample: %d obs\n", nrow(dt_sample)))

  factor_cols <- names(dt_sample)[sapply(dt_sample, is.factor) | sapply(dt_sample, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt_sample[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  # Sparse matrix
  X_df <- dt_sample[, ..covariates]
  X <- sparse.model.matrix(~ . - 1, data = X_df)
  Si <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  n_subgroups <- ncol(Si)

  cat("  Creating interactions...\n")

  Xi_Si_list <- lapply(1:n_subgroups, function(k) {
    si_k <- as.vector(Si[, k])
    X_k <- X * si_k
    colnames(X_k) <- paste0(colnames(X), ":", colnames(Si)[k])
    X_k
  })

  Xi_Si <- do.call(cbind, Xi_Si_list)
  rm(Xi_Si_list); invisible(gc())

  design_matrix <- cbind(X, Si, Xi_Si)

  # Penalty factor
  penalty.factor <- c(
    rep(0, ncol(X) + ncol(Si)),
    rep(1, ncol(Xi_Si))
  )

  cat("  Running LASSO...\n")

  # LASSO
  lasso_fit <- cv.glmnet(
    design_matrix, dt_sample$Z,
    family = "binomial",
    alpha = 1,
    penalty.factor = penalty.factor,
    nfolds = 3,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 2000
  )

  selected_coef <- coef(lasso_fit, s = "lambda.min")
  selected_vars <- rownames(selected_coef)[selected_coef[, 1] != 0]
  selected_vars <- selected_vars[selected_vars != "(Intercept)"]
  selected_interactions <- selected_vars[grepl(":", selected_vars)]

  cat(sprintf("  Selected %d interactions\n", length(selected_interactions)))

  # Base variables + selected interactions
  union_vars <- c(colnames(X), colnames(Si), selected_interactions)

  return(list(
    union_vars = union_vars,
    selected_interactions = selected_interactions,
    base_vars = c(colnames(X), colnames(Si))
  ))
}

# ==============================================================================
# Propensity Score Estimation with Selected Variables
# ==============================================================================
analyze_with_selected_vars <- function(imp_data, imp_num,
                                       covariates, subgroup_var, treatment,
                                       union_vars, tau = 1095) {

  cat(sprintf("\n=== Processing Imputation %d ===\n", imp_num))

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]

  dt[, Si := factor(get(subgroup_var), levels = c("1065405", "1065406"))]
# dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(target_sample_size / n_groups)

  cat(sprintf("  Target sample: %d (stratified by Si × Z)\n", target_sample_size))

  # Stratified sampling
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]

  cat(sprintf("  Actual sample: %d\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample <- sparse.model.matrix(~ . - 1, data = X_df_sample)
  Si_sample <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  interaction_names <- union_vars[grepl(":", union_vars)]

  if(length(interaction_names) > 0 && length(interaction_names) < 800) {
    cat(sprintf("  Creating %d interactions (sample)...\n", length(interaction_names)))

    n_subgroups <- ncol(Si_sample)
    Xi_Si_list <- list()

    for(k in 1:n_subgroups) {
      si_k <- as.vector(Si_sample[, k])
      X_k <- X_sample * si_k
      colnames(X_k) <- paste0(colnames(X_sample), ":", colnames(Si_sample)[k])

      selected_cols <- intersect(colnames(X_k), interaction_names)
      if(length(selected_cols) > 0) {
        Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols, drop = FALSE]
      }
    }

    if(length(Xi_Si_list) > 0) {
      Xi_Si_sample <- do.call(cbind, Xi_Si_list)
      design_sample <- cbind(X_sample, Si_sample, Xi_Si_sample)
      rm(Xi_Si_list); invisible(gc())
    } else {
      design_sample <- cbind(X_sample, Si_sample)
    }
  } else {
    cat("  Too many interactions, using main effects only\n")
    design_sample <- cbind(X_sample, Si_sample)
  }

  all_vars <- colnames(design_sample)
  selected_cols <- intersect(union_vars, all_vars)
  design_selected_sample <- design_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d × %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model...\n")

  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family = "binomial",
    alpha = 0,
    lambda = 0.01,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 1000
  )

  cat("  Predicting on full data...\n")

  chunk_size <- 200000
  n_chunks <- ceiling(n_total / chunk_size)
  ps_all <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) {
      cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))
    }

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx <- min(chunk_i * chunk_size, n_total)

    dt_chunk <- dt[start_idx:end_idx]

    # Design matrix for chunk
    X_df_chunk <- dt_chunk[, ..covariates]
    X_chunk <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    Si_chunk <- sparse.model.matrix(~ Si - 1, data = dt_chunk)

    if(length(interaction_names) > 0 && length(interaction_names) < 800) {
      n_subgroups <- ncol(Si_chunk)
      Xi_Si_list <- list()

      for(k in 1:n_subgroups) {
        si_k <- as.vector(Si_chunk[, k])
        X_k <- X_chunk * si_k
        colnames(X_k) <- paste0(colnames(X_chunk), ":", colnames(Si_chunk)[k])

        selected_cols_chunk <- intersect(colnames(X_k), interaction_names)
        if(length(selected_cols_chunk) > 0) {
          Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols_chunk, drop = FALSE]
        }
      }

      if(length(Xi_Si_list) > 0) {
        Xi_Si_chunk <- do.call(cbind, Xi_Si_list)
        design_chunk <- cbind(X_chunk, Si_chunk, Xi_Si_chunk)
        rm(Xi_Si_list)
      } else {
        design_chunk <- cbind(X_chunk, Si_chunk)
      }
    } else {
      design_chunk <- cbind(X_chunk, Si_chunk)
    }

    design_chunk <- design_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, Si_chunk, design_chunk)
    invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  # Overlap weights
  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = .(Si, Z)]

  data <- dt[, .(Si, Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data, X = NULL, Si = NULL))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis (Subgroup-Specific, Interaction)
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  cat(sprintf("  Cox models...\n"))

  cox_main <- coxph(
    Surv(Survival_Time, Outcome) ~ Z + Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  main_summary <- summary(cox_main)

  main_effects <- list(
    treatment_coef = main_summary$coefficients["Z", "coef"],
    treatment_se = main_summary$coefficients["Z", "robust se"],
    treatment_hr = main_summary$coefficients["Z", "exp(coef)"],
    treatment_pvalue = main_summary$coefficients["Z", "Pr(>|z|)"]
  )

  subgroup_coef_name <- grep("^Si", rownames(main_summary$coefficients), value = TRUE)
  if(length(subgroup_coef_name) > 0) {
    main_effects$subgroup_coef <- main_summary$coefficients[subgroup_coef_name[1], "coef"]
    main_effects$subgroup_se <- main_summary$coefficients[subgroup_coef_name[1], "robust se"]
    main_effects$subgroup_hr <- main_summary$coefficients[subgroup_coef_name[1], "exp(coef)"]
    main_effects$subgroup_pvalue <- main_summary$coefficients[subgroup_coef_name[1], "Pr(>|z|)"]
    main_effects$subgroup_var <- subgroup_var
  }

  subgroup_levels <- levels(data$Si)
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)
  interaction_coefs <- grep("Z:Si", rownames(interaction_summary$coefficients), value = TRUE)

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
# ==============================================================================
# RMST and Risk Difference Calculation
# ==============================================================================

calculate_rmst_weighted <- function(time, status, weights, tau) {

  ord <- order(time)
  time <- time[ord]
  status <- status[ord]
  weights <- weights[ord]

  w_atrisk <- rev(cumsum(rev(weights)))
  w_event  <- status * weights

  w_surv <- cumprod(1 - w_event / pmax(w_atrisk, 1e-10))
  w_surv <- c(1, w_surv)
  time_ext <- c(0, time)

  idx <- time_ext <= tau
  time_use <- time_ext[idx]
  surv_use <- w_surv[idx]

  if (max(time_use) < tau) {
    time_use <- c(time_use, tau)
    surv_use <- c(surv_use, tail(surv_use, 1))
  }

  rmst <- sum(diff(time_use) * head(surv_use, -1))
  return(rmst)
}

calculate_rmst_point <- function(results, tau = 1095, verbose = TRUE) {

  if (verbose) cat("  RMST (weighted KM, point estimates only)...\n")

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  results_list <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if (nrow(data_sg) <= 10) return(NULL)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl  <- data_sg[data_sg$Z == 0, ]

    if (nrow(data_treat) <= 10 || nrow(data_ctrl) <= 10) return(NULL)

    # Weighted Kaplan–Meier
    fit_treat <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_treat,
      weights = data_treat$w
    )
    fit_ctrl <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_ctrl,
      weights = data_ctrl$w
    )

    # RMST
    rmst_treat <- tryCatch(
      calculate_rmst_weighted(
        time = data_treat$Survival_Time,
        status = data_treat$Outcome,
        weights = data_treat$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_ctrl <- tryCatch(
      calculate_rmst_weighted(
        time = data_ctrl$Survival_Time,
        status = data_ctrl$Outcome,
        weights = data_ctrl$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_diff <- rmst_treat - rmst_ctrl

    # Risk Differences
    time_points <- c(365, 730, 1095)
    surv_at_times <- lapply(time_points, function(t) {
      sum_treat <- summary(fit_treat, times = t, extend = TRUE)
      sum_ctrl  <- summary(fit_ctrl,  times = t, extend = TRUE)

      surv_t <- if (length(sum_treat$surv) > 0) sum_treat$surv else NA_real_
      surv_c <- if (length(sum_ctrl$surv) > 0) sum_ctrl$surv else NA_real_
      se_t   <- if (length(sum_treat$std.err) > 0) sum_treat$std.err else NA_real_
      se_c   <- if (length(sum_ctrl$std.err) > 0) sum_ctrl$std.err else NA_real_

      risk_t <- 1 - surv_t
      risk_c <- 1 - surv_c
      risk_diff <- risk_c - risk_t

      risk_diff_var <- if (!is.na(se_t) && !is.na(se_c)) {
        se_t^2 + se_c^2
      } else NA_real_

      baseline_risk_var <- if (!is.na(se_c)) {
        se_c^2
      } else NA_real_

      list(
        time = t,
        surv_treat = surv_t,
        se_treat = se_t,
        surv_ctrl = surv_c,
        se_ctrl = se_c,
        risk_treat = risk_t,
        risk_ctrl = risk_c,
        risk_diff = risk_diff,
        risk_diff_var = risk_diff_var,
        baseline_risk = risk_c,
        baseline_risk_var = baseline_risk_var
      )
    })

    list(
      rmst = list(
        subgroup = sg,
        rmst_treat = rmst_treat,
        rmst_ctrl = rmst_ctrl,
        rmst_diff = rmst_diff,
        tau = tau
      ),
      surv_prob = list(
        subgroup = sg,
        time_points = surv_at_times
      )
    )
  })

  results_list <- results_list[!sapply(results_list, is.null)]

  rmst_results <- lapply(results_list, function(x) x$rmst)
  names(rmst_results) <- sapply(rmst_results, function(x) x$subgroup)

  surv_prob_results <- lapply(results_list, function(x) x$surv_prob)
  names(surv_prob_results) <- sapply(surv_prob_results, function(x) x$subgroup)

  return(list(
    rmst_results = rmst_results,
    surv_prob_results = surv_prob_results
  ))
}

bootstrap_rmst_diff <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  set.seed(seed)

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  base_est <- calculate_rmst_point(results, tau = tau, verbose = TRUE)
  rmst_point <- base_est$rmst_results

  use_parallel <- !is.null(n_cores) && n_cores > 1

  if (use_parallel) {
    if (n_cores == "auto") {
      n_cores <- max(1, parallel::detectCores() - 1)
    }

    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac,
        ", cores =", n_cores, ")...\n")

    cl <- parallel::makeCluster(n_cores)

    parallel::clusterExport(cl,
                           c("calculate_rmst_point", "calculate_rmst_weighted",
                             "data_by_sg", "tau", "subsample_frac"),
                           envir = environment())

    parallel::clusterEvalQ(cl, library(survival))

    boot_results <- parallel::parLapply(cl, 1:B, function(b) {

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- tryCatch(
        calculate_rmst_point(results_boot, tau = tau, verbose = FALSE),
        error = function(e) NULL
      )

      if (is.null(boot_est)) return(NULL)

      sapply(boot_est$rmst_results, function(x) x$rmst_diff)
    })

    parallel::stopCluster(cl)

    boot_results <- boot_results[!sapply(boot_results, is.null)]
    boot_mat_data <- do.call(rbind, boot_results)

    boot_mat <- lapply(seq_along(subgroup_levels), function(i) {
      boot_mat_data[, i]
    })
    names(boot_mat) <- subgroup_levels

  } else {
    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac, ")...\n")

    boot_mat <- lapply(subgroup_levels, function(sg) {
      numeric(B)
    })
    names(boot_mat) <- subgroup_levels

    for (b in seq_len(B)) {
      if (b %% 20 == 0) cat("    Iteration", b, "of", B, "\n")

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- calculate_rmst_point(results_boot, tau = tau, verbose = FALSE)
      rmst_boot <- boot_est$rmst_results

      for (sg in names(rmst_boot)) {
        boot_mat[[sg]][b] <- rmst_boot[[sg]]$rmst_diff
      }
    }
  }

  out <- lapply(rmst_point, function(x) {
    sg <- x$subgroup
    diffs <- boot_mat[[sg]]
    diffs <- diffs[is.finite(diffs)]

    if (length(diffs) < 10) {
      se <- NA_real_
      ci_lower <- NA_real_
      ci_upper <- NA_real_
    } else {
      se <- sd(diffs)
      ci_lower <- quantile(diffs, 0.025, na.rm = TRUE)
      ci_upper <- quantile(diffs, 0.975, na.rm = TRUE)
    }

    c(x, list(
      se_rmst_diff = se,
      ci_lower = ci_lower,
      ci_upper = ci_upper,
      B = length(diffs),
      subsample_frac = subsample_frac
    ))
  })
  names(out) <- names(rmst_point)

  return(list(
    rmst_results = out,
    surv_prob_results = base_est$surv_prob_results
  ))
}

calculate_rmst_fast <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  cat("\n=== RMST with Subsample Bootstrap ===\n")
  cat("References:\n")
  cat("  - Zeng et al. (2022) Stat Med [bootstrap SE for weighted estimates]\n")
  cat("  - adjustedCurves::adjusted_rmst [RMST bootstrap implementation]\n\n")

  bootstrap_rmst_diff(
    results = results,
    tau = tau,
    B = B,
    subsample_frac = subsample_frac,
    seed = seed,
    n_cores = n_cores
  )
}

In [ ]:
# ==============================================================================
# Rubin's Pooling Across Multiple Imputations
# ==============================================================================
pool_results <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  m <- length(all_results)

  treat_coefs <- sapply(all_results, function(x) x$main_effects$treatment_coef)
  treat_ses <- sapply(all_results, function(x) x$main_effects$treatment_se)

  Q_bar_treat <- mean(treat_coefs)
  U_bar_treat <- mean(treat_ses^2)
  B_treat <- var(treat_coefs)
  T_treat <- U_bar_treat + (1 + 1/m) * B_treat
  se_treat <- sqrt(T_treat)

  df_treat <- if(B_treat < 1e-10) Inf else {
    (m - 1) * (1 + U_bar_treat / ((1 + 1/m) * B_treat))^2
  }

  pooled_main_effects <- list(
    treatment = list(
      coef = Q_bar_treat,
      se = se_treat,
      hr = exp(Q_bar_treat),
      ci_lower = exp(Q_bar_treat - qt(0.975, df_treat) * se_treat),
      ci_upper = exp(Q_bar_treat + qt(0.975, df_treat) * se_treat),
      pvalue = 2 * pt(-abs(Q_bar_treat / se_treat), df = df_treat),
      df = df_treat
    )
  )

  if(!is.null(all_results[[1]]$main_effects$subgroup_coef)) {
    subgroup_coefs <- sapply(all_results, function(x) x$main_effects$subgroup_coef)
    subgroup_ses <- sapply(all_results, function(x) x$main_effects$subgroup_se)

    Q_bar_subgroup <- mean(subgroup_coefs)
    U_bar_subgroup <- mean(subgroup_ses^2)
    B_subgroup <- var(subgroup_coefs)
    T_subgroup <- U_bar_subgroup + (1 + 1/m) * B_subgroup
    se_subgroup <- sqrt(T_subgroup)

    df_subgroup <- if(B_subgroup < 1e-10) Inf else {
      (m - 1) * (1 + U_bar_subgroup / ((1 + 1/m) * B_subgroup))^2
    }

    pooled_main_effects$subgroup <- list(
      coef = Q_bar_subgroup,
      se = se_subgroup,
      hr = exp(Q_bar_subgroup),
      ci_lower = exp(Q_bar_subgroup - qt(0.975, df_subgroup) * se_subgroup),
      ci_upper = exp(Q_bar_subgroup + qt(0.975, df_subgroup) * se_subgroup),
      pvalue = 2 * pt(-abs(Q_bar_subgroup / se_subgroup), df = df_subgroup),
      df = df_subgroup,
      var_name = all_results[[1]]$main_effects$subgroup_var
    )
  }

  pooled_subgroups <- lapply(subgroup_names, function(sg) {
    coefs <- sapply(all_results, function(x) x$subgroup_results[[sg]]$coef)
    ses <- sapply(all_results, function(x) x$subgroup_results[[sg]]$se_robust)

    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m - 1) * (1 + U_bar / ((1 + 1/m) * B))^2

    list(
      subgroup = sg,
      coef = Q_bar,
      se = se_pooled,
      hr = exp(Q_bar),
      ci_lower = exp(Q_bar - qt(0.975, df) * se_pooled),
      ci_upper = exp(Q_bar + qt(0.975, df) * se_pooled),
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })
  names(pooled_subgroups) <- subgroup_names

  interaction_terms <- sapply(all_results[[1]]$interaction_results, function(x) x$term)

  pooled_interactions <- lapply(interaction_terms, function(term) {
    coefs <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$coef else NA
    })

    ses <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$se_robust else NA
    })

    valid_idx <- !is.na(coefs) & !is.na(ses)
    coefs <- coefs[valid_idx]
    ses <- ses[valid_idx]

    if(length(coefs) == 0) return(NULL)

    m_valid <- length(coefs)
    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      term = term,
      coef = Q_bar,
      se = se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })
  pooled_interactions <- pooled_interactions[!sapply(pooled_interactions, is.null)]

  pooled_rmst <- lapply(subgroup_names, function(sg) {
    rmst_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$rmst_diff)
    se_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$se_rmst_diff)

    valid_idx <- !is.na(rmst_diffs) & !is.na(se_diffs)
    rmst_diffs <- rmst_diffs[valid_idx]
    se_diffs <- se_diffs[valid_idx]

    if(length(rmst_diffs) == 0) return(NULL)

    m_valid <- length(rmst_diffs)
    Q_bar <- mean(rmst_diffs)
    U_bar <- mean(se_diffs^2)
    B <- var(rmst_diffs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      subgroup = sg,
      rmst_diff = Q_bar,
      se = se_pooled,
      ci_lower = Q_bar - qt(0.975, df) * se_pooled,
      ci_upper = Q_bar + qt(0.975, df) * se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      tau = all_results[[1]]$rmst_results[[sg]]$tau
    )
  })
  names(pooled_rmst) <- subgroup_names
  pooled_rmst <- pooled_rmst[!sapply(pooled_rmst, is.null)]

  # ===== 5. Pool Absolute Risk Differences and Baseline Risks =====
  time_points <- c(365, 730, 1095)

  pooled_surv_probs <- lapply(subgroup_names, function(sg) {
    time_results <- lapply(seq_along(time_points), function(t_idx) {

      risk_diffs <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff else NA
      })

      risk_diff_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff_var else NA
      })

      baseline_risks <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk else NA
      })

      baseline_risk_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk_var else NA
      })

      valid_idx_rd <- !is.na(risk_diffs) & !is.na(risk_diff_vars)
      risk_diffs_valid <- risk_diffs[valid_idx_rd]
      risk_diff_vars_valid <- risk_diff_vars[valid_idx_rd]

      valid_idx_br <- !is.na(baseline_risks) & !is.na(baseline_risk_vars)
      baseline_risks_valid <- baseline_risks[valid_idx_br]
      baseline_risk_vars_valid <- baseline_risk_vars[valid_idx_br]

      if(length(risk_diffs_valid) == 0) return(NULL)

      # Rubin's Rules for Risk Difference
      m_valid_rd <- length(risk_diffs_valid)
      Q_bar_rd <- mean(risk_diffs_valid)
      U_bar_rd <- mean(risk_diff_vars_valid)
      B_rd <- var(risk_diffs_valid)
      T_rd <- U_bar_rd + (1 + 1/m_valid_rd) * B_rd
      se_pooled_rd <- sqrt(T_rd)

      df_rd <- if(B_rd < 1e-10) Inf else {
        (m_valid_rd - 1) * (1 + U_bar_rd / ((1 + 1/m_valid_rd) * B_rd))^2
      }

      ci_lower_rd <- Q_bar_rd - qt(0.975, df_rd) * se_pooled_rd
      ci_upper_rd <- Q_bar_rd + qt(0.975, df_rd) * se_pooled_rd
      pvalue_rd <- 2 * pt(-abs(Q_bar_rd / se_pooled_rd), df = df_rd)

      baseline_result <- if(length(baseline_risks_valid) > 0) {
        m_valid_br <- length(baseline_risks_valid)
        Q_bar_br <- mean(baseline_risks_valid)
        U_bar_br <- mean(baseline_risk_vars_valid)
        B_br <- var(baseline_risks_valid)
        T_br <- U_bar_br + (1 + 1/m_valid_br) * B_br
        se_pooled_br <- sqrt(T_br)

        df_br <- if(B_br < 1e-10) Inf else {
          (m_valid_br - 1) * (1 + U_bar_br / ((1 + 1/m_valid_br) * B_br))^2
        }

        ci_lower_br <- Q_bar_br - qt(0.975, df_br) * se_pooled_br
        ci_upper_br <- Q_bar_br + qt(0.975, df_br) * se_pooled_br
        pvalue_br <- 2 * pt(-abs(Q_bar_br / se_pooled_br), df = df_br)

        list(
          baseline_risk = Q_bar_br,
          baseline_risk_se = se_pooled_br,
          baseline_risk_ci_lower = ci_lower_br,
          baseline_risk_ci_upper = ci_upper_br,
          baseline_risk_df = df_br
        )
      } else {
        list(
          baseline_risk = NA,
          baseline_risk_se = NA,
          baseline_risk_ci_lower = NA,
          baseline_risk_ci_upper = NA,
          baseline_risk_df = NA
        )
      }

      c(
        list(
          time = time_points[t_idx],
          risk_diff = Q_bar_rd,
          se = se_pooled_rd,
          ci_lower = ci_lower_rd,
          ci_upper = ci_upper_rd,
          pvalue = pvalue_rd,
          df = df_rd,
          within_var = U_bar_rd,
          between_var = B_rd,
          fmi = (1 + 1/m_valid_rd) * B_rd / T_rd
        ),
        baseline_result
      )
    })

    time_results <- time_results[!sapply(time_results, is.null)]
    list(subgroup = sg, time_results = time_results)
  })
  names(pooled_surv_probs) <- subgroup_names

  return(list(
    main_effects = pooled_main_effects,
    subgroup_results = pooled_subgroups,
    interaction_results = pooled_interactions,
    rmst_results = pooled_rmst,
    surv_prob_results = pooled_surv_probs
  ))
}

In [ ]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
subgroup_var <- "Sex"
covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
  "HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
  "HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
  "HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
  "HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant",
  "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone", "HasOtherAntiHypertensive",
  "HasHospitalization", "Ethnicity", "Race", "CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
  "NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
  "NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Age", "CalendarYear",
  "AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HemoglobinValue",
  "LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
  "SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue", "HbA1cValue",
  "AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
  "AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)
covariates <- setdiff(covariates, subgroup_var)
treatment <- "FirstMedGroup"

start_time <- Sys.time()

# ==============================================================================
# PHASE 1: LASSO
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 1: LASSO\n")
cat(rep("=", 80), "\n")

lasso_result <- perform_lasso_first_only(
  imp_data = imputed_t2d_list[[1]],
  covariates = covariates,
  subgroup_var = subgroup_var,
  treatment = treatment,
  sample_size = 50000
)

union_vars <- lasso_result$union_vars

# ==============================================================================
# PHASE 2: Full Analysis
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 2: Full Analysis (All 5 Imputations - SEQUENTIAL)\n")
cat(rep("=", 80), "\n")

all_results <- list()

for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Processing Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_with_selected_vars(
    imp_data = imputed_t2d_list[[i]],
    imp_num = i,
    covariates = covariates,
    subgroup_var = subgroup_var,
    treatment = treatment,
    union_vars = union_vars,
    tau = 1095
  )

  cox_results <- complete_analysis_fast(
    data_list,
    imp_num = i,
    subgroup_var = subgroup_var,
    tau = 1095
  )

  rmst_results <- calculate_rmst_fast(
    cox_results,
    tau = 1095,
    B = 100,
    subsample_frac = 0.3,
    n_cores = 4
  )

  all_results[[i]] <- list(
    main_effects = cox_results$main_effects,
    subgroup_results = cox_results$subgroup_results,
    interaction_results = cox_results$interaction_results,
    data_by_sg = if(i == 1) cox_results$data_by_sg else NULL,
    rmst_results = rmst_results$rmst_results,
    surv_prob_results = rmst_results$surv_prob_results
  )

  cat("Imputation", i, "completed!\n")

  invisible(gc())
}

cat("\nAll imputations completed!\n")

# ==============================================================================
# PHASE 3: Pooling
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results(all_results)

end_time <- Sys.time()
elapsed <- as.numeric(end_time - start_time, units = "mins")

cat("\n", rep("=", 80), "\n")
cat(sprintf("Total Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# Display Results
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("FINAL POOLED RESULTS (3-Year Follow-up)\n")
cat(sprintf("Subgroup Analysis by: %s\n", subgroup_var))
cat(rep("=", 80), "\n\n")

# Subgroup-Specific HRs
cat("\n2. SUBGROUP-SPECIFIC TREATMENT EFFECTS (Hazard Ratios)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$subgroup_results)) {
  res <- pooled_results$subgroup_results[[sg]]
  cat(sprintf("Subgroup %s: HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
              sg, res$hr, res$ci_lower, res$ci_upper, res$pvalue))
}

# Interaction
cat(sprintf("\n3. INTERACTION TEST (Treatment × %s)\n", subgroup_var))
cat(rep("-", 80), "\n")
for(res in pooled_results$interaction_results) {
  cat(sprintf("%s: Coef = %.4f (SE = %.4f), p = %.4f\n",
              res$term, res$coef, res$se, res$pvalue))
  if(res$pvalue < 0.05) {
    cat("→ SIGNIFICANT heterogeneity on relative scale\n")
  } else {
    cat("→ No significant heterogeneity on relative scale\n")
  }
}

# RMST
cat("\n4. RESTRICTED MEAN SURVIVAL TIME (3-year RMST)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$rmst_results)) {
  res <- pooled_results$rmst_results[[sg]]
  cat(sprintf("Subgroup %s: RMST Diff = %.1f days (95%% CI: %.1f-%.1f), p = %.4f\n",
              sg, res$rmst_diff, res$ci_lower, res$ci_upper, res$pvalue))
  cat(sprintf("  → Treatment %s %.1f days of event-free survival\n",
              if(res$rmst_diff > 0) "extends by" else "reduces by",
              abs(res$rmst_diff)))
}

# Absolute Risk Differences
cat("\n5. ABSOLUTE RISK DIFFERENCES\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$surv_prob_results)) {
  cat(sprintf("\nSubgroup %s:\n", sg))
  time_res <- pooled_results$surv_prob_results[[sg]]$time_results
  for(tr in time_res) {
    cat(sprintf("  At %.1f years: Risk Diff = %.4f (95%% CI: %.4f-%.4f)\n",
                tr$time/365.25, tr$risk_diff, tr$ci_lower, tr$ci_upper))
    cat(sprintf("    SE = %.4f, p = %.4f (%.2f%% reduction)\n",
                tr$se, tr$pvalue, tr$risk_diff * 100))
    cat(sprintf("    FMI = %.3f (%.1f%% of variance from imputation uncertainty)\n",
                tr$fmi, tr$fmi * 100))
  }
}

cat("\n", rep("=", 80), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# SUBGROUP SAMPLE STATISTICS
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("6. SUBGROUP SAMPLE STATISTICS\n")
cat(rep("=", 80), "\n")

cat("\n6-1. ORIGINAL DATA (Before Overlap Weighting)\n")
cat(rep("-", 80), "\n\n")

cat("A. Overall Dataset Statistics\n")
cat(rep("-", 40), "\n")

overall_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := factor(get(subgroup_var), levels = c("1065405", "1065406"))]
  dt[, Z := as.integer(get(treatment) == "glp")]


  total_n <- nrow(dt)

  gender_counts <- dt[, .N, by = Si]

  list(
    total_n = total_n,
    gender_counts = gender_counts
  )
})

avg_total_n <- round(mean(sapply(overall_stats, function(x) x$total_n)))

avg_gender_counts <- lapply(c("1065405", "1065406"), function(sg) {
  counts <- sapply(overall_stats, function(x) {
    x$gender_counts[x$gender_counts$Si == sg, ]$N
  })
  list(
    subgroup = sg,
    count = round(mean(counts))
  )
})

cat(sprintf("Total Sample Size: n = %d\n\n", avg_total_n))
cat("Gender Distribution:\n")
for(gc in avg_gender_counts) {
  cat(sprintf("  %s: n = %d (%.1f%%)\n",
              gc$subgroup, gc$count, gc$count/avg_total_n*100))
}
cat("\n")

cat("\nB. Statistics by Gender Subgroup\n")
cat(rep("-", 40), "\n\n")

original_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := factor(get(subgroup_var), levels = c("1065405", "1065406"))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  stats_by_sg <- dt[, .(
    n_total = .N,
    n_treat = sum(Z == 1),
    n_ctrl = sum(Z == 0),
    events_total = sum(Outcome),
    events_treat = sum(Outcome[Z == 1]),
    events_ctrl = sum(Outcome[Z == 0])
  ), by = Si]

  setDF(stats_by_sg)
  stats_by_sg
})


avg_original_stats <- lapply(unique(original_stats[[1]]$Si), function(sg) {
  n_total_vec <- sapply(original_stats, function(x) x$n_total[x$Si == sg])
  n_treat_vec <- sapply(original_stats, function(x) x$n_treat[x$Si == sg])
  n_ctrl_vec <- sapply(original_stats, function(x) x$n_ctrl[x$Si == sg])
  events_total_vec <- sapply(original_stats, function(x) x$events_total[x$Si == sg])
  events_treat_vec <- sapply(original_stats, function(x) x$events_treat[x$Si == sg])
  events_ctrl_vec <- sapply(original_stats, function(x) x$events_ctrl[x$Si == sg])

  list(
    subgroup = as.character(sg),
    n_total = round(mean(n_total_vec)),
    n_treat = round(mean(n_treat_vec)),
    n_ctrl = round(mean(n_ctrl_vec)),
    events_total = round(mean(events_total_vec)),
    events_treat = round(mean(events_treat_vec)),
    events_ctrl = round(mean(events_ctrl_vec))
  )
})

for(stats in avg_original_stats) {
  cat(sprintf("Subgroup: %s\n", stats$subgroup))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size:\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat("\n6-2. AFTER OVERLAP WEIGHTING (Pseudo Population)\n")
cat(rep("-", 80), "\n\n")

subgroup_levels <- names(all_results[[1]]$data_by_sg)

avg_weighted_stats <- lapply(subgroup_levels, function(sg) {

  stats_list <- lapply(1:5, function(i) {
    data_sg <- all_results[[i]]$data_by_sg[[sg]]

    if(is.null(data_sg)) return(NULL)

    n_total <- nrow(data_sg)
    n_events <- sum(data_sg$Outcome)
    ess_total <- sum(data_sg$w)^2 / sum(data_sg$w^2)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl <- data_sg[data_sg$Z == 0, ]

    n_treat <- nrow(data_treat)
    n_ctrl <- nrow(data_ctrl)
    events_treat <- sum(data_treat$Outcome)
    events_ctrl <- sum(data_ctrl$Outcome)
    ess_treat <- if(n_treat > 0) sum(data_treat$w)^2 / sum(data_treat$w^2) else 0
    ess_ctrl <- if(n_ctrl > 0) sum(data_ctrl$w)^2 / sum(data_ctrl$w^2) else 0

    list(
      n_total = n_total,
      n_treat = n_treat,
      n_ctrl = n_ctrl,
      events_total = n_events,
      events_treat = events_treat,
      events_ctrl = events_ctrl,
      ess_total = ess_total,
      ess_treat = ess_treat,
      ess_ctrl = ess_ctrl
    )
  })

  stats_list <- stats_list[!sapply(stats_list, is.null)]

  list(
    subgroup = sg,
    n_total = round(mean(sapply(stats_list, function(x) x$n_total))),
    n_treat = round(mean(sapply(stats_list, function(x) x$n_treat))),
    n_ctrl = round(mean(sapply(stats_list, function(x) x$n_ctrl))),
    events_total = round(mean(sapply(stats_list, function(x) x$events_total))),
    events_treat = round(mean(sapply(stats_list, function(x) x$events_treat))),
    events_ctrl = round(mean(sapply(stats_list, function(x) x$events_ctrl))),
    ess_total = round(mean(sapply(stats_list, function(x) x$ess_total)), 1),
    ess_treat = round(mean(sapply(stats_list, function(x) x$ess_treat)), 1),
    ess_ctrl = round(mean(sapply(stats_list, function(x) x$ess_ctrl)), 1)
  )
})
names(avg_weighted_stats) <- subgroup_levels

for(sg in names(avg_weighted_stats)) {
  stats <- avg_weighted_stats[[sg]]

  cat(sprintf("Subgroup: %s\n", sg))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size (Original):\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Effective Sample Size (ESS):\n"))
  cat(sprintf("    Total:     ESS = %.1f\n", stats$ess_total))
  cat(sprintf("    Treatment: ESS = %.1f\n", stats$ess_treat))
  cat(sprintf("    Control:   ESS = %.1f\n", stats$ess_ctrl))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

In [ ]:
data_by_sg <- all_results[[1]]$data_by_sg

data_female <- data_by_sg[["1065405"]]
data_f_treat <- data_female[data_female$Z == 1, ]
data_f_ctrl <- data_female[data_female$Z == 0, ]

fit_f_treat <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_f_treat, weights = w)
fit_f_ctrl <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_f_ctrl, weights = w)

data_male <- data_by_sg[["1065406"]]
data_m_treat <- data_male[data_male$Z == 1, ]
data_m_ctrl <- data_male[data_male$Z == 0, ]

fit_m_treat <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_m_treat, weights = w)
fit_m_ctrl <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_m_ctrl, weights = w)

image_path <- file.path(artifacts_images_dir, "t2d_secondary_primary_endpoint_sex.png")

png(filename = image_path, width = 9.5, height = 7, units = "in", res = 300)

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.20),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.20, by = 0.05), col = "gray50", lwd = 0.5)

lines(c(0, fit_f_treat$time / 30.44), c(0, 1 - fit_f_treat$surv),
      col = "#E41A1C", lty = 1, lwd = 2.5)
lines(c(0, fit_f_ctrl$time / 30.44), c(0, 1 - fit_f_ctrl$surv),
      col = "#E41A1C", lty = 3, lwd = 2.5)
lines(c(0, fit_m_treat$time / 30.44), c(0, 1 - fit_m_treat$surv),
      col = "#377EB8", lty = 1, lwd = 2.5)
lines(c(0, fit_m_ctrl$time / 30.44), c(0, 1 - fit_m_ctrl$surv),
      col = "#377EB8", lty = 3, lwd = 2.5)

# legend("topleft",
#        legend = c("Female - GLP-1 RAs", "Female - Non-GLP-1 RAs",
#                   "Male - GLP-1 RAs", "Male - Non-GLP-1 RAs"),
#        col = c("#E41A1C", "#E41A1C", "#377EB8", "#377EB8"),
#        lty = c(1, 3, 1, 3), lwd = 2.5, bty = "n", cex = 1.4)

dev.off()

cat("Plot saved to:", image_path, "\n")

## AGEGROUP

In [ ]:
setDTthreads(0)

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
perform_lasso_first_only <- function(imp_data, covariates, subgroup_var, treatment,
                                     sample_size = 50000) {

  cat("\n=== LASSO (First Imputation Only, Stratified Sampling) ===\n")

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment)
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(sample_size / n_groups)

  cat(sprintf("  Total groups (Si × Z): %d\n", n_groups))
  cat(sprintf("  Target per group: %d\n", sample_per_group))

  # Stratified sampling by Si × Z
  set.seed(12345)
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Actual sample: %d obs\n", nrow(dt_sample)))

  factor_cols <- names(dt_sample)[sapply(dt_sample, is.factor) | sapply(dt_sample, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt_sample[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  # Sparse matrix
  X_df <- dt_sample[, ..covariates]
  X <- sparse.model.matrix(~ . - 1, data = X_df)
  Si <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  n_subgroups <- ncol(Si)

  cat("  Creating interactions...\n")

  Xi_Si_list <- lapply(1:n_subgroups, function(k) {
    si_k <- as.vector(Si[, k])
    X_k <- X * si_k
    colnames(X_k) <- paste0(colnames(X), ":", colnames(Si)[k])
    X_k
  })

  Xi_Si <- do.call(cbind, Xi_Si_list)
  rm(Xi_Si_list); invisible(gc())

  design_matrix <- cbind(X, Si, Xi_Si)

  # Penalty factor
  penalty.factor <- c(
    rep(0, ncol(X) + ncol(Si)),
    rep(1, ncol(Xi_Si))
  )

  cat("  Running LASSO...\n")

  # LASSO
  lasso_fit <- cv.glmnet(
    design_matrix, dt_sample$Z,
    family = "binomial",
    alpha = 1,
    penalty.factor = penalty.factor,
    nfolds = 3,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 2000
  )

  selected_coef <- coef(lasso_fit, s = "lambda.min")
  selected_vars <- rownames(selected_coef)[selected_coef[, 1] != 0]
  selected_vars <- selected_vars[selected_vars != "(Intercept)"]
  selected_interactions <- selected_vars[grepl(":", selected_vars)]

  cat(sprintf("  Selected %d interactions\n", length(selected_interactions)))

  # Base variables + selected interactions
  union_vars <- c(colnames(X), colnames(Si), selected_interactions)

  return(list(
    union_vars = union_vars,
    selected_interactions = selected_interactions,
    base_vars = c(colnames(X), colnames(Si))
  ))
}

# ==============================================================================
# Propensity Score Estimation with Selected Variables
# ==============================================================================
analyze_with_selected_vars <- function(imp_data, imp_num,
                                       covariates, subgroup_var, treatment,
                                       union_vars, tau = 1095) {

  cat(sprintf("\n=== Processing Imputation %d ===\n", imp_num))

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(target_sample_size / n_groups)

  cat(sprintf("  Target sample: %d (stratified by Si × Z)\n", target_sample_size))

  # Stratified sampling
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]

  cat(sprintf("  Actual sample: %d\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample <- sparse.model.matrix(~ . - 1, data = X_df_sample)
  Si_sample <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  interaction_names <- union_vars[grepl(":", union_vars)]

  if(length(interaction_names) > 0 && length(interaction_names) < 800) {
    cat(sprintf("  Creating %d interactions (sample)...\n", length(interaction_names)))

    n_subgroups <- ncol(Si_sample)
    Xi_Si_list <- list()

    for(k in 1:n_subgroups) {
      si_k <- as.vector(Si_sample[, k])
      X_k <- X_sample * si_k
      colnames(X_k) <- paste0(colnames(X_sample), ":", colnames(Si_sample)[k])

      selected_cols <- intersect(colnames(X_k), interaction_names)
      if(length(selected_cols) > 0) {
        Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols, drop = FALSE]
      }
    }

    if(length(Xi_Si_list) > 0) {
      Xi_Si_sample <- do.call(cbind, Xi_Si_list)
      design_sample <- cbind(X_sample, Si_sample, Xi_Si_sample)
      rm(Xi_Si_list); invisible(gc())
    } else {
      design_sample <- cbind(X_sample, Si_sample)
    }
  } else {
    cat("  Too many interactions, using main effects only\n")
    design_sample <- cbind(X_sample, Si_sample)
  }

  all_vars <- colnames(design_sample)
  selected_cols <- intersect(union_vars, all_vars)
  design_selected_sample <- design_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d × %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model...\n")

  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family = "binomial",
    alpha = 0,
    lambda = 0.01,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 1000
  )

  cat("  Predicting on full data...\n")

  chunk_size <- 200000
  n_chunks <- ceiling(n_total / chunk_size)
  ps_all <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) {
      cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))
    }

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx <- min(chunk_i * chunk_size, n_total)

    dt_chunk <- dt[start_idx:end_idx]

    # Design matrix for chunk
    X_df_chunk <- dt_chunk[, ..covariates]
    X_chunk <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    Si_chunk <- sparse.model.matrix(~ Si - 1, data = dt_chunk)

    if(length(interaction_names) > 0 && length(interaction_names) < 800) {
      n_subgroups <- ncol(Si_chunk)
      Xi_Si_list <- list()

      for(k in 1:n_subgroups) {
        si_k <- as.vector(Si_chunk[, k])
        X_k <- X_chunk * si_k
        colnames(X_k) <- paste0(colnames(X_chunk), ":", colnames(Si_chunk)[k])

        selected_cols_chunk <- intersect(colnames(X_k), interaction_names)
        if(length(selected_cols_chunk) > 0) {
          Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols_chunk, drop = FALSE]
        }
      }

      if(length(Xi_Si_list) > 0) {
        Xi_Si_chunk <- do.call(cbind, Xi_Si_list)
        design_chunk <- cbind(X_chunk, Si_chunk, Xi_Si_chunk)
        rm(Xi_Si_list)
      } else {
        design_chunk <- cbind(X_chunk, Si_chunk)
      }
    } else {
      design_chunk <- cbind(X_chunk, Si_chunk)
    }

    design_chunk <- design_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, Si_chunk, design_chunk)
    invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  # Overlap weights
  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = .(Si, Z)]

  data <- dt[, .(Si, Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data, X = NULL, Si = NULL))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  cat(sprintf("  Cox models...\n"))

  # Subgroup-Specific Models
  subgroup_levels <- levels(data$Si)
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  # Interaction Model
  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)
  interaction_coefs <- grep("Z:Si", rownames(interaction_summary$coefficients), value = TRUE)

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
 # ==============================================================================
# RMST and Risk Difference Calculation
# ==============================================================================

calculate_rmst_weighted <- function(time, status, weights, tau) {

  ord <- order(time)
  time <- time[ord]
  status <- status[ord]
  weights <- weights[ord]

  w_atrisk <- rev(cumsum(rev(weights)))
  w_event  <- status * weights

  w_surv <- cumprod(1 - w_event / pmax(w_atrisk, 1e-10))
  w_surv <- c(1, w_surv)
  time_ext <- c(0, time)

  idx <- time_ext <= tau
  time_use <- time_ext[idx]
  surv_use <- w_surv[idx]

  if (max(time_use) < tau) {
    time_use <- c(time_use, tau)
    surv_use <- c(surv_use, tail(surv_use, 1))
  }

  rmst <- sum(diff(time_use) * head(surv_use, -1))
  return(rmst)
}

calculate_rmst_point <- function(results, tau = 1095, verbose = TRUE) {

  if (verbose) cat("  RMST (weighted KM, point estimates only)...\n")

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  results_list <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if (nrow(data_sg) <= 10) return(NULL)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl  <- data_sg[data_sg$Z == 0, ]

    if (nrow(data_treat) <= 10 || nrow(data_ctrl) <= 10) return(NULL)

    # Weighted Kaplan–Meier
    fit_treat <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_treat,
      weights = data_treat$w
    )
    fit_ctrl <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_ctrl,
      weights = data_ctrl$w
    )

    # RMST
    rmst_treat <- tryCatch(
      calculate_rmst_weighted(
        time = data_treat$Survival_Time,
        status = data_treat$Outcome,
        weights = data_treat$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_ctrl <- tryCatch(
      calculate_rmst_weighted(
        time = data_ctrl$Survival_Time,
        status = data_ctrl$Outcome,
        weights = data_ctrl$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_diff <- rmst_treat - rmst_ctrl

    # Risk Differences
    time_points <- c(365, 730, 1095)
    surv_at_times <- lapply(time_points, function(t) {
      sum_treat <- summary(fit_treat, times = t, extend = TRUE)
      sum_ctrl  <- summary(fit_ctrl,  times = t, extend = TRUE)

      surv_t <- if (length(sum_treat$surv) > 0) sum_treat$surv else NA_real_
      surv_c <- if (length(sum_ctrl$surv) > 0) sum_ctrl$surv else NA_real_
      se_t   <- if (length(sum_treat$std.err) > 0) sum_treat$std.err else NA_real_
      se_c   <- if (length(sum_ctrl$std.err) > 0) sum_ctrl$std.err else NA_real_

      risk_t <- 1 - surv_t
      risk_c <- 1 - surv_c
      risk_diff <- risk_c - risk_t

      risk_diff_var <- if (!is.na(se_t) && !is.na(se_c)) {
        se_t^2 + se_c^2
      } else NA_real_

      baseline_risk_var <- if (!is.na(se_c)) {
        se_c^2
      } else NA_real_

      list(
        time = t,
        surv_treat = surv_t,
        se_treat = se_t,
        surv_ctrl = surv_c,
        se_ctrl = se_c,
        risk_treat = risk_t,
        risk_ctrl = risk_c,
        risk_diff = risk_diff,
        risk_diff_var = risk_diff_var,
        baseline_risk = risk_c,
        baseline_risk_var = baseline_risk_var
      )
    })

    list(
      rmst = list(
        subgroup = sg,
        rmst_treat = rmst_treat,
        rmst_ctrl = rmst_ctrl,
        rmst_diff = rmst_diff,
        tau = tau
      ),
      surv_prob = list(
        subgroup = sg,
        time_points = surv_at_times
      )
    )
  })

  results_list <- results_list[!sapply(results_list, is.null)]

  rmst_results <- lapply(results_list, function(x) x$rmst)
  names(rmst_results) <- sapply(rmst_results, function(x) x$subgroup)

  surv_prob_results <- lapply(results_list, function(x) x$surv_prob)
  names(surv_prob_results) <- sapply(surv_prob_results, function(x) x$subgroup)

  return(list(
    rmst_results = rmst_results,
    surv_prob_results = surv_prob_results
  ))
}
bootstrap_rmst_diff <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  set.seed(seed)

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  base_est <- calculate_rmst_point(results, tau = tau, verbose = TRUE)
  rmst_point <- base_est$rmst_results

  use_parallel <- !is.null(n_cores) && n_cores > 1

  if (use_parallel) {
    if (n_cores == "auto") {
      n_cores <- max(1, parallel::detectCores() - 1)
    }

    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac,
        ", cores =", n_cores, ")...\n")

    cl <- parallel::makeCluster(n_cores)

    parallel::clusterExport(cl,
                           c("calculate_rmst_point", "calculate_rmst_weighted",
                             "data_by_sg", "tau", "subsample_frac"),
                           envir = environment())

    parallel::clusterEvalQ(cl, library(survival))

    boot_results <- parallel::parLapply(cl, 1:B, function(b) {

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- tryCatch(
        calculate_rmst_point(results_boot, tau = tau, verbose = FALSE),
        error = function(e) NULL
      )

      if (is.null(boot_est)) return(NULL)

      sapply(boot_est$rmst_results, function(x) x$rmst_diff)
    })

    parallel::stopCluster(cl)

    boot_results <- boot_results[!sapply(boot_results, is.null)]
    boot_mat_data <- do.call(rbind, boot_results)

    boot_mat <- lapply(seq_along(subgroup_levels), function(i) {
      boot_mat_data[, i]
    })
    names(boot_mat) <- subgroup_levels

  } else {
    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac, ")...\n")

    boot_mat <- lapply(subgroup_levels, function(sg) {
      numeric(B)
    })
    names(boot_mat) <- subgroup_levels

    for (b in seq_len(B)) {
      if (b %% 20 == 0) cat("    Iteration", b, "of", B, "\n")

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- calculate_rmst_point(results_boot, tau = tau, verbose = FALSE)
      rmst_boot <- boot_est$rmst_results

      for (sg in names(rmst_boot)) {
        boot_mat[[sg]][b] <- rmst_boot[[sg]]$rmst_diff
      }
    }
  }

  out <- lapply(rmst_point, function(x) {
    sg <- x$subgroup
    diffs <- boot_mat[[sg]]
    diffs <- diffs[is.finite(diffs)]

    if (length(diffs) < 10) {
      se <- NA_real_
      ci_lower <- NA_real_
      ci_upper <- NA_real_
    } else {
      se <- sd(diffs)
      # Percentile CI
      ci_lower <- quantile(diffs, 0.025, na.rm = TRUE)
      ci_upper <- quantile(diffs, 0.975, na.rm = TRUE)
    }

    c(x, list(
      se_rmst_diff = se,
      ci_lower = ci_lower,
      ci_upper = ci_upper,
      B = length(diffs),
      subsample_frac = subsample_frac
    ))
  })
  names(out) <- names(rmst_point)

  return(list(
    rmst_results = out,
    surv_prob_results = base_est$surv_prob_results
  ))
}

calculate_rmst_fast <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  cat("\n=== RMST with Subsample Bootstrap ===\n")
  cat("References:\n")
  cat("  - Zeng et al. (2022) Stat Med [bootstrap SE for weighted estimates]\n")
  cat("  - adjustedCurves::adjusted_rmst [RMST bootstrap implementation]\n\n")

  bootstrap_rmst_diff(
    results = results,
    tau = tau,
    B = B,
    subsample_frac = subsample_frac,
    seed = seed,
    n_cores = n_cores
  )
}

In [ ]:
# ==============================================================================
# Pool Main Effects
# ==============================================================================
pool_main_effects <- function(all_results) {

  m <- length(all_results)

  # Pool Treatment Effect
  treat_coefs <- sapply(all_results, function(x) x$main_effects$treatment_coef)
  treat_ses <- sapply(all_results, function(x) x$main_effects$treatment_se)

  Q_bar_treat <- mean(treat_coefs)
  U_bar_treat <- mean(treat_ses^2)
  B_treat <- var(treat_coefs)
  T_treat <- U_bar_treat + (1 + 1/m) * B_treat
  se_treat <- sqrt(T_treat)

  df_treat <- if(B_treat < 1e-10) Inf else {
    (m - 1) * (1 + U_bar_treat / ((1 + 1/m) * B_treat))^2
  }

  pooled_main_effects <- list(
    treatment = list(
      coef = Q_bar_treat,
      se = se_treat,
      hr = exp(Q_bar_treat),
      ci_lower = exp(Q_bar_treat - qt(0.975, df_treat) * se_treat),
      ci_upper = exp(Q_bar_treat + qt(0.975, df_treat) * se_treat),
      pvalue = 2 * pt(-abs(Q_bar_treat / se_treat), df = df_treat),
      df = df_treat
    )
  )

  # Pool All Subgroup Coefficients
  if (!is.null(all_results[[1]]$main_effects$subgroup_coefs)) {

    common_names <- Reduce(
      intersect,
      lapply(all_results, function(res)
        vapply(res$main_effects$subgroup_coefs, `[[`, character(1), "name"))
    )

    pooled_subgroup_coefs <- lapply(common_names, function(sg_name) {

      coefs <- sapply(all_results, function(res) {
        idx <- which(vapply(res$main_effects$subgroup_coefs,
                            `[[`, character(1), "name") == sg_name)
        if (length(idx) == 0) NA_real_ else
          as.numeric(res$main_effects$subgroup_coefs[[idx]]$coef)
      })

      ses <- sapply(all_results, function(res) {
        idx <- which(vapply(res$main_effects$subgroup_coefs,
                            `[[`, character(1), "name") == sg_name)
        if (length(idx) == 0) NA_real_ else
          as.numeric(res$main_effects$subgroup_coefs[[idx]]$se)
      })

      valid <- is.finite(coefs) & is.finite(ses)
      coefs <- coefs[valid]
      ses   <- ses[valid]

      if (length(coefs) == 0L) return(NULL)

      m_valid        <- length(coefs)
      Q_bar_subgroup <- mean(coefs)
      U_bar_subgroup <- mean(ses^2)
      B_subgroup     <- var(coefs)
      T_subgroup     <- U_bar_subgroup + (1 + 1/m_valid) * B_subgroup
      se_subgroup    <- sqrt(T_subgroup)
      df_subgroup    <- if (B_subgroup < 1e-10) Inf else {
        (m_valid - 1) * (1 + U_bar_subgroup / ((1 + 1/m_valid) * B_subgroup))^2
      }

      list(
        name    = sg_name,
        coef    = Q_bar_subgroup,
        se      = se_subgroup,
        hr      = exp(Q_bar_subgroup),
        ci_lower = exp(Q_bar_subgroup - qt(0.975, df_subgroup) * se_subgroup),
        ci_upper = exp(Q_bar_subgroup + qt(0.975, df_subgroup) * se_subgroup),
        pvalue  = 2 * pt(-abs(Q_bar_subgroup / se_subgroup), df = df_subgroup),
        df      = df_subgroup
      )
    })

    pooled_subgroup_coefs <- pooled_subgroup_coefs[!vapply(pooled_subgroup_coefs, is.null, logical(1))]

    pooled_main_effects$subgroup_coefs <- pooled_subgroup_coefs
    pooled_main_effects$subgroup_var   <- all_results[[1]]$main_effects$subgroup_var
  }

  return(pooled_main_effects)
}

# ==============================================================================
# Pool Subgroup-Specific Treatment Effects
# ==============================================================================
pool_subgroup_results <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  m <- length(all_results)

  pooled_subgroups <- lapply(subgroup_names, function(sg) {
    coefs <- sapply(all_results, function(x) x$subgroup_results[[sg]]$coef)
    ses <- sapply(all_results, function(x) x$subgroup_results[[sg]]$se_robust)

    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m - 1) * (1 + U_bar / ((1 + 1/m) * B))^2

    list(
      subgroup = sg,
      coef = Q_bar,
      se = se_pooled,
      hr = exp(Q_bar),
      ci_lower = exp(Q_bar - qt(0.975, df) * se_pooled),
      ci_upper = exp(Q_bar + qt(0.975, df) * se_pooled),
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })

  names(pooled_subgroups) <- subgroup_names
  return(pooled_subgroups)
}

In [ ]:
# ==============================================================================
# Pool Interaction Terms
# ==============================================================================
pool_interactions <- function(all_results) {

  interaction_terms <- sapply(all_results[[1]]$interaction_results, function(x) x$term)

  pooled_interactions <- lapply(interaction_terms, function(term) {
    coefs <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$coef else NA
    })

    ses <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$se_robust else NA
    })

    valid_idx <- !is.na(coefs) & !is.na(ses)
    coefs <- coefs[valid_idx]
    ses <- ses[valid_idx]

    if(length(coefs) == 0) return(NULL)

    m_valid <- length(coefs)
    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      term = term,
      coef = Q_bar,
      se = se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })

  pooled_interactions <- pooled_interactions[!sapply(pooled_interactions, is.null)]
  return(pooled_interactions)
}

# ==============================================================================
# Pool RMST Differences
# ==============================================================================
pool_rmst <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)

  pooled_rmst <- lapply(subgroup_names, function(sg) {
    rmst_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$rmst_diff)
    se_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$se_rmst_diff)

    valid_idx <- !is.na(rmst_diffs) & !is.na(se_diffs)
    rmst_diffs <- rmst_diffs[valid_idx]
    se_diffs <- se_diffs[valid_idx]

    if(length(rmst_diffs) == 0) return(NULL)

    m_valid <- length(rmst_diffs)
    Q_bar <- mean(rmst_diffs)
    U_bar <- mean(se_diffs^2)
    B <- var(rmst_diffs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      subgroup = sg,
      rmst_diff = Q_bar,
      se = se_pooled,
      ci_lower = Q_bar - qt(0.975, df) * se_pooled,
      ci_upper = Q_bar + qt(0.975, df) * se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      tau = all_results[[1]]$rmst_results[[sg]]$tau
    )
  })

  names(pooled_rmst) <- subgroup_names
  pooled_rmst <- pooled_rmst[!sapply(pooled_rmst, is.null)]

  return(pooled_rmst)
}

In [ ]:
# ==============================================================================
# Pool Absolute Risk Differences and Baseline Risks
# ==============================================================================
pool_survival_probs <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  time_points <- c(365, 730, 1095)

  pooled_surv_probs <- lapply(subgroup_names, function(sg) {
    time_results <- lapply(seq_along(time_points), function(t_idx) {

      risk_diffs <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff else NA
      })

      risk_diff_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff_var else NA
      })

      baseline_risks <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk else NA
      })

      baseline_risk_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk_var else NA
      })

      valid_idx_rd <- !is.na(risk_diffs) & !is.na(risk_diff_vars)
      risk_diffs_valid <- risk_diffs[valid_idx_rd]
      risk_diff_vars_valid <- risk_diff_vars[valid_idx_rd]

      valid_idx_br <- !is.na(baseline_risks) & !is.na(baseline_risk_vars)
      baseline_risks_valid <- baseline_risks[valid_idx_br]
      baseline_risk_vars_valid <- baseline_risk_vars[valid_idx_br]

      if(length(risk_diffs_valid) == 0) return(NULL)

      # Rubin's Rules for Risk Difference
      m_valid_rd <- length(risk_diffs_valid)
      Q_bar_rd <- mean(risk_diffs_valid)
      U_bar_rd <- mean(risk_diff_vars_valid)
      B_rd <- var(risk_diffs_valid)
      T_rd <- U_bar_rd + (1 + 1/m_valid_rd) * B_rd
      se_pooled_rd <- sqrt(T_rd)

      df_rd <- if(B_rd < 1e-10) Inf else {
        (m_valid_rd - 1) * (1 + U_bar_rd / ((1 + 1/m_valid_rd) * B_rd))^2
      }

      ci_lower_rd <- Q_bar_rd - qt(0.975, df_rd) * se_pooled_rd
      ci_upper_rd <- Q_bar_rd + qt(0.975, df_rd) * se_pooled_rd
      pvalue_rd <- 2 * pt(-abs(Q_bar_rd / se_pooled_rd), df = df_rd)

      # Rubin's Rules for Baseline Risk
      baseline_result <- if(length(baseline_risks_valid) > 0) {
        m_valid_br <- length(baseline_risks_valid)
        Q_bar_br <- mean(baseline_risks_valid)
        U_bar_br <- mean(baseline_risk_vars_valid)
        B_br <- var(baseline_risks_valid)
        T_br <- U_bar_br + (1 + 1/m_valid_br) * B_br
        se_pooled_br <- sqrt(T_br)

        df_br <- if(B_br < 1e-10) Inf else {
          (m_valid_br - 1) * (1 + U_bar_br / ((1 + 1/m_valid_br) * B_br))^2
        }

        ci_lower_br <- Q_bar_br - qt(0.975, df_br) * se_pooled_br
        ci_upper_br <- Q_bar_br + qt(0.975, df_br) * se_pooled_br
        pvalue_br <- 2 * pt(-abs(Q_bar_br / se_pooled_br), df = df_br)

        list(
          baseline_risk = Q_bar_br,
          baseline_risk_se = se_pooled_br,
          baseline_risk_ci_lower = ci_lower_br,
          baseline_risk_ci_upper = ci_upper_br,
          baseline_risk_df = df_br
        )
      } else {
        list(
          baseline_risk = NA,
          baseline_risk_se = NA,
          baseline_risk_ci_lower = NA,
          baseline_risk_ci_upper = NA,
          baseline_risk_df = NA
        )
      }

      c(
        list(
          time = time_points[t_idx],
          risk_diff = Q_bar_rd,
          se = se_pooled_rd,
          ci_lower = ci_lower_rd,
          ci_upper = ci_upper_rd,
          pvalue = pvalue_rd,
          df = df_rd,
          within_var = U_bar_rd,
          between_var = B_rd,
          fmi = (1 + 1/m_valid_rd) * B_rd / T_rd
        ),
        baseline_result
      )
    })

    time_results <- time_results[!sapply(time_results, is.null)]
    list(subgroup = sg, time_results = time_results)
  })

  names(pooled_surv_probs) <- subgroup_names
  return(pooled_surv_probs)
}

# ==============================================================================
# Execute All Pooling Functions
# ==============================================================================
pool_results <- function(all_results) {
    cat("\n", rep("=", 80), "\n")
    cat("EXECUTING RUBIN'S POOLING...\n")
    cat(rep("=", 80), "\n\n")

    # Pool Subgroup-Specific Results
    cat("Step 2/5: Pooling subgroup-specific treatment effects...\n")
    pooled_subgroups <- pool_subgroup_results(all_results)
    cat(sprintf(" ✓ %d subgroups pooled\n\n", length(pooled_subgroups)))

    # Pool Interaction Terms
    cat("Step 3/5: Pooling interaction terms...\n")
    pooled_interactions <- pool_interactions(all_results)
    cat(sprintf(" ✓ %d interaction terms pooled\n\n", length(pooled_interactions)))

    # Pool RMST Results
    cat("Step 4/5: Pooling RMST differences...\n")
    pooled_rmst <- pool_rmst(all_results)
    cat(sprintf(" ✓ RMST for %d subgroups pooled\n\n", length(pooled_rmst)))

    # Pool Survival Probabilities and Risk Differences
    cat("Step 5/5: Pooling absolute risk differences and baseline risks...\n")
    pooled_surv_probs <- pool_survival_probs(all_results)
    cat(sprintf(" ✓ Risk differences for %d subgroups pooled\n\n", length(pooled_surv_probs)))

    # Combine all results
    pooled_results <- list(
        subgroup_results = pooled_subgroups,
        interaction_results = pooled_interactions,
        rmst_results = pooled_rmst,
        surv_prob_results = pooled_surv_probs
    )

    return(pooled_results)
}

In [ ]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
subgroup_var <- "AgeGroup.y"
covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
  "HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
  "HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
  "HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
  "HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant",
  "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone", "HasOtherAntiHypertensive",
  "HasHospitalization", "Ethnicity", "Race", "CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
  "NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
  "NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Sex", "CalendarYear",
  "AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HemoglobinValue",
  "LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
  "SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue", "HbA1cValue",
  "AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
  "AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)
covariates <- setdiff(covariates, subgroup_var)
treatment <- "FirstMedGroup"

start_time <- Sys.time()

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 1: LASSO (First Imputation Only, 50K Stratified)\n")
cat(rep("=", 80), "\n")

lasso_result <- perform_lasso_first_only(
  imp_data = imputed_t2d_list[[1]],
  covariates = covariates,
  subgroup_var = subgroup_var,
  treatment = treatment,
  sample_size = 50000
)

union_vars <- lasso_result$union_vars

cat("\n", rep("=", 80), "\n")
cat(sprintf("Selected variables: %d (including %d interactions)\n",
            length(union_vars), length(lasso_result$selected_interactions)))
cat(rep("=", 80), "\n")

# ==============================================================================
# Full Analysis
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("Full Analysis (All 5 Imputations - SEQUENTIAL)\n")
cat(rep("=", 80), "\n")

all_results <- list()

for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Processing Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_with_selected_vars(
    imp_data = imputed_t2d_list[[i]],
    imp_num = i,
    covariates = covariates,
    subgroup_var = subgroup_var,
    treatment = treatment,
    union_vars = union_vars,
    tau = 1095
  )

  cox_results <- complete_analysis_fast(
    data_list,
    imp_num = i,
    subgroup_var = subgroup_var,
    tau = 1095
  )

  rmst_results <- calculate_rmst_fast(
    cox_results,
    tau = 1095,
    B = 100,
    subsample_frac = 0.3,
    n_cores = 4
  )

  all_results[[i]] <- list(
    main_effects = cox_results$main_effects,
    subgroup_results = cox_results$subgroup_results,
    interaction_results = cox_results$interaction_results,
    data_by_sg = if(i == 1) cox_results$data_by_sg else NULL,
    rmst_results = rmst_results$rmst_results,
    surv_prob_results = rmst_results$surv_prob_results
  )

  cat("Imputation", i, "completed!\n")

  invisible(gc())
}

cat("\nAll imputations completed!\n")

# ==============================================================================
# Rubin's Pooling
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results(all_results)

end_time <- Sys.time()
elapsed <- as.numeric(end_time - start_time, units = "mins")

cat("\n", rep("=", 80), "\n")
cat(sprintf("Total Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# Display Results
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("FINAL POOLED RESULTS (3-Year Follow-up)\n")
cat(sprintf("Subgroup Analysis by: %s\n", subgroup_var))
cat(rep("=", 80), "\n\n")

# Subgroup-Specific HRs
cat("\n2. SUBGROUP-SPECIFIC TREATMENT EFFECTS (Hazard Ratios)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$subgroup_results)) {
  res <- pooled_results$subgroup_results[[sg]]
  cat(sprintf("Subgroup %s: HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
              sg, res$hr, res$ci_lower, res$ci_upper, res$pvalue))
}

# Interaction
cat(sprintf("\n3. INTERACTION TEST (Treatment × %s)\n", subgroup_var))
cat(rep("-", 80), "\n")
for(res in pooled_results$interaction_results) {
  cat(sprintf("%s: Coef = %.4f (SE = %.4f), p = %.4f\n",
              res$term, res$coef, res$se, res$pvalue))
  if(res$pvalue < 0.05) {
    cat("  → SIGNIFICANT heterogeneity on relative scale\n")
  } else {
    cat("  → No significant heterogeneity on relative scale\n")
  }
}

# Overall interaction interpretation
any_interaction_sig <- any(sapply(pooled_results$interaction_results,
                                  function(x) x$pvalue < 0.05))
cat("\n")
if(any_interaction_sig) {
  cat("OVERALL: Significant treatment effect heterogeneity detected\n")
} else {
  cat("OVERALL: No significant treatment effect heterogeneity\n")
}

# RMST
cat("\n4. RESTRICTED MEAN SURVIVAL TIME (3-year RMST)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$rmst_results)) {
  res <- pooled_results$rmst_results[[sg]]
  cat(sprintf("Subgroup %s: RMST Diff = %.1f days (95%% CI: %.1f-%.1f), p = %.4f\n",
              sg, res$rmst_diff, res$ci_lower, res$ci_upper, res$pvalue))
  cat(sprintf("  → Treatment %s %.1f days of event-free survival\n",
              if(res$rmst_diff > 0) "extends by" else "reduces by",
              abs(res$rmst_diff)))
}

# Absolute Risk Differences
cat("\n5. ABSOLUTE RISK DIFFERENCES\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$surv_prob_results)) {
  cat(sprintf("\nSubgroup %s:\n", sg))
  time_res <- pooled_results$surv_prob_results[[sg]]$time_results
  for(tr in time_res) {
    cat(sprintf("  At %.1f years: Risk Diff = %.4f (95%% CI: %.4f-%.4f)\n",
                tr$time/365.25, tr$risk_diff, tr$ci_lower, tr$ci_upper))
    cat(sprintf("    SE = %.4f, p = %.4f (%.2f%% reduction)\n",
                tr$se, tr$pvalue, tr$risk_diff * 100))
    cat(sprintf("    FMI = %.3f (%.1f%% of variance from imputation uncertainty)\n",
                tr$fmi, tr$fmi * 100))
  }
}

cat("\n", rep("=", 80), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# SUBGROUP SAMPLE STATISTICS
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("6. SUBGROUP SAMPLE STATISTICS\n")
cat(rep("=", 80), "\n")

cat("\n6-1. ORIGINAL DATA (Before Overlap Weighting)\n")
cat(rep("-", 80), "\n\n")

cat("A. Overall Dataset Statistics\n")
cat(rep("-", 40), "\n")

overall_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  total_n <- nrow(dt)

  age_counts <- dt[, .N, by = Si]

  list(
    total_n = total_n,
    age_counts = age_counts
  )
})

avg_total_n <- round(mean(sapply(overall_stats, function(x) x$total_n)))

age_group_levels <- unique(overall_stats[[1]]$age_counts$Si)

avg_age_counts <- lapply(age_group_levels, function(sg) {
  counts <- sapply(overall_stats, function(x) {
    x$age_counts[x$age_counts$Si == sg, ]$N
  })
  list(
    subgroup = as.character(sg),
    count = round(mean(counts))
  )
})

cat(sprintf("Total Sample Size: n = %d\n\n", avg_total_n))
cat("Age Group Distribution:\n")
for(ac in avg_age_counts) {
  cat(sprintf("  %s: n = %d (%.1f%%)\n",
              ac$subgroup, ac$count, ac$count/avg_total_n*100))
}
cat("\n")

cat("\nB. Statistics by Age Subgroup\n")
cat(rep("-", 40), "\n\n")

original_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  stats_by_sg <- dt[, .(
    n_total = .N,
    n_treat = sum(Z == 1),
    n_ctrl = sum(Z == 0),
    events_total = sum(Outcome),
    events_treat = sum(Outcome[Z == 1]),
    events_ctrl = sum(Outcome[Z == 0])
  ), by = Si]

  setDF(stats_by_sg)
  stats_by_sg
})

avg_original_stats <- lapply(unique(original_stats[[1]]$Si), function(sg) {
  n_total_vec <- sapply(original_stats, function(x) x$n_total[x$Si == sg])
  n_treat_vec <- sapply(original_stats, function(x) x$n_treat[x$Si == sg])
  n_ctrl_vec <- sapply(original_stats, function(x) x$n_ctrl[x$Si == sg])
  events_total_vec <- sapply(original_stats, function(x) x$events_total[x$Si == sg])
  events_treat_vec <- sapply(original_stats, function(x) x$events_treat[x$Si == sg])
  events_ctrl_vec <- sapply(original_stats, function(x) x$events_ctrl[x$Si == sg])

  list(
    subgroup = as.character(sg),
    n_total = round(mean(n_total_vec)),
    n_treat = round(mean(n_treat_vec)),
    n_ctrl = round(mean(n_ctrl_vec)),
    events_total = round(mean(events_total_vec)),
    events_treat = round(mean(events_treat_vec)),
    events_ctrl = round(mean(events_ctrl_vec))
  )
})

for(stats in avg_original_stats) {
  cat(sprintf("Age Group: %s\n", stats$subgroup))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size:\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat("\n6-2. AFTER OVERLAP WEIGHTING (Pseudo Population)\n")
cat(rep("-", 80), "\n\n")

subgroup_levels <- names(all_results[[1]]$data_by_sg)

avg_weighted_stats <- lapply(subgroup_levels, function(sg) {

  stats_list <- lapply(1:5, function(i) {
    data_sg <- all_results[[i]]$data_by_sg[[sg]]

    if(is.null(data_sg)) return(NULL)

    n_total <- nrow(data_sg)
    n_events <- sum(data_sg$Outcome)
    ess_total <- sum(data_sg$w)^2 / sum(data_sg$w^2)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl <- data_sg[data_sg$Z == 0, ]

    n_treat <- nrow(data_treat)
    n_ctrl <- nrow(data_ctrl)
    events_treat <- sum(data_treat$Outcome)
    events_ctrl <- sum(data_ctrl$Outcome)
    ess_treat <- if(n_treat > 0) sum(data_treat$w)^2 / sum(data_treat$w^2) else 0
    ess_ctrl <- if(n_ctrl > 0) sum(data_ctrl$w)^2 / sum(data_ctrl$w^2) else 0

    list(
      n_total = n_total,
      n_treat = n_treat,
      n_ctrl = n_ctrl,
      events_total = n_events,
      events_treat = events_treat,
      events_ctrl = events_ctrl,
      ess_total = ess_total,
      ess_treat = ess_treat,
      ess_ctrl = ess_ctrl
    )
  })

  stats_list <- stats_list[!sapply(stats_list, is.null)]

  list(
    subgroup = sg,
    n_total = round(mean(sapply(stats_list, function(x) x$n_total))),
    n_treat = round(mean(sapply(stats_list, function(x) x$n_treat))),
    n_ctrl = round(mean(sapply(stats_list, function(x) x$n_ctrl))),
    events_total = round(mean(sapply(stats_list, function(x) x$events_total))),
    events_treat = round(mean(sapply(stats_list, function(x) x$events_treat))),
    events_ctrl = round(mean(sapply(stats_list, function(x) x$events_ctrl))),
    ess_total = round(mean(sapply(stats_list, function(x) x$ess_total)), 1),
    ess_treat = round(mean(sapply(stats_list, function(x) x$ess_treat)), 1),
    ess_ctrl = round(mean(sapply(stats_list, function(x) x$ess_ctrl)), 1)
  )
})
names(avg_weighted_stats) <- subgroup_levels

for(sg in names(avg_weighted_stats)) {
  stats <- avg_weighted_stats[[sg]]

  cat(sprintf("Age Group: %s\n", sg))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size (Original):\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Effective Sample Size (ESS):\n"))
  cat(sprintf("    Total:     ESS = %.1f\n", stats$ess_total))
  cat(sprintf("    Treatment: ESS = %.1f\n", stats$ess_treat))
  cat(sprintf("    Control:   ESS = %.1f\n", stats$ess_ctrl))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

In [ ]:
data_by_sg <- all_results[[1]]$data_by_sg

age_groups <- names(data_by_sg)
age_labels <- c("18-44 y", "45-64 y", "≥65 y")
colors <- c("#E41A1C", "#377EB8", "#4DAF4A")  # Red, Blue, Green

fit_list <- list()
for(i in seq_along(age_groups)) {
  sg <- age_groups[i]
  data_sg <- data_by_sg[[sg]]

  data_treat <- data_sg[data_sg$Z == 1, ]
  data_ctrl <- data_sg[data_sg$Z == 0, ]

  fit_list[[i]] <- list(
    treat = survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_treat, weights = w),
    ctrl = survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_ctrl, weights = w)
  )
}

image_path <- file.path(artifacts_images_dir, "t2d_secondary_primary_endpoint_age.png")

png(filename = image_path, width = 9.5, height = 7, units = "in", res = 300)

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.2),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.2, by = 0.05), col = "gray50", lwd = 0.5)

# Plot lines for each age group
for(i in seq_along(age_groups)) {
  lines(c(0, fit_list[[i]]$treat$time / 30.44),
        c(0, 1 - fit_list[[i]]$treat$surv),
        col = colors[i], lty = 1, lwd = 2.5)
  lines(c(0, fit_list[[i]]$ctrl$time / 30.44),
        c(0, 1 - fit_list[[i]]$ctrl$surv),
        col = colors[i], lty = 3, lwd = 2.5)
}

# # Legend
# legend_labels <- c()
# legend_cols <- c()
# legend_ltys <- c()

# for(i in seq_along(age_labels)) {
#   legend_labels <- c(legend_labels,
#                      paste0("Age ", age_labels[i], " - GLP-1 RAs"),
#                      paste0("Age ", age_labels[i], " - Non-GLP-1 RAs"))
#   legend_cols <- c(legend_cols, colors[i], colors[i])
#   legend_ltys <- c(legend_ltys, 1, 3)
# }

# legend("topleft",
#        legend = legend_labels,
#        col = legend_cols,
#        lty = legend_ltys,
#        lwd = 2.5,
#        bty = "n",
#        cex = 1.4)

dev.off()

cat("Plot saved to:", image_path, "\n")

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.2),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.2, by = 0.05), col = "gray50", lwd = 0.5)

# Plot lines for each age group
for(i in seq_along(age_groups)) {
  lines(c(0, fit_list[[i]]$treat$time / 30.44),
        c(0, 1 - fit_list[[i]]$treat$surv),
        col = colors[i], lty = 1, lwd = 2.5)
  lines(c(0, fit_list[[i]]$ctrl$time / 30.44),
        c(0, 1 - fit_list[[i]]$ctrl$surv),
        col = colors[i], lty = 3, lwd = 2.5)
}

# legend("topleft",
#        legend = legend_labels,
#        col = legend_cols,
#        lty = legend_ltys,
#        lwd = 2.5,
#        bty = "n",
#        cex = 1.4)

## RACE

In [ ]:
setDTthreads(0)

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
perform_lasso_first_only <- function(imp_data, covariates, subgroup_var, treatment,
                                     sample_size = 50000) {

  cat("\n=== LASSO (First Imputation Only, Stratified Sampling) ===\n")

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment)
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(sample_size / n_groups)

  cat(sprintf("  Total groups (Si × Z): %d\n", n_groups))
  cat(sprintf("  Target per group: %d\n", sample_per_group))

  set.seed(12345)
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Actual sample: %d obs\n", nrow(dt_sample)))

  factor_cols <- names(dt_sample)[sapply(dt_sample, is.factor) | sapply(dt_sample, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt_sample[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  X_df <- dt_sample[, ..covariates]
  X <- sparse.model.matrix(~ . - 1, data = X_df)
  Si <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  n_subgroups <- ncol(Si)

  cat("  Creating interactions...\n")

  Xi_Si_list <- lapply(1:n_subgroups, function(k) {
    si_k <- as.vector(Si[, k])
    X_k <- X * si_k
    colnames(X_k) <- paste0(colnames(X), ":", colnames(Si)[k])
    X_k
  })

  Xi_Si <- do.call(cbind, Xi_Si_list)
  rm(Xi_Si_list); invisible(gc())

  design_matrix <- cbind(X, Si, Xi_Si)

  penalty.factor <- c(
    rep(0, ncol(X) + ncol(Si)),
    rep(1, ncol(Xi_Si))
  )

  cat("  Running LASSO...\n")

  lasso_fit <- cv.glmnet(
    design_matrix, dt_sample$Z,
    family = "binomial",
    alpha = 1,
    penalty.factor = penalty.factor,
    nfolds = 3,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 2000
  )

  selected_coef <- coef(lasso_fit, s = "lambda.min")
  selected_vars <- rownames(selected_coef)[selected_coef[, 1] != 0]
  selected_vars <- selected_vars[selected_vars != "(Intercept)"]
  selected_interactions <- selected_vars[grepl(":", selected_vars)]

  cat(sprintf("  Selected %d interactions\n", length(selected_interactions)))

  union_vars <- c(colnames(X), colnames(Si), selected_interactions)

  return(list(
    union_vars = union_vars,
    selected_interactions = selected_interactions,
    base_vars = c(colnames(X), colnames(Si))
  ))
}

# ==============================================================================
# Propensity Score Estimation with Selected Variables
# ==============================================================================
analyze_with_selected_vars <- function(imp_data, imp_num,
                                       covariates, subgroup_var, treatment,
                                       union_vars, tau = 1095) {

  cat(sprintf("\n=== Processing Imputation %d ===\n", imp_num))

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(target_sample_size / n_groups)

  cat(sprintf("  Target sample: %d (stratified by Si × Z)\n", target_sample_size))

  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]

  cat(sprintf("  Actual sample: %d\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample <- sparse.model.matrix(~ . - 1, data = X_df_sample)
  Si_sample <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  interaction_names <- union_vars[grepl(":", union_vars)]

  if(length(interaction_names) > 0 && length(interaction_names) < 800) {
    cat(sprintf("  Creating %d interactions (sample)...\n", length(interaction_names)))

    n_subgroups <- ncol(Si_sample)
    Xi_Si_list <- list()

    for(k in 1:n_subgroups) {
      si_k <- as.vector(Si_sample[, k])
      X_k <- X_sample * si_k
      colnames(X_k) <- paste0(colnames(X_sample), ":", colnames(Si_sample)[k])

      selected_cols <- intersect(colnames(X_k), interaction_names)
      if(length(selected_cols) > 0) {
        Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols, drop = FALSE]
      }
    }

    if(length(Xi_Si_list) > 0) {
      Xi_Si_sample <- do.call(cbind, Xi_Si_list)
      design_sample <- cbind(X_sample, Si_sample, Xi_Si_sample)
      rm(Xi_Si_list); invisible(gc())
    } else {
      design_sample <- cbind(X_sample, Si_sample)
    }
  } else {
    cat("  Too many interactions, using main effects only\n")
    design_sample <- cbind(X_sample, Si_sample)
  }

  all_vars <- colnames(design_sample)
  selected_cols <- intersect(union_vars, all_vars)
  design_selected_sample <- design_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d × %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model...\n")

  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family = "binomial",
    alpha = 0,
    lambda = 0.01,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 1000
  )

  cat("  Predicting on full data...\n")

  chunk_size <- 200000
  n_chunks <- ceiling(n_total / chunk_size)
  ps_all <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) {
      cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))
    }

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx <- min(chunk_i * chunk_size, n_total)

    dt_chunk <- dt[start_idx:end_idx]

    X_df_chunk <- dt_chunk[, ..covariates]
    X_chunk <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    Si_chunk <- sparse.model.matrix(~ Si - 1, data = dt_chunk)

    if(length(interaction_names) > 0 && length(interaction_names) < 800) {
      n_subgroups <- ncol(Si_chunk)
      Xi_Si_list <- list()

      for(k in 1:n_subgroups) {
        si_k <- as.vector(Si_chunk[, k])
        X_k <- X_chunk * si_k
        colnames(X_k) <- paste0(colnames(X_chunk), ":", colnames(Si_chunk)[k])

        selected_cols_chunk <- intersect(colnames(X_k), interaction_names)
        if(length(selected_cols_chunk) > 0) {
          Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols_chunk, drop = FALSE]
        }
      }

      if(length(Xi_Si_list) > 0) {
        Xi_Si_chunk <- do.call(cbind, Xi_Si_list)
        design_chunk <- cbind(X_chunk, Si_chunk, Xi_Si_chunk)
        rm(Xi_Si_list)
      } else {
        design_chunk <- cbind(X_chunk, Si_chunk)
      }
    } else {
      design_chunk <- cbind(X_chunk, Si_chunk)
    }

    design_chunk <- design_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, Si_chunk, design_chunk)
    invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = .(Si, Z)]

  data <- dt[, .(Si, Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data, X = NULL, Si = NULL))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  cat(sprintf("  Cox models...\n"))

  # Subgroup-Specific Models
  subgroup_levels <- levels(data$Si)
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  # Interaction Model
  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)
  interaction_coefs <- grep("Z:Si", rownames(interaction_summary$coefficients), value = TRUE)

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
# ==============================================================================
# RMST and Risk Difference Calculation
# ==============================================================================

calculate_rmst_weighted <- function(time, status, weights, tau) {

  ord <- order(time)
  time <- time[ord]
  status <- status[ord]
  weights <- weights[ord]

  w_atrisk <- rev(cumsum(rev(weights)))
  w_event  <- status * weights

  w_surv <- cumprod(1 - w_event / pmax(w_atrisk, 1e-10))
  w_surv <- c(1, w_surv)
  time_ext <- c(0, time)

  idx <- time_ext <= tau
  time_use <- time_ext[idx]
  surv_use <- w_surv[idx]

  if (max(time_use) < tau) {
    time_use <- c(time_use, tau)
    surv_use <- c(surv_use, tail(surv_use, 1))
  }

  rmst <- sum(diff(time_use) * head(surv_use, -1))
  return(rmst)
}

calculate_rmst_point <- function(results, tau = 1095, verbose = TRUE) {

  if (verbose) cat("  RMST (weighted KM, point estimates only)...\n")

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  results_list <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if (nrow(data_sg) <= 10) return(NULL)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl  <- data_sg[data_sg$Z == 0, ]

    if (nrow(data_treat) <= 10 || nrow(data_ctrl) <= 10) return(NULL)

    # Weighted Kaplan–Meier
    fit_treat <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_treat,
      weights = data_treat$w
    )
    fit_ctrl <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_ctrl,
      weights = data_ctrl$w
    )

    # RMST
    rmst_treat <- tryCatch(
      calculate_rmst_weighted(
        time = data_treat$Survival_Time,
        status = data_treat$Outcome,
        weights = data_treat$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_ctrl <- tryCatch(
      calculate_rmst_weighted(
        time = data_ctrl$Survival_Time,
        status = data_ctrl$Outcome,
        weights = data_ctrl$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_diff <- rmst_treat - rmst_ctrl

    # Risk Differences
    time_points <- c(365, 730, 1095)
    surv_at_times <- lapply(time_points, function(t) {
      sum_treat <- summary(fit_treat, times = t, extend = TRUE)
      sum_ctrl  <- summary(fit_ctrl,  times = t, extend = TRUE)

      surv_t <- if (length(sum_treat$surv) > 0) sum_treat$surv else NA_real_
      surv_c <- if (length(sum_ctrl$surv) > 0) sum_ctrl$surv else NA_real_
      se_t   <- if (length(sum_treat$std.err) > 0) sum_treat$std.err else NA_real_
      se_c   <- if (length(sum_ctrl$std.err) > 0) sum_ctrl$std.err else NA_real_

      risk_t <- 1 - surv_t
      risk_c <- 1 - surv_c
      risk_diff <- risk_c - risk_t

      risk_diff_var <- if (!is.na(se_t) && !is.na(se_c)) {
        se_t^2 + se_c^2
      } else NA_real_

      baseline_risk_var <- if (!is.na(se_c)) {
        se_c^2
      } else NA_real_

      list(
        time = t,
        surv_treat = surv_t,
        se_treat = se_t,
        surv_ctrl = surv_c,
        se_ctrl = se_c,
        risk_treat = risk_t,
        risk_ctrl = risk_c,
        risk_diff = risk_diff,
        risk_diff_var = risk_diff_var,
        baseline_risk = risk_c,
        baseline_risk_var = baseline_risk_var
      )
    })

    list(
      rmst = list(
        subgroup = sg,
        rmst_treat = rmst_treat,
        rmst_ctrl = rmst_ctrl,
        rmst_diff = rmst_diff,
        tau = tau
      ),
      surv_prob = list(
        subgroup = sg,
        time_points = surv_at_times
      )
    )
  })

  results_list <- results_list[!sapply(results_list, is.null)]

  rmst_results <- lapply(results_list, function(x) x$rmst)
  names(rmst_results) <- sapply(rmst_results, function(x) x$subgroup)

  surv_prob_results <- lapply(results_list, function(x) x$surv_prob)
  names(surv_prob_results) <- sapply(surv_prob_results, function(x) x$subgroup)

  return(list(
    rmst_results = rmst_results,
    surv_prob_results = surv_prob_results
  ))
}

bootstrap_rmst_diff <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  set.seed(seed)

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  base_est <- calculate_rmst_point(results, tau = tau, verbose = TRUE)
  rmst_point <- base_est$rmst_results

  use_parallel <- !is.null(n_cores) && n_cores > 1

  if (use_parallel) {
    if (n_cores == "auto") {
      n_cores <- max(1, parallel::detectCores() - 1)
    }

    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac,
        ", cores =", n_cores, ")...\n")

    cl <- parallel::makeCluster(n_cores)

    parallel::clusterExport(cl,
                           c("calculate_rmst_point", "calculate_rmst_weighted",
                             "data_by_sg", "tau", "subsample_frac"),
                           envir = environment())

    parallel::clusterEvalQ(cl, library(survival))

    boot_results <- parallel::parLapply(cl, 1:B, function(b) {

      # Within-subgroup subsample bootstrap
      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- tryCatch(
        calculate_rmst_point(results_boot, tau = tau, verbose = FALSE),
        error = function(e) NULL
      )

      if (is.null(boot_est)) return(NULL)

      sapply(boot_est$rmst_results, function(x) x$rmst_diff)
    })

    parallel::stopCluster(cl)

    boot_results <- boot_results[!sapply(boot_results, is.null)]
    boot_mat_data <- do.call(rbind, boot_results)

    boot_mat <- lapply(seq_along(subgroup_levels), function(i) {
      boot_mat_data[, i]
    })
    names(boot_mat) <- subgroup_levels

  } else {
    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac, ")...\n")

    boot_mat <- lapply(subgroup_levels, function(sg) {
      numeric(B)
    })
    names(boot_mat) <- subgroup_levels

    for (b in seq_len(B)) {
      if (b %% 20 == 0) cat("    Iteration", b, "of", B, "\n")

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- calculate_rmst_point(results_boot, tau = tau, verbose = FALSE)
      rmst_boot <- boot_est$rmst_results

      for (sg in names(rmst_boot)) {
        boot_mat[[sg]][b] <- rmst_boot[[sg]]$rmst_diff
      }
    }
  }

  out <- lapply(rmst_point, function(x) {
    sg <- x$subgroup
    diffs <- boot_mat[[sg]]
    diffs <- diffs[is.finite(diffs)]

    if (length(diffs) < 10) {
      se <- NA_real_
      ci_lower <- NA_real_
      ci_upper <- NA_real_
    } else {
      se <- sd(diffs)
      # Percentile CI (더 robust)
      ci_lower <- quantile(diffs, 0.025, na.rm = TRUE)
      ci_upper <- quantile(diffs, 0.975, na.rm = TRUE)
    }

    c(x, list(
      se_rmst_diff = se,
      ci_lower = ci_lower,
      ci_upper = ci_upper,
      B = length(diffs),
      subsample_frac = subsample_frac
    ))
  })
  names(out) <- names(rmst_point)

  return(list(
    rmst_results = out,
    surv_prob_results = base_est$surv_prob_results
  ))
}

calculate_rmst_fast <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  bootstrap_rmst_diff(
    results = results,
    tau = tau,
    B = B,
    subsample_frac = subsample_frac,
    seed = seed,
    n_cores = n_cores
  )
}

In [ ]:
# ==============================================================================
# Pool Main Effects
# ==============================================================================
pool_main_effects <- function(all_results) {

  m <- length(all_results)

  # Pool Treatment Effect
  treat_coefs <- sapply(all_results, function(x) x$main_effects$treatment_coef)
  treat_ses <- sapply(all_results, function(x) x$main_effects$treatment_se)

  Q_bar_treat <- mean(treat_coefs)
  U_bar_treat <- mean(treat_ses^2)
  B_treat <- var(treat_coefs)
  T_treat <- U_bar_treat + (1 + 1/m) * B_treat
  se_treat <- sqrt(T_treat)

  df_treat <- if(B_treat < 1e-10) Inf else {
    (m - 1) * (1 + U_bar_treat / ((1 + 1/m) * B_treat))^2
  }

  pooled_main_effects <- list(
    treatment = list(
      coef = Q_bar_treat,
      se = se_treat,
      hr = exp(Q_bar_treat),
      ci_lower = exp(Q_bar_treat - qt(0.975, df_treat) * se_treat),
      ci_upper = exp(Q_bar_treat + qt(0.975, df_treat) * se_treat),
      pvalue = 2 * pt(-abs(Q_bar_treat / se_treat), df = df_treat),
      df = df_treat
    )
  )

  # Pool All Subgroup Coefficients
  if (!is.null(all_results[[1]]$main_effects$subgroup_coefs)) {

    common_names <- Reduce(
      intersect,
      lapply(all_results, function(res)
        vapply(res$main_effects$subgroup_coefs, `[[`, character(1), "name"))
    )

    pooled_subgroup_coefs <- lapply(common_names, function(sg_name) {

      coefs <- sapply(all_results, function(res) {
        idx <- which(vapply(res$main_effects$subgroup_coefs,
                            `[[`, character(1), "name") == sg_name)
        if (length(idx) == 0) NA_real_ else
          as.numeric(res$main_effects$subgroup_coefs[[idx]]$coef)
      })

      ses <- sapply(all_results, function(res) {
        idx <- which(vapply(res$main_effects$subgroup_coefs,
                            `[[`, character(1), "name") == sg_name)
        if (length(idx) == 0) NA_real_ else
          as.numeric(res$main_effects$subgroup_coefs[[idx]]$se)
      })

      valid <- is.finite(coefs) & is.finite(ses)
      coefs <- coefs[valid]
      ses   <- ses[valid]

      if (length(coefs) == 0L) return(NULL)

      m_valid        <- length(coefs)
      Q_bar_subgroup <- mean(coefs)
      U_bar_subgroup <- mean(ses^2)
      B_subgroup     <- var(coefs)
      T_subgroup     <- U_bar_subgroup + (1 + 1/m_valid) * B_subgroup
      se_subgroup    <- sqrt(T_subgroup)
      df_subgroup    <- if (B_subgroup < 1e-10) Inf else {
        (m_valid - 1) * (1 + U_bar_subgroup / ((1 + 1/m_valid) * B_subgroup))^2
      }

      list(
        name    = sg_name,
        coef    = Q_bar_subgroup,
        se      = se_subgroup,
        hr      = exp(Q_bar_subgroup),
        ci_lower = exp(Q_bar_subgroup - qt(0.975, df_subgroup) * se_subgroup),
        ci_upper = exp(Q_bar_subgroup + qt(0.975, df_subgroup) * se_subgroup),
        pvalue  = 2 * pt(-abs(Q_bar_subgroup / se_subgroup), df = df_subgroup),
        df      = df_subgroup
      )
    })

    pooled_subgroup_coefs <- pooled_subgroup_coefs[!vapply(pooled_subgroup_coefs, is.null, logical(1))]

    pooled_main_effects$subgroup_coefs <- pooled_subgroup_coefs
    pooled_main_effects$subgroup_var   <- all_results[[1]]$main_effects$subgroup_var
  }

  return(pooled_main_effects)
}

# ==============================================================================
# Pool Subgroup-Specific Treatment Effects
# ==============================================================================
pool_subgroup_results <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  m <- length(all_results)

  pooled_subgroups <- lapply(subgroup_names, function(sg) {
    coefs <- sapply(all_results, function(x) x$subgroup_results[[sg]]$coef)
    ses <- sapply(all_results, function(x) x$subgroup_results[[sg]]$se_robust)

    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m - 1) * (1 + U_bar / ((1 + 1/m) * B))^2

    list(
      subgroup = sg,
      coef = Q_bar,
      se = se_pooled,
      hr = exp(Q_bar),
      ci_lower = exp(Q_bar - qt(0.975, df) * se_pooled),
      ci_upper = exp(Q_bar + qt(0.975, df) * se_pooled),
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })

  names(pooled_subgroups) <- subgroup_names
  return(pooled_subgroups)
}

In [ ]:
# ==============================================================================
# Pool Interaction Terms
# ==============================================================================
pool_interactions <- function(all_results) {

  interaction_terms <- sapply(all_results[[1]]$interaction_results, function(x) x$term)

  pooled_interactions <- lapply(interaction_terms, function(term) {
    coefs <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$coef else NA
    })

    ses <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$se_robust else NA
    })

    valid_idx <- !is.na(coefs) & !is.na(ses)
    coefs <- coefs[valid_idx]
    ses <- ses[valid_idx]

    if(length(coefs) == 0) return(NULL)

    m_valid <- length(coefs)
    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      term = term,
      coef = Q_bar,
      se = se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })

  pooled_interactions <- pooled_interactions[!sapply(pooled_interactions, is.null)]
  return(pooled_interactions)
}

# ==============================================================================
# Pool RMST Differences
# ==============================================================================
pool_rmst <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)

  pooled_rmst <- lapply(subgroup_names, function(sg) {
    rmst_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$rmst_diff)
    se_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$se_rmst_diff)

    valid_idx <- !is.na(rmst_diffs) & !is.na(se_diffs)
    rmst_diffs <- rmst_diffs[valid_idx]
    se_diffs <- se_diffs[valid_idx]

    if(length(rmst_diffs) == 0) return(NULL)

    m_valid <- length(rmst_diffs)
    Q_bar <- mean(rmst_diffs)
    U_bar <- mean(se_diffs^2)
    B <- var(rmst_diffs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      subgroup = sg,
      rmst_diff = Q_bar,
      se = se_pooled,
      ci_lower = Q_bar - qt(0.975, df) * se_pooled,
      ci_upper = Q_bar + qt(0.975, df) * se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      tau = all_results[[1]]$rmst_results[[sg]]$tau
    )
  })

  names(pooled_rmst) <- subgroup_names
  pooled_rmst <- pooled_rmst[!sapply(pooled_rmst, is.null)]

  return(pooled_rmst)
}

In [ ]:
# ==============================================================================
# Pool Absolute Risk Differences and Baseline Risks
# ==============================================================================
pool_survival_probs <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  time_points <- c(365, 730, 1095)

  pooled_surv_probs <- lapply(subgroup_names, function(sg) {
    time_results <- lapply(seq_along(time_points), function(t_idx) {

      risk_diffs <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff else NA
      })

      risk_diff_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff_var else NA
      })

      baseline_risks <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk else NA
      })

      baseline_risk_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk_var else NA
      })

      valid_idx_rd <- !is.na(risk_diffs) & !is.na(risk_diff_vars)
      risk_diffs_valid <- risk_diffs[valid_idx_rd]
      risk_diff_vars_valid <- risk_diff_vars[valid_idx_rd]

      valid_idx_br <- !is.na(baseline_risks) & !is.na(baseline_risk_vars)
      baseline_risks_valid <- baseline_risks[valid_idx_br]
      baseline_risk_vars_valid <- baseline_risk_vars[valid_idx_br]

      if(length(risk_diffs_valid) == 0) return(NULL)

      # Rubin's Rules for Risk Difference
      m_valid_rd <- length(risk_diffs_valid)
      Q_bar_rd <- mean(risk_diffs_valid)
      U_bar_rd <- mean(risk_diff_vars_valid)
      B_rd <- var(risk_diffs_valid)
      T_rd <- U_bar_rd + (1 + 1/m_valid_rd) * B_rd
      se_pooled_rd <- sqrt(T_rd)

      df_rd <- if(B_rd < 1e-10) Inf else {
        (m_valid_rd - 1) * (1 + U_bar_rd / ((1 + 1/m_valid_rd) * B_rd))^2
      }

      ci_lower_rd <- Q_bar_rd - qt(0.975, df_rd) * se_pooled_rd
      ci_upper_rd <- Q_bar_rd + qt(0.975, df_rd) * se_pooled_rd
      pvalue_rd <- 2 * pt(-abs(Q_bar_rd / se_pooled_rd), df = df_rd)

      # Rubin's Rules for Baseline Risk
      baseline_result <- if(length(baseline_risks_valid) > 0) {
        m_valid_br <- length(baseline_risks_valid)
        Q_bar_br <- mean(baseline_risks_valid)
        U_bar_br <- mean(baseline_risk_vars_valid)
        B_br <- var(baseline_risks_valid)
        T_br <- U_bar_br + (1 + 1/m_valid_br) * B_br
        se_pooled_br <- sqrt(T_br)

        df_br <- if(B_br < 1e-10) Inf else {
          (m_valid_br - 1) * (1 + U_bar_br / ((1 + 1/m_valid_br) * B_br))^2
        }

        ci_lower_br <- Q_bar_br - qt(0.975, df_br) * se_pooled_br
        ci_upper_br <- Q_bar_br + qt(0.975, df_br) * se_pooled_br
        pvalue_br <- 2 * pt(-abs(Q_bar_br / se_pooled_br), df = df_br)

        list(
          baseline_risk = Q_bar_br,
          baseline_risk_se = se_pooled_br,
          baseline_risk_ci_lower = ci_lower_br,
          baseline_risk_ci_upper = ci_upper_br,
          baseline_risk_df = df_br
        )
      } else {
        list(
          baseline_risk = NA,
          baseline_risk_se = NA,
          baseline_risk_ci_lower = NA,
          baseline_risk_ci_upper = NA,
          baseline_risk_df = NA
        )
      }

      c(
        list(
          time = time_points[t_idx],
          risk_diff = Q_bar_rd,
          se = se_pooled_rd,
          ci_lower = ci_lower_rd,
          ci_upper = ci_upper_rd,
          pvalue = pvalue_rd,
          df = df_rd,
          within_var = U_bar_rd,
          between_var = B_rd,
          fmi = (1 + 1/m_valid_rd) * B_rd / T_rd
        ),
        baseline_result
      )
    })

    time_results <- time_results[!sapply(time_results, is.null)]
    list(subgroup = sg, time_results = time_results)
  })

  names(pooled_surv_probs) <- subgroup_names
  return(pooled_surv_probs)
}

# ==============================================================================
# Execute All Pooling Functions
# ==============================================================================
pool_results <- function(all_results) {
    cat("\n", rep("=", 80), "\n")
    cat("EXECUTING RUBIN'S POOLING...\n")
    cat(rep("=", 80), "\n\n")

    # Pool Subgroup-Specific Results
    cat("Step 2/5: Pooling subgroup-specific treatment effects...\n")
    pooled_subgroups <- pool_subgroup_results(all_results)
    cat(sprintf(" ✓ %d subgroups pooled\n\n", length(pooled_subgroups)))

    # Pool Interaction Terms
    cat("Step 3/5: Pooling interaction terms...\n")
    pooled_interactions <- pool_interactions(all_results)
    cat(sprintf(" ✓ %d interaction terms pooled\n\n", length(pooled_interactions)))

    # Pool RMST Results
    cat("Step 4/5: Pooling RMST differences...\n")
    pooled_rmst <- pool_rmst(all_results)
    cat(sprintf(" ✓ RMST for %d subgroups pooled\n\n", length(pooled_rmst)))

    # Pool Survival Probabilities and Risk Differences
    cat("Step 5/5: Pooling absolute risk differences and baseline risks...\n")
    pooled_surv_probs <- pool_survival_probs(all_results)
    cat(sprintf(" ✓ Risk differences for %d subgroups pooled\n\n", length(pooled_surv_probs)))

    # Combine all results
    pooled_results <- list(
        main_effects = pooled_main_effects,
        subgroup_results = pooled_subgroups,
        interaction_results = pooled_interactions,
        rmst_results = pooled_rmst,
        surv_prob_results = pooled_surv_probs
    )

    return(pooled_results)
}

In [ ]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
subgroup_var <- "Race"

race_mapping <- c(
  "1067364" = "White",
  "1067319" = "Black",
  "1067385" = "Others",
  "1066466" = "Others",
  "1067338" = "Others",
  "1067294" = "Asian"
)

covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
  "HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
  "HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
  "HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
  "HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant",
  "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone", "HasOtherAntiHypertensive",
  "HasHospitalization", "Ethnicity", "Age", "CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
  "NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
  "NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Sex", "CalendarYear",
  "AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HemoglobinValue",
  "LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
  "SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue", "HbA1cValue",
  "AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
  "AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)
covariates <- setdiff(covariates, subgroup_var)
treatment <- "FirstMedGroup"

imputed_t2d_list <- lapply(imputed_t2d_list, function(dt) {
  dt <- as.data.frame(dt)
  dt$Race <- as.character(dt$Race)
  dt <- dt[dt$Race != "Unknown", ]
  dt$Race <- ifelse(dt$Race %in% names(race_mapping),
                    race_mapping[dt$Race],
                    dt$Race)

  dt$Race <- factor(dt$Race, levels = c("White", "Black", "Asian", "Others"))

  return(dt)
})

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
start_time <- Sys.time()

cat("\n", rep("=", 80), "\n")
cat("PHASE 1: LASSO (First Imputation Only, 50K Stratified)\n")
cat(rep("=", 80), "\n")

lasso_result <- perform_lasso_first_only(
  imp_data = imputed_t2d_list[[1]],
  covariates = covariates,
  subgroup_var = subgroup_var,
  treatment = treatment,
  sample_size = 50000
)

union_vars <- lasso_result$union_vars

cat("\n", rep("=", 80), "\n")
cat(sprintf("Selected variables: %d (including %d interactions)\n",
            length(union_vars), length(lasso_result$selected_interactions)))
cat(rep("=", 80), "\n")

# ==============================================================================
# Full Analysis
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 2: Full Analysis (All 5 Imputations - SEQUENTIAL)\n")
cat(rep("=", 80), "\n")

all_results <- list()

for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Processing Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_with_selected_vars(
    imp_data = imputed_t2d_list[[i]],
    imp_num = i,
    covariates = covariates,
    subgroup_var = subgroup_var,
    treatment = treatment,
    union_vars = union_vars,
    tau = 1095
  )

  cox_results <- complete_analysis_fast(
    data_list,
    imp_num = i,
    subgroup_var = subgroup_var,
    tau = 1095
  )

  rmst_results <- calculate_rmst_fast(
    cox_results,
    tau = 1095,
    B = 100,
    subsample_frac = 0.3,
    n_cores = 4
  )

  all_results[[i]] <- list(
    main_effects = cox_results$main_effects,
    subgroup_results = cox_results$subgroup_results,
    interaction_results = cox_results$interaction_results,
    data_by_sg = if(i == 1) cox_results$data_by_sg else NULL,
    rmst_results = rmst_results$rmst_results,
    surv_prob_results = rmst_results$surv_prob_results
  )

  cat("Imputation", i, "completed!\n")

  invisible(gc())
}

cat("\nAll imputations completed!\n")

# ==============================================================================
# Rubin's Pooling + Summary
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results(all_results)

end_time <- Sys.time()
elapsed <- as.numeric(end_time - start_time, units = "mins")

cat("\n", rep("=", 80), "\n")
cat(sprintf("Total Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# Display Results
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("FINAL POOLED RESULTS (3-Year Follow-up)\n")
cat(sprintf("Subgroup Analysis by: %s\n", subgroup_var))
cat(rep("=", 80), "\n\n")

# Subgroup-Specific HRs
cat("\n2. SUBGROUP-SPECIFIC TREATMENT EFFECTS (Hazard Ratios)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$subgroup_results)) {
  res <- pooled_results$subgroup_results[[sg]]
  cat(sprintf("Subgroup %s: HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
              sg, res$hr, res$ci_lower, res$ci_upper, res$pvalue))
}

# Interaction
cat(sprintf("\n3. INTERACTION TEST (Treatment × %s)\n", subgroup_var))
cat(rep("-", 80), "\n")
for(res in pooled_results$interaction_results) {
  cat(sprintf("%s: Coef = %.4f (SE = %.4f), p = %.4f\n",
              res$term, res$coef, res$se, res$pvalue))
  if(res$pvalue < 0.05) {
    cat("  → SIGNIFICANT heterogeneity on relative scale\n")
  } else {
    cat("  → No significant heterogeneity on relative scale\n")
  }
}

# Overall interaction interpretation
any_interaction_sig <- any(sapply(pooled_results$interaction_results,
                                  function(x) x$pvalue < 0.05))
cat("\n")
if(any_interaction_sig) {
  cat("OVERALL: Significant treatment effect heterogeneity detected\n")
} else {
  cat("OVERALL: No significant treatment effect heterogeneity\n")
}

# RMST
cat("\n4. RESTRICTED MEAN SURVIVAL TIME (3-year RMST)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$rmst_results)) {
  res <- pooled_results$rmst_results[[sg]]
  cat(sprintf("Subgroup %s: RMST Diff = %.1f days (95%% CI: %.1f-%.1f), p = %.4f\n",
              sg, res$rmst_diff, res$ci_lower, res$ci_upper, res$pvalue))
  cat(sprintf("  → Treatment %s %.1f days of event-free survival\n",
              if(res$rmst_diff > 0) "extends by" else "reduces by",
              abs(res$rmst_diff)))
}

# Absolute Risk Differences
cat("\n5. ABSOLUTE RISK DIFFERENCES\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$surv_prob_results)) {
  cat(sprintf("\nSubgroup %s:\n", sg))
  time_res <- pooled_results$surv_prob_results[[sg]]$time_results
  for(tr in time_res) {
    cat(sprintf("  At %.1f years: Risk Diff = %.4f (95%% CI: %.4f-%.4f)\n",
                tr$time/365.25, tr$risk_diff, tr$ci_lower, tr$ci_upper))
    cat(sprintf("    SE = %.4f, p = %.4f (%.2f%% reduction)\n",
                tr$se, tr$pvalue, tr$risk_diff * 100))
    cat(sprintf("    FMI = %.3f (%.1f%% of variance from imputation uncertainty)\n",
                tr$fmi, tr$fmi * 100))
  }
}

cat("\n", rep("=", 80), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# SUBGROUP SAMPLE STATISTICS
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("6. SUBGROUP SAMPLE STATISTICS\n")
cat(rep("=", 80), "\n")

cat("\n6-1. ORIGINAL DATA (Before Overlap Weighting)\n")
cat(rep("-", 80), "\n\n")

cat("A. Overall Dataset Statistics\n")
cat(rep("-", 40), "\n")

overall_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  total_n <- nrow(dt)

  age_counts <- dt[, .N, by = Si]

  list(
    total_n = total_n,
    age_counts = age_counts
  )
})

avg_total_n <- round(mean(sapply(overall_stats, function(x) x$total_n)))

age_group_levels <- unique(overall_stats[[1]]$age_counts$Si)

avg_age_counts <- lapply(age_group_levels, function(sg) {
  counts <- sapply(overall_stats, function(x) {
    x$age_counts[x$age_counts$Si == sg, ]$N
  })
  list(
    subgroup = as.character(sg),
    count = round(mean(counts))
  )
})

cat(sprintf("Total Sample Size: n = %d\n\n", avg_total_n))
cat("Age Group Distribution:\n")
for(ac in avg_age_counts) {
  cat(sprintf("  %s: n = %d (%.1f%%)\n",
              ac$subgroup, ac$count, ac$count/avg_total_n*100))
}
cat("\n")

cat("\nB. Statistics by Age Subgroup\n")
cat(rep("-", 40), "\n\n")

original_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  stats_by_sg <- dt[, .(
    n_total = .N,
    n_treat = sum(Z == 1),
    n_ctrl = sum(Z == 0),
    events_total = sum(Outcome),
    events_treat = sum(Outcome[Z == 1]),
    events_ctrl = sum(Outcome[Z == 0])
  ), by = Si]

  setDF(stats_by_sg)
  stats_by_sg
})

avg_original_stats <- lapply(unique(original_stats[[1]]$Si), function(sg) {
  n_total_vec <- sapply(original_stats, function(x) x$n_total[x$Si == sg])
  n_treat_vec <- sapply(original_stats, function(x) x$n_treat[x$Si == sg])
  n_ctrl_vec <- sapply(original_stats, function(x) x$n_ctrl[x$Si == sg])
  events_total_vec <- sapply(original_stats, function(x) x$events_total[x$Si == sg])
  events_treat_vec <- sapply(original_stats, function(x) x$events_treat[x$Si == sg])
  events_ctrl_vec <- sapply(original_stats, function(x) x$events_ctrl[x$Si == sg])

  list(
    subgroup = as.character(sg),
    n_total = round(mean(n_total_vec)),
    n_treat = round(mean(n_treat_vec)),
    n_ctrl = round(mean(n_ctrl_vec)),
    events_total = round(mean(events_total_vec)),
    events_treat = round(mean(events_treat_vec)),
    events_ctrl = round(mean(events_ctrl_vec))
  )
})


for(stats in avg_original_stats) {
  cat(sprintf("Age Group: %s\n", stats$subgroup))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size:\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat("\n6-2. AFTER OVERLAP WEIGHTING (Pseudo Population)\n")
cat(rep("-", 80), "\n\n")

subgroup_levels <- names(all_results[[1]]$data_by_sg)

avg_weighted_stats <- lapply(subgroup_levels, function(sg) {

  stats_list <- lapply(1:5, function(i) {
    data_sg <- all_results[[i]]$data_by_sg[[sg]]

    if(is.null(data_sg)) return(NULL)

    n_total <- nrow(data_sg)
    n_events <- sum(data_sg$Outcome)
    ess_total <- sum(data_sg$w)^2 / sum(data_sg$w^2)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl <- data_sg[data_sg$Z == 0, ]

    n_treat <- nrow(data_treat)
    n_ctrl <- nrow(data_ctrl)
    events_treat <- sum(data_treat$Outcome)
    events_ctrl <- sum(data_ctrl$Outcome)
    ess_treat <- if(n_treat > 0) sum(data_treat$w)^2 / sum(data_treat$w^2) else 0
    ess_ctrl <- if(n_ctrl > 0) sum(data_ctrl$w)^2 / sum(data_ctrl$w^2) else 0

    list(
      n_total = n_total,
      n_treat = n_treat,
      n_ctrl = n_ctrl,
      events_total = n_events,
      events_treat = events_treat,
      events_ctrl = events_ctrl,
      ess_total = ess_total,
      ess_treat = ess_treat,
      ess_ctrl = ess_ctrl
    )
  })

  stats_list <- stats_list[!sapply(stats_list, is.null)]

  list(
    subgroup = sg,
    n_total = round(mean(sapply(stats_list, function(x) x$n_total))),
    n_treat = round(mean(sapply(stats_list, function(x) x$n_treat))),
    n_ctrl = round(mean(sapply(stats_list, function(x) x$n_ctrl))),
    events_total = round(mean(sapply(stats_list, function(x) x$events_total))),
    events_treat = round(mean(sapply(stats_list, function(x) x$events_treat))),
    events_ctrl = round(mean(sapply(stats_list, function(x) x$events_ctrl))),
    ess_total = round(mean(sapply(stats_list, function(x) x$ess_total)), 1),
    ess_treat = round(mean(sapply(stats_list, function(x) x$ess_treat)), 1),
    ess_ctrl = round(mean(sapply(stats_list, function(x) x$ess_ctrl)), 1)
  )
})
names(avg_weighted_stats) <- subgroup_levels

for(sg in names(avg_weighted_stats)) {
  stats <- avg_weighted_stats[[sg]]

  cat(sprintf("Age Group: %s\n", sg))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size (Original):\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Effective Sample Size (ESS):\n"))
  cat(sprintf("    Total:     ESS = %.1f\n", stats$ess_total))
  cat(sprintf("    Treatment: ESS = %.1f\n", stats$ess_treat))
  cat(sprintf("    Control:   ESS = %.1f\n", stats$ess_ctrl))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

In [ ]:
data_by_sg <- all_results[[1]]$data_by_sg

race_groups <- names(data_by_sg)
race_labels <- c("White", "Black", "Asian", "Other")
colors <- c("#E41A1C", "#377EB8", "#4DAF4A", "#FF7F00")  # Red, Blue, Green, Orange

fit_list <- list()
for(i in seq_along(race_groups)) {
  sg <- race_groups[i]
  data_sg <- data_by_sg[[sg]]

  data_treat <- data_sg[data_sg$Z == 1, ]
  data_ctrl <- data_sg[data_sg$Z == 0, ]

  fit_list[[i]] <- list(
    treat = survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_treat, weights = w),
    ctrl = survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_ctrl, weights = w)
  )
}

image_path <- file.path(artifacts_images_dir, "t2d_secondary_primary_endpoint_race.png")

png(filename = image_path, width = 9.5, height = 7, units = "in", res = 300)

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.20),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.20, by = 0.05), col = "gray50", lwd = 0.5)

# Plot lines for each race group
for(i in seq_along(race_groups)) {
  lines(c(0, fit_list[[i]]$treat$time / 30.44),
        c(0, 1 - fit_list[[i]]$treat$surv),
        col = colors[i], lty = 1, lwd = 2.5)
  lines(c(0, fit_list[[i]]$ctrl$time / 30.44),
        c(0, 1 - fit_list[[i]]$ctrl$surv),
        col = colors[i], lty = 3, lwd = 2.5)
}

# Legend
# legend_labels <- c()
# legend_cols <- c()
# legend_ltys <- c()

# for(i in seq_along(race_labels)) {
#   legend_labels <- c(legend_labels,
#                      paste0(race_labels[i], " - GLP-1 RAs"),
#                      paste0(race_labels[i], " - Non-GLP-1 RAs"))
#   legend_cols <- c(legend_cols, colors[i], colors[i])
#   legend_ltys <- c(legend_ltys, 1, 3)
# }

# legend("topleft",
#        legend = legend_labels,
#        col = legend_cols,
#        lty = legend_ltys,
#        lwd = 2.5,
#        bty = "n",
#        cex = 1.4,
#        ncol = 2)

dev.off()

cat("Plot saved to:", image_path, "\n")

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.20),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.20, by = 0.05), col = "gray50", lwd = 0.5)

# Plot lines for each race group
for(i in seq_along(race_groups)) {
  lines(c(0, fit_list[[i]]$treat$time / 30.44),
        c(0, 1 - fit_list[[i]]$treat$surv),
        col = colors[i], lty = 1, lwd = 2.5)
  lines(c(0, fit_list[[i]]$ctrl$time / 30.44),
        c(0, 1 - fit_list[[i]]$ctrl$surv),
        col = colors[i], lty = 3, lwd = 2.5)
}

## SMART Score

### Settings

In [ ]:
for(i in 1:5) {
  imputed_t2d_list[[i]]$YearsSinceVascEvent_original <- imputed_t2d_list[[i]]$YearsSinceVascEvent

  n_capped <- sum(imputed_t2d_list[[i]]$YearsSinceVascEvent > 50, na.rm = TRUE)

  imputed_t2d_list[[i]]$YearsSinceVascEvent <- pmin(imputed_t2d_list[[i]]$YearsSinceVascEvent, 50)
}

for(i in 1:5) {
  cat("Dataset", i, "HsCRPValue Summary:\n")
  print(summary(imputed_t2d_list[[i]]$HsCRPValue))
  cat("NA count:", sum(is.na(imputed_t2d_list[[i]]$HsCRPValue)), "\n\n")
}

cat("\n=== SMART score Calculation ===\n\n")

calc_smart_final <- function(df) {

  HDL_mmol <- df$HdlCholesterolValue / 38.67
  TC_mmol <- df$TotalCholesterolValue / 38.67

  male <- ifelse(as.character(df$Sex) == "1065406", 1, 0)

  hsCRP_mgdL <- df$HsCRPValue / 10
  log_hsCRP <- log(pmax(hsCRP_mgdL, 0.0001))

  HasSmoking <- as.numeric(as.character(df$HasSmoking))
  HasCoronaryArteryDisease <- as.numeric(as.character(df$HasCoronaryArteryDisease))
  HasCerebrovascularDisease <- as.numeric(as.character(df$HasCerebrovascularDisease))
  HasAbdominalAorticAneurysm <- as.numeric(as.character(df$HasAbdominalAorticAneurysm))
  HasPeripheralArterialDisease <- as.numeric(as.character(df$HasPeripheralArterialDisease))

  A <- -0.0850 * df$Age +
       0.00105 * (df$Age^2) +
       0.156 * male +
       0.262 * HasSmoking +
       0.00429 * df$SystolicBloodPressureValue +
       0.223 * 1 +
       0.140 * HasCoronaryArteryDisease +
       0.406 * HasCerebrovascularDisease +
       0.558 * HasAbdominalAorticAneurysm +
       0.283 * HasPeripheralArterialDisease +
       0.0229 * df$YearsSinceVascEvent +
       (-0.426) * HDL_mmol +
       0.0959 * TC_mmol +
       (-0.0532) * df$EGfrValue +
       0.000306 * (df$EGfrValue^2) +
       0.139 * log_hsCRP

  linear_pred <- A + 2.099
  hazard_ratio <- exp(pmin(pmax(linear_pred, -20), 20))
  survival_prob <- 0.81066^hazard_ratio
  smart_risk <- (1 - survival_prob) * 100

  df$SMARTscore <- pmin(pmax(smart_risk, 0), 100)
  df$A_value <- A
  df$hazard_ratio <- hazard_ratio
  df$HDL_mmol <- HDL_mmol
  df$TC_mmol <- TC_mmol

  return(df)
}

for(i in 1:5) {
  imputed_t2d_list[[i]] <- calc_smart_final(imputed_t2d_list[[i]])
}

for(i in 1:5) {
  cat("Dataset", i, ":\n")
  print(summary(imputed_t2d_list[[i]]$SMARTscore))
  cat("\n")
}

for(i in 1:5) {
  low <- sum(imputed_t2d_list[[i]]$SMARTscore <= 10, na.rm = TRUE)
  moderate <- sum(imputed_t2d_list[[i]]$SMARTscore > 10 &
                  imputed_t2d_list[[i]]$SMARTscore <= 20, na.rm = TRUE)
  high <- sum(imputed_t2d_list[[i]]$SMARTscore > 20, na.rm = TRUE)
  total <- sum(!is.na(imputed_t2d_list[[i]]$SMARTscore))

  cat("\nDataset", i, ":\n")
  cat("  Low (≤10%):", low, "(", round(100*low/total, 1), "%)\n")
  cat("  Moderate (10-20%):", moderate, "(", round(100*moderate/total, 1), "%)\n")
  cat("  High (>20%):", high, "(", round(100*high/total, 1), "%)\n")
}

### FINAL SMART

In [ ]:
if(!"SMART_risk" %in% names(imputed_t2d_list[[1]])) {
  for(i in 1:5) {
    imputed_t2d_list[[i]]$SMART_risk <- cut(
      imputed_t2d_list[[i]]$SMARTscore,
      breaks = c(0, 10, 20, 100),
      labels = c("Low", "Moderate", "High"),
      include.lowest = TRUE,
      right = FALSE
    )
  }
}

In [ ]:
setDTthreads(0)

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
perform_lasso_first_only <- function(imp_data, covariates, subgroup_var, treatment,
                                     sample_size = 50000) {

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment)
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(sample_size / n_groups)

  cat(sprintf("  Total groups (Si × Z): %d\n", n_groups))
  cat(sprintf("  Target per group: %d\n", sample_per_group))

  set.seed(12345)
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Actual sample: %d obs\n", nrow(dt_sample)))

  factor_cols <- names(dt_sample)[sapply(dt_sample, is.factor) | sapply(dt_sample, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt_sample[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  X_df <- dt_sample[, ..covariates]
  X <- sparse.model.matrix(~ . - 1, data = X_df)
  Si <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  n_subgroups <- ncol(Si)

  cat("  Creating interactions...\n")

  Xi_Si_list <- lapply(1:n_subgroups, function(k) {
    si_k <- as.vector(Si[, k])
    X_k <- X * si_k
    colnames(X_k) <- paste0(colnames(X), ":", colnames(Si)[k])
    X_k
  })

  Xi_Si <- do.call(cbind, Xi_Si_list)
  rm(Xi_Si_list); invisible(gc())

  design_matrix <- cbind(X, Si, Xi_Si)

  penalty.factor <- c(
    rep(0, ncol(X) + ncol(Si)),
    rep(1, ncol(Xi_Si))
  )

  cat("  Running LASSO...\n")

  lasso_fit <- cv.glmnet(
    design_matrix, dt_sample$Z,
    family = "binomial",
    alpha = 1,
    penalty.factor = penalty.factor,
    nfolds = 3,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 2000
  )

  selected_coef <- coef(lasso_fit, s = "lambda.min")
  selected_vars <- rownames(selected_coef)[selected_coef[, 1] != 0]
  selected_vars <- selected_vars[selected_vars != "(Intercept)"]
  selected_interactions <- selected_vars[grepl(":", selected_vars)]

  cat(sprintf("  Selected %d interactions\n", length(selected_interactions)))

  union_vars <- c(colnames(X), colnames(Si), selected_interactions)

  return(list(
    union_vars = union_vars,
    selected_interactions = selected_interactions,
    base_vars = c(colnames(X), colnames(Si))
  ))
}

# ==============================================================================
# Propensity Score Estimation with Selected Variables
# ==============================================================================
analyze_with_selected_vars <- function(imp_data, imp_num,
                                       covariates, subgroup_var, treatment,
                                       union_vars, tau = 1095) {

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(target_sample_size / n_groups)

  cat(sprintf("  Target sample: %d (stratified by Si × Z)\n", target_sample_size))

  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]

  cat(sprintf("  Actual sample: %d\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample <- sparse.model.matrix(~ . - 1, data = X_df_sample)
  Si_sample <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  interaction_names <- union_vars[grepl(":", union_vars)]

  if(length(interaction_names) > 0 && length(interaction_names) < 800) {
    cat(sprintf("  Creating %d interactions (sample)...\n", length(interaction_names)))

    n_subgroups <- ncol(Si_sample)
    Xi_Si_list <- list()

    for(k in 1:n_subgroups) {
      si_k <- as.vector(Si_sample[, k])
      X_k <- X_sample * si_k
      colnames(X_k) <- paste0(colnames(X_sample), ":", colnames(Si_sample)[k])

      selected_cols <- intersect(colnames(X_k), interaction_names)
      if(length(selected_cols) > 0) {
        Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols, drop = FALSE]
      }
    }

    if(length(Xi_Si_list) > 0) {
      Xi_Si_sample <- do.call(cbind, Xi_Si_list)
      design_sample <- cbind(X_sample, Si_sample, Xi_Si_sample)
      rm(Xi_Si_list); invisible(gc())
    } else {
      design_sample <- cbind(X_sample, Si_sample)
    }
  } else {
    cat("  Too many interactions, using main effects only\n")
    design_sample <- cbind(X_sample, Si_sample)
  }

  all_vars <- colnames(design_sample)
  selected_cols <- intersect(union_vars, all_vars)
  design_selected_sample <- design_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d × %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model...\n")

  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family = "binomial",
    alpha = 0,
    lambda = 0.01,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 1000
  )

  cat("  Predicting on full data...\n")

  chunk_size <- 200000
  n_chunks <- ceiling(n_total / chunk_size)
  ps_all <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) {
      cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))
    }

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx <- min(chunk_i * chunk_size, n_total)

    dt_chunk <- dt[start_idx:end_idx]

    X_df_chunk <- dt_chunk[, ..covariates]
    X_chunk <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    Si_chunk <- sparse.model.matrix(~ Si - 1, data = dt_chunk)

    if(length(interaction_names) > 0 && length(interaction_names) < 800) {
      n_subgroups <- ncol(Si_chunk)
      Xi_Si_list <- list()

      for(k in 1:n_subgroups) {
        si_k <- as.vector(Si_chunk[, k])
        X_k <- X_chunk * si_k
        colnames(X_k) <- paste0(colnames(X_chunk), ":", colnames(Si_chunk)[k])

        selected_cols_chunk <- intersect(colnames(X_k), interaction_names)
        if(length(selected_cols_chunk) > 0) {
          Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols_chunk, drop = FALSE]
        }
      }

      if(length(Xi_Si_list) > 0) {
        Xi_Si_chunk <- do.call(cbind, Xi_Si_list)
        design_chunk <- cbind(X_chunk, Si_chunk, Xi_Si_chunk)
        rm(Xi_Si_list)
      } else {
        design_chunk <- cbind(X_chunk, Si_chunk)
      }
    } else {
      design_chunk <- cbind(X_chunk, Si_chunk)
    }

    design_chunk <- design_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, Si_chunk, design_chunk)
    invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = .(Si, Z)]

  data <- dt[, .(Si, Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data, X = NULL, Si = NULL))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  cat(sprintf("  Cox models...\n"))

  # Subgroup-Specific Models
  subgroup_levels <- levels(data$Si)
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  # Interaction Model
  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)
  interaction_coefs <- grep("Z:Si", rownames(interaction_summary$coefficients), value = TRUE)

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
# ==============================================================================
# RMST and Risk Difference Calculation
# ==============================================================================

calculate_rmst_weighted <- function(time, status, weights, tau) {

  ord <- order(time)
  time <- time[ord]
  status <- status[ord]
  weights <- weights[ord]

  w_atrisk <- rev(cumsum(rev(weights)))
  w_event  <- status * weights

  w_surv <- cumprod(1 - w_event / pmax(w_atrisk, 1e-10))
  w_surv <- c(1, w_surv)
  time_ext <- c(0, time)

  idx <- time_ext <= tau
  time_use <- time_ext[idx]
  surv_use <- w_surv[idx]

  if (max(time_use) < tau) {
    time_use <- c(time_use, tau)
    surv_use <- c(surv_use, tail(surv_use, 1))
  }

  rmst <- sum(diff(time_use) * head(surv_use, -1))
  return(rmst)
}

calculate_rmst_point <- function(results, tau = 1095, verbose = TRUE) {

  if (verbose) cat("  RMST (weighted KM, point estimates only)...\n")

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  results_list <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if (nrow(data_sg) <= 10) return(NULL)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl  <- data_sg[data_sg$Z == 0, ]

    if (nrow(data_treat) <= 10 || nrow(data_ctrl) <= 10) return(NULL)

    # Weighted Kaplan–Meier
    fit_treat <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_treat,
      weights = data_treat$w
    )
    fit_ctrl <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_ctrl,
      weights = data_ctrl$w
    )

    # RMST
    rmst_treat <- tryCatch(
      calculate_rmst_weighted(
        time = data_treat$Survival_Time,
        status = data_treat$Outcome,
        weights = data_treat$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_ctrl <- tryCatch(
      calculate_rmst_weighted(
        time = data_ctrl$Survival_Time,
        status = data_ctrl$Outcome,
        weights = data_ctrl$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_diff <- rmst_treat - rmst_ctrl

    # Risk Differences
    time_points <- c(365, 730, 1095)
    surv_at_times <- lapply(time_points, function(t) {
      sum_treat <- summary(fit_treat, times = t, extend = TRUE)
      sum_ctrl  <- summary(fit_ctrl,  times = t, extend = TRUE)

      surv_t <- if (length(sum_treat$surv) > 0) sum_treat$surv else NA_real_
      surv_c <- if (length(sum_ctrl$surv) > 0) sum_ctrl$surv else NA_real_
      se_t   <- if (length(sum_treat$std.err) > 0) sum_treat$std.err else NA_real_
      se_c   <- if (length(sum_ctrl$std.err) > 0) sum_ctrl$std.err else NA_real_

      risk_t <- 1 - surv_t
      risk_c <- 1 - surv_c
      risk_diff <- risk_c - risk_t

      risk_diff_var <- if (!is.na(se_t) && !is.na(se_c)) {
        se_t^2 + se_c^2
      } else NA_real_

      baseline_risk_var <- if (!is.na(se_c)) {
        se_c^2
      } else NA_real_

      list(
        time = t,
        surv_treat = surv_t,
        se_treat = se_t,
        surv_ctrl = surv_c,
        se_ctrl = se_c,
        risk_treat = risk_t,
        risk_ctrl = risk_c,
        risk_diff = risk_diff,
        risk_diff_var = risk_diff_var,
        baseline_risk = risk_c,
        baseline_risk_var = baseline_risk_var
      )
    })

    list(
      rmst = list(
        subgroup = sg,
        rmst_treat = rmst_treat,
        rmst_ctrl = rmst_ctrl,
        rmst_diff = rmst_diff,
        tau = tau
      ),
      surv_prob = list(
        subgroup = sg,
        time_points = surv_at_times
      )
    )
  })

  results_list <- results_list[!sapply(results_list, is.null)]

  rmst_results <- lapply(results_list, function(x) x$rmst)
  names(rmst_results) <- sapply(rmst_results, function(x) x$subgroup)

  surv_prob_results <- lapply(results_list, function(x) x$surv_prob)
  names(surv_prob_results) <- sapply(surv_prob_results, function(x) x$subgroup)

  return(list(
    rmst_results = rmst_results,
    surv_prob_results = surv_prob_results
  ))
}

bootstrap_rmst_diff <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  set.seed(seed)

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  base_est <- calculate_rmst_point(results, tau = tau, verbose = TRUE)
  rmst_point <- base_est$rmst_results

  use_parallel <- !is.null(n_cores) && n_cores > 1

  if (use_parallel) {
    if (n_cores == "auto") {
      n_cores <- max(1, parallel::detectCores() - 1)
    }

    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac,
        ", cores =", n_cores, ")...\n")

    cl <- parallel::makeCluster(n_cores)

    parallel::clusterExport(cl,
                           c("calculate_rmst_point", "calculate_rmst_weighted",
                             "data_by_sg", "tau", "subsample_frac"),
                           envir = environment())

    parallel::clusterEvalQ(cl, library(survival))

    boot_results <- parallel::parLapply(cl, 1:B, function(b) {

      # Within-subgroup subsample bootstrap
      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- tryCatch(
        calculate_rmst_point(results_boot, tau = tau, verbose = FALSE),
        error = function(e) NULL
      )

      if (is.null(boot_est)) return(NULL)

      sapply(boot_est$rmst_results, function(x) x$rmst_diff)
    })

    parallel::stopCluster(cl)

    boot_results <- boot_results[!sapply(boot_results, is.null)]
    boot_mat_data <- do.call(rbind, boot_results)

    boot_mat <- lapply(seq_along(subgroup_levels), function(i) {
      boot_mat_data[, i]
    })
    names(boot_mat) <- subgroup_levels

  } else {
    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac, ")...\n")

    boot_mat <- lapply(subgroup_levels, function(sg) {
      numeric(B)
    })
    names(boot_mat) <- subgroup_levels

    for (b in seq_len(B)) {
      if (b %% 20 == 0) cat("    Iteration", b, "of", B, "\n")

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- calculate_rmst_point(results_boot, tau = tau, verbose = FALSE)
      rmst_boot <- boot_est$rmst_results

      for (sg in names(rmst_boot)) {
        boot_mat[[sg]][b] <- rmst_boot[[sg]]$rmst_diff
      }
    }
  }

  out <- lapply(rmst_point, function(x) {
    sg <- x$subgroup
    diffs <- boot_mat[[sg]]
    diffs <- diffs[is.finite(diffs)]

    if (length(diffs) < 10) {
      se <- NA_real_
      ci_lower <- NA_real_
      ci_upper <- NA_real_
    } else {
      se <- sd(diffs)
      # Percentile CI (더 robust)
      ci_lower <- quantile(diffs, 0.025, na.rm = TRUE)
      ci_upper <- quantile(diffs, 0.975, na.rm = TRUE)
    }

    c(x, list(
      se_rmst_diff = se,
      ci_lower = ci_lower,
      ci_upper = ci_upper,
      B = length(diffs),
      subsample_frac = subsample_frac
    ))
  })
  names(out) <- names(rmst_point)

  return(list(
    rmst_results = out,
    surv_prob_results = base_est$surv_prob_results
  ))
}

calculate_rmst_fast <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  bootstrap_rmst_diff(
    results = results,
    tau = tau,
    B = B,
    subsample_frac = subsample_frac,
    seed = seed,
    n_cores = n_cores
  )
}

In [ ]:
# ==============================================================================
# Pool Main Effects (Treatment + Subgroup Coefficients)
# ==============================================================================
pool_main_effects <- function(all_results) {

  m <- length(all_results)

  # Pool Treatment Effect
  treat_coefs <- sapply(all_results, function(x) x$main_effects$treatment_coef)
  treat_ses <- sapply(all_results, function(x) x$main_effects$treatment_se)

  Q_bar_treat <- mean(treat_coefs)
  U_bar_treat <- mean(treat_ses^2)
  B_treat <- var(treat_coefs)
  T_treat <- U_bar_treat + (1 + 1/m) * B_treat
  se_treat <- sqrt(T_treat)

  df_treat <- if(B_treat < 1e-10) Inf else {
    (m - 1) * (1 + U_bar_treat / ((1 + 1/m) * B_treat))^2
  }

  pooled_main_effects <- list(
    treatment = list(
      coef = Q_bar_treat,
      se = se_treat,
      hr = exp(Q_bar_treat),
      ci_lower = exp(Q_bar_treat - qt(0.975, df_treat) * se_treat),
      ci_upper = exp(Q_bar_treat + qt(0.975, df_treat) * se_treat),
      pvalue = 2 * pt(-abs(Q_bar_treat / se_treat), df = df_treat),
      df = df_treat
    )
  )

  # Pool All Subgroup Coefficients
  if (!is.null(all_results[[1]]$main_effects$subgroup_coefs)) {

    common_names <- Reduce(
      intersect,
      lapply(all_results, function(res)
        vapply(res$main_effects$subgroup_coefs, `[[`, character(1), "name"))
    )

    pooled_subgroup_coefs <- lapply(common_names, function(sg_name) {

      coefs <- sapply(all_results, function(res) {
        idx <- which(vapply(res$main_effects$subgroup_coefs,
                            `[[`, character(1), "name") == sg_name)
        if (length(idx) == 0) NA_real_ else
          as.numeric(res$main_effects$subgroup_coefs[[idx]]$coef)
      })

      ses <- sapply(all_results, function(res) {
        idx <- which(vapply(res$main_effects$subgroup_coefs,
                            `[[`, character(1), "name") == sg_name)
        if (length(idx) == 0) NA_real_ else
          as.numeric(res$main_effects$subgroup_coefs[[idx]]$se)
      })

      valid <- is.finite(coefs) & is.finite(ses)
      coefs <- coefs[valid]
      ses   <- ses[valid]

      if (length(coefs) == 0L) return(NULL)

      m_valid        <- length(coefs)
      Q_bar_subgroup <- mean(coefs)
      U_bar_subgroup <- mean(ses^2)
      B_subgroup     <- var(coefs)
      T_subgroup     <- U_bar_subgroup + (1 + 1/m_valid) * B_subgroup
      se_subgroup    <- sqrt(T_subgroup)
      df_subgroup    <- if (B_subgroup < 1e-10) Inf else {
        (m_valid - 1) * (1 + U_bar_subgroup / ((1 + 1/m_valid) * B_subgroup))^2
      }

      list(
        name    = sg_name,
        coef    = Q_bar_subgroup,
        se      = se_subgroup,
        hr      = exp(Q_bar_subgroup),
        ci_lower = exp(Q_bar_subgroup - qt(0.975, df_subgroup) * se_subgroup),
        ci_upper = exp(Q_bar_subgroup + qt(0.975, df_subgroup) * se_subgroup),
        pvalue  = 2 * pt(-abs(Q_bar_subgroup / se_subgroup), df = df_subgroup),
        df      = df_subgroup
      )
    })

    pooled_subgroup_coefs <- pooled_subgroup_coefs[!vapply(pooled_subgroup_coefs, is.null, logical(1))]

    pooled_main_effects$subgroup_coefs <- pooled_subgroup_coefs
    pooled_main_effects$subgroup_var   <- all_results[[1]]$main_effects$subgroup_var
  }

  return(pooled_main_effects)
}

# ==============================================================================
# Pool Subgroup-Specific Treatment Effects
# ==============================================================================
pool_subgroup_results <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  m <- length(all_results)

  pooled_subgroups <- lapply(subgroup_names, function(sg) {
    coefs <- sapply(all_results, function(x) x$subgroup_results[[sg]]$coef)
    ses <- sapply(all_results, function(x) x$subgroup_results[[sg]]$se_robust)

    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m - 1) * (1 + U_bar / ((1 + 1/m) * B))^2

    list(
      subgroup = sg,
      coef = Q_bar,
      se = se_pooled,
      hr = exp(Q_bar),
      ci_lower = exp(Q_bar - qt(0.975, df) * se_pooled),
      ci_upper = exp(Q_bar + qt(0.975, df) * se_pooled),
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })

  names(pooled_subgroups) <- subgroup_names
  return(pooled_subgroups)
}

In [ ]:
# ==============================================================================
# Pool Interaction Terms
# ==============================================================================
pool_interactions <- function(all_results) {

  interaction_terms <- sapply(all_results[[1]]$interaction_results, function(x) x$term)

  pooled_interactions <- lapply(interaction_terms, function(term) {
    coefs <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$coef else NA
    })

    ses <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$se_robust else NA
    })

    valid_idx <- !is.na(coefs) & !is.na(ses)
    coefs <- coefs[valid_idx]
    ses <- ses[valid_idx]

    if(length(coefs) == 0) return(NULL)

    m_valid <- length(coefs)
    Q_bar <- mean(coefs)
    U_bar <- mean(ses^2)
    B <- var(coefs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      term = term,
      coef = Q_bar,
      se = se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      df = df
    )
  })

  pooled_interactions <- pooled_interactions[!sapply(pooled_interactions, is.null)]
  return(pooled_interactions)
}

# ==============================================================================
# Pool RMST Differences
# ==============================================================================
pool_rmst <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)

  pooled_rmst <- lapply(subgroup_names, function(sg) {
    rmst_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$rmst_diff)
    se_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$se_rmst_diff)

    valid_idx <- !is.na(rmst_diffs) & !is.na(se_diffs)
    rmst_diffs <- rmst_diffs[valid_idx]
    se_diffs <- se_diffs[valid_idx]

    if(length(rmst_diffs) == 0) return(NULL)

    m_valid <- length(rmst_diffs)
    Q_bar <- mean(rmst_diffs)
    U_bar <- mean(se_diffs^2)
    B <- var(rmst_diffs)
    T <- U_bar + (1 + 1/m_valid) * B
    se_pooled <- sqrt(T)

    df <- if(B < 1e-10) Inf else (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2

    list(
      subgroup = sg,
      rmst_diff = Q_bar,
      se = se_pooled,
      ci_lower = Q_bar - qt(0.975, df) * se_pooled,
      ci_upper = Q_bar + qt(0.975, df) * se_pooled,
      pvalue = 2 * pt(-abs(Q_bar / se_pooled), df = df),
      tau = all_results[[1]]$rmst_results[[sg]]$tau
    )
  })

  names(pooled_rmst) <- subgroup_names
  pooled_rmst <- pooled_rmst[!sapply(pooled_rmst, is.null)]

  return(pooled_rmst)
}

In [ ]:
# ==============================================================================
# Pool Absolute Risk Differences and Baseline Risks
# ==============================================================================
pool_survival_probs <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  time_points <- c(365, 730, 1095)

  pooled_surv_probs <- lapply(subgroup_names, function(sg) {
    time_results <- lapply(seq_along(time_points), function(t_idx) {

      risk_diffs <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff else NA
      })

      risk_diff_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff_var else NA
      })

      baseline_risks <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk else NA
      })

      baseline_risk_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk_var else NA
      })

      valid_idx_rd <- !is.na(risk_diffs) & !is.na(risk_diff_vars)
      risk_diffs_valid <- risk_diffs[valid_idx_rd]
      risk_diff_vars_valid <- risk_diff_vars[valid_idx_rd]

      valid_idx_br <- !is.na(baseline_risks) & !is.na(baseline_risk_vars)
      baseline_risks_valid <- baseline_risks[valid_idx_br]
      baseline_risk_vars_valid <- baseline_risk_vars[valid_idx_br]

      if(length(risk_diffs_valid) == 0) return(NULL)

      # Rubin's Rules for Risk Difference
      m_valid_rd <- length(risk_diffs_valid)
      Q_bar_rd <- mean(risk_diffs_valid)
      U_bar_rd <- mean(risk_diff_vars_valid)
      B_rd <- var(risk_diffs_valid)
      T_rd <- U_bar_rd + (1 + 1/m_valid_rd) * B_rd
      se_pooled_rd <- sqrt(T_rd)

      df_rd <- if(B_rd < 1e-10) Inf else {
        (m_valid_rd - 1) * (1 + U_bar_rd / ((1 + 1/m_valid_rd) * B_rd))^2
      }

      ci_lower_rd <- Q_bar_rd - qt(0.975, df_rd) * se_pooled_rd
      ci_upper_rd <- Q_bar_rd + qt(0.975, df_rd) * se_pooled_rd
      pvalue_rd <- 2 * pt(-abs(Q_bar_rd / se_pooled_rd), df = df_rd)

      # Rubin's Rules for Baseline Risk
      baseline_result <- if(length(baseline_risks_valid) > 0) {
        m_valid_br <- length(baseline_risks_valid)
        Q_bar_br <- mean(baseline_risks_valid)
        U_bar_br <- mean(baseline_risk_vars_valid)
        B_br <- var(baseline_risks_valid)
        T_br <- U_bar_br + (1 + 1/m_valid_br) * B_br
        se_pooled_br <- sqrt(T_br)

        df_br <- if(B_br < 1e-10) Inf else {
          (m_valid_br - 1) * (1 + U_bar_br / ((1 + 1/m_valid_br) * B_br))^2
        }

        ci_lower_br <- Q_bar_br - qt(0.975, df_br) * se_pooled_br
        ci_upper_br <- Q_bar_br + qt(0.975, df_br) * se_pooled_br
        pvalue_br <- 2 * pt(-abs(Q_bar_br / se_pooled_br), df = df_br)

        list(
          baseline_risk = Q_bar_br,
          baseline_risk_se = se_pooled_br,
          baseline_risk_ci_lower = ci_lower_br,
          baseline_risk_ci_upper = ci_upper_br,
          baseline_risk_df = df_br
        )
      } else {
        list(
          baseline_risk = NA,
          baseline_risk_se = NA,
          baseline_risk_ci_lower = NA,
          baseline_risk_ci_upper = NA,
          baseline_risk_df = NA
        )
      }

      c(
        list(
          time = time_points[t_idx],
          risk_diff = Q_bar_rd,
          se = se_pooled_rd,
          ci_lower = ci_lower_rd,
          ci_upper = ci_upper_rd,
          pvalue = pvalue_rd,
          df = df_rd,
          within_var = U_bar_rd,
          between_var = B_rd,
          fmi = (1 + 1/m_valid_rd) * B_rd / T_rd
        ),
        baseline_result
      )
    })

    time_results <- time_results[!sapply(time_results, is.null)]
    list(subgroup = sg, time_results = time_results)
  })

  names(pooled_surv_probs) <- subgroup_names
  return(pooled_surv_probs)
}

# ==============================================================================
# Execute All Pooling Functions
# ==============================================================================
pool_results <- function(all_results) {
    cat("\n", rep("=", 80), "\n")
    cat("EXECUTING RUBIN'S POOLING...\n")
    cat(rep("=", 80), "\n\n")

    # Pool Subgroup-Specific Results
    cat("Step 2/5: Pooling subgroup-specific treatment effects...\n")
    pooled_subgroups <- pool_subgroup_results(all_results)
    cat(sprintf(" ✓ %d subgroups pooled\n\n", length(pooled_subgroups)))

    # Pool Interaction Terms
    cat("Step 3/5: Pooling interaction terms...\n")
    pooled_interactions <- pool_interactions(all_results)
    cat(sprintf(" ✓ %d interaction terms pooled\n\n", length(pooled_interactions)))

    # Pool RMST Results
    cat("Step 4/5: Pooling RMST differences...\n")
    pooled_rmst <- pool_rmst(all_results)
    cat(sprintf(" ✓ RMST for %d subgroups pooled\n\n", length(pooled_rmst)))

    # Pool Survival Probabilities and Risk Differences
    cat("Step 5/5: Pooling absolute risk differences and baseline risks...\n")
    pooled_surv_probs <- pool_survival_probs(all_results)
    cat(sprintf(" ✓ Risk differences for %d subgroups pooled\n\n", length(pooled_surv_probs)))

    # Combine all results
    pooled_results <- list(
        main_effects = pooled_main_effects,
        subgroup_results = pooled_subgroups,
        interaction_results = pooled_interactions,
        rmst_results = pooled_rmst,
        surv_prob_results = pooled_surv_probs
    )

    return(pooled_results)
}

In [ ]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

subgroup_var <- "SMART_risk"
covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
  "HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
  "HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
  "HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
  "HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant",
  "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone", "HasOtherAntiHypertensive",
  "HasHospitalization", "Race", "Ethnicity", "CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
  "NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
  "NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Sex", "Age", "CalendarYear",
  "AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HemoglobinValue",
  "LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
  "SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue", "HbA1cValue",
  "AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
  "AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)
covariates <- setdiff(covariates, subgroup_var)
treatment <- "FirstMedGroup"

start_time <- Sys.time()

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 1: LASSO Variable Selection (First Imputation Only)\n")
cat(rep("=", 80), "\n")

lasso_result <- perform_lasso_first_only(
  imp_data = imputed_t2d_list[[1]],
  covariates = covariates,
  subgroup_var = subgroup_var,
  treatment = treatment,
  sample_size = 50000
)

union_vars <- lasso_result$union_vars

cat("\n", rep("=", 80), "\n")
cat(sprintf("Selected variables: %d (including %d interactions)\n",
            length(union_vars), length(lasso_result$selected_interactions)))
cat(rep("=", 80), "\n")

# ==============================================================================
# Full Analysis
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 2: Full Analysis (All 5 Imputations - SEQUENTIAL)\n")
cat(rep("=", 80), "\n")

all_results <- list()

for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Processing Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_with_selected_vars(
    imp_data = imputed_t2d_list[[i]],
    imp_num = i,
    covariates = covariates,
    subgroup_var = subgroup_var,
    treatment = treatment,
    union_vars = union_vars,
    tau = 1095
  )

  cox_results <- complete_analysis_fast(
    data_list,
    imp_num = i,
    subgroup_var = subgroup_var,
    tau = 1095
  )

  rmst_results <- calculate_rmst_fast(
    cox_results,
    tau = 1095,
    B = 100,
    subsample_frac = 0.3,
    n_cores = 4
  )

  all_results[[i]] <- list(
    main_effects = cox_results$main_effects,
    subgroup_results = cox_results$subgroup_results,
    interaction_results = cox_results$interaction_results,
    data_by_sg = if(i == 1) cox_results$data_by_sg else NULL,
    rmst_results = rmst_results$rmst_results,
    surv_prob_results = rmst_results$surv_prob_results
  )

  cat("Imputation", i, "completed!\n")

  invisible(gc())
}

cat("\nAll imputations completed!\n")

# ==============================================================================
# Rubin's Pooling
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Rubin's Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results(all_results)

# ==============================================================================
# Summary
# ==============================================================================
end_time <- Sys.time()
elapsed <- as.numeric(end_time - start_time, units = "mins")

cat("\n", rep("=", 80), "\n")
cat(sprintf("Total Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# Display Results
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("FINAL POOLED RESULTS (3-Year Follow-up)\n")
cat(sprintf("Subgroup Analysis by: %s\n", subgroup_var))
cat(rep("=", 80), "\n\n")

cat("\n1. SUBGROUP-SPECIFIC TREATMENT EFFECTS (Hazard Ratios)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$subgroup_results)) {
  res <- pooled_results$subgroup_results[[sg]]
  cat(sprintf("Subgroup %s: HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
              sg, res$hr, res$ci_lower, res$ci_upper, res$pvalue))
}

cat(sprintf("\n2. INTERACTION TEST (Treatment × %s)\n", subgroup_var))
cat(rep("-", 80), "\n")
for(res in pooled_results$interaction_results) {
  cat(sprintf("%s: Coef = %.4f (SE = %.4f), p = %.4f\n",
              res$term, res$coef, res$se, res$pvalue))
  if(res$pvalue < 0.05) {
    cat("→ SIGNIFICANT heterogeneity on relative scale\n")
  } else {
    cat("→ No significant heterogeneity on relative scale\n")
  }
}

cat("\n3. RESTRICTED MEAN SURVIVAL TIME (3-year RMST)\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$rmst_results)) {
  res <- pooled_results$rmst_results[[sg]]
  cat(sprintf("Subgroup %s: RMST Diff = %.1f days (95%% CI: %.1f-%.1f), p = %.4f\n",
              sg, res$rmst_diff, res$ci_lower, res$ci_upper, res$pvalue))
  cat(sprintf("  → Treatment %s %.1f days of event-free survival\n",
              if(res$rmst_diff > 0) "extends by" else "reduces by",
              abs(res$rmst_diff)))
}

cat("\n4. ABSOLUTE RISK DIFFERENCES\n")
cat(rep("-", 80), "\n")
for(sg in names(pooled_results$surv_prob_results)) {
  cat(sprintf("\nSubgroup %s:\n", sg))
  time_res <- pooled_results$surv_prob_results[[sg]]$time_results
  for(tr in time_res) {
    cat(sprintf("  At %.1f years: Risk Diff = %.4f (95%% CI: %.4f-%.4f)\n",
                tr$time/365.25, tr$risk_diff, tr$ci_lower, tr$ci_upper))
    cat(sprintf("    SE = %.4f, p = %.4f (%.2f%% reduction)\n",
                tr$se, tr$pvalue, tr$risk_diff * 100))
    cat(sprintf("    FMI = %.3f (%.1f%% of variance from imputation uncertainty)\n",
                tr$fmi, tr$fmi * 100))
  }
}

cat("\n", rep("=", 80), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# SUBGROUP SAMPLE STATISTICS
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("6. SUBGROUP SAMPLE STATISTICS\n")
cat(rep("=", 80), "\n")

cat("\n6-1. ORIGINAL DATA (Before Overlap Weighting)\n")
cat(rep("-", 80), "\n\n")

cat("A. Overall Dataset Statistics\n")
cat(rep("-", 40), "\n")

overall_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  total_n <- nrow(dt)

  age_counts <- dt[, .N, by = Si]

  list(
    total_n = total_n,
    age_counts = age_counts
  )
})

avg_total_n <- round(mean(sapply(overall_stats, function(x) x$total_n)))

age_group_levels <- unique(overall_stats[[1]]$age_counts$Si)

avg_age_counts <- lapply(age_group_levels, function(sg) {
  counts <- sapply(overall_stats, function(x) {
    x$age_counts[x$age_counts$Si == sg, ]$N
  })
  list(
    subgroup = as.character(sg),
    count = round(mean(counts))
  )
})

cat(sprintf("Total Sample Size: n = %d\n\n", avg_total_n))
cat("Age Group Distribution:\n")
for(ac in avg_age_counts) {
  cat(sprintf("  %s: n = %d (%.1f%%)\n",
              ac$subgroup, ac$count, ac$count/avg_total_n*100))
}
cat("\n")

cat("\nB. Statistics by Age Subgroup\n")
cat(rep("-", 40), "\n\n")

original_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  stats_by_sg <- dt[, .(
    n_total = .N,
    n_treat = sum(Z == 1),
    n_ctrl = sum(Z == 0),
    events_total = sum(Outcome),
    events_treat = sum(Outcome[Z == 1]),
    events_ctrl = sum(Outcome[Z == 0])
  ), by = Si]

  setDF(stats_by_sg)
  stats_by_sg
})

avg_original_stats <- lapply(unique(original_stats[[1]]$Si), function(sg) {
  n_total_vec <- sapply(original_stats, function(x) x$n_total[x$Si == sg])
  n_treat_vec <- sapply(original_stats, function(x) x$n_treat[x$Si == sg])
  n_ctrl_vec <- sapply(original_stats, function(x) x$n_ctrl[x$Si == sg])
  events_total_vec <- sapply(original_stats, function(x) x$events_total[x$Si == sg])
  events_treat_vec <- sapply(original_stats, function(x) x$events_treat[x$Si == sg])
  events_ctrl_vec <- sapply(original_stats, function(x) x$events_ctrl[x$Si == sg])

  list(
    subgroup = as.character(sg),
    n_total = round(mean(n_total_vec)),
    n_treat = round(mean(n_treat_vec)),
    n_ctrl = round(mean(n_ctrl_vec)),
    events_total = round(mean(events_total_vec)),
    events_treat = round(mean(events_treat_vec)),
    events_ctrl = round(mean(events_ctrl_vec))
  )
})

for(stats in avg_original_stats) {
  cat(sprintf("Age Group: %s\n", stats$subgroup))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size:\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat("\n6-2. AFTER OVERLAP WEIGHTING (Pseudo Population)\n")
cat(rep("-", 80), "\n\n")

subgroup_levels <- names(all_results[[1]]$data_by_sg)

avg_weighted_stats <- lapply(subgroup_levels, function(sg) {

  stats_list <- lapply(1:5, function(i) {
    data_sg <- all_results[[i]]$data_by_sg[[sg]]

    if(is.null(data_sg)) return(NULL)

    n_total <- nrow(data_sg)
    n_events <- sum(data_sg$Outcome)
    ess_total <- sum(data_sg$w)^2 / sum(data_sg$w^2)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl <- data_sg[data_sg$Z == 0, ]

    n_treat <- nrow(data_treat)
    n_ctrl <- nrow(data_ctrl)
    events_treat <- sum(data_treat$Outcome)
    events_ctrl <- sum(data_ctrl$Outcome)
    ess_treat <- if(n_treat > 0) sum(data_treat$w)^2 / sum(data_treat$w^2) else 0
    ess_ctrl <- if(n_ctrl > 0) sum(data_ctrl$w)^2 / sum(data_ctrl$w^2) else 0

    list(
      n_total = n_total,
      n_treat = n_treat,
      n_ctrl = n_ctrl,
      events_total = n_events,
      events_treat = events_treat,
      events_ctrl = events_ctrl,
      ess_total = ess_total,
      ess_treat = ess_treat,
      ess_ctrl = ess_ctrl
    )
  })

  stats_list <- stats_list[!sapply(stats_list, is.null)]

  list(
    subgroup = sg,
    n_total = round(mean(sapply(stats_list, function(x) x$n_total))),
    n_treat = round(mean(sapply(stats_list, function(x) x$n_treat))),
    n_ctrl = round(mean(sapply(stats_list, function(x) x$n_ctrl))),
    events_total = round(mean(sapply(stats_list, function(x) x$events_total))),
    events_treat = round(mean(sapply(stats_list, function(x) x$events_treat))),
    events_ctrl = round(mean(sapply(stats_list, function(x) x$events_ctrl))),
    ess_total = round(mean(sapply(stats_list, function(x) x$ess_total)), 1),
    ess_treat = round(mean(sapply(stats_list, function(x) x$ess_treat)), 1),
    ess_ctrl = round(mean(sapply(stats_list, function(x) x$ess_ctrl)), 1)
  )
})
names(avg_weighted_stats) <- subgroup_levels

for(sg in names(avg_weighted_stats)) {
  stats <- avg_weighted_stats[[sg]]

  cat(sprintf("Age Group: %s\n", sg))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size (Original):\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Effective Sample Size (ESS):\n"))
  cat(sprintf("    Total:     ESS = %.1f\n", stats$ess_total))
  cat(sprintf("    Treatment: ESS = %.1f\n", stats$ess_treat))
  cat(sprintf("    Control:   ESS = %.1f\n", stats$ess_ctrl))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

In [ ]:
data_by_sg <- all_results[[1]]$data_by_sg

risk_groups <- names(data_by_sg)
risk_labels <- c("Low-risk", "Intermediate-risk", "High-risk")
colors <- c("#E41A1C", "#377EB8", "#4DAF4A")  # Red, Blue, Green

fit_list <- list()
for(i in seq_along(risk_groups)) {
  sg <- risk_groups[i]
  data_sg <- data_by_sg[[sg]]

  data_treat <- data_sg[data_sg$Z == 1, ]
  data_ctrl <- data_sg[data_sg$Z == 0, ]

  fit_list[[i]] <- list(
    treat = survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_treat, weights = w),
    ctrl = survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_ctrl, weights = w)
  )
}

image_path <- file.path(artifacts_images_dir, "t2d_secondary_primary_endpoint_smart_risk.png")

png(filename = image_path, width = 9.5, height = 7, units = "in", res = 300)

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.2),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.2, by = 0.05), col = "gray50", lwd = 0.5)

# Plot lines for each risk group
for(i in seq_along(risk_groups)) {
  lines(c(0, fit_list[[i]]$treat$time / 30.44),
        c(0, 1 - fit_list[[i]]$treat$surv),
        col = colors[i], lty = 1, lwd = 2.5)
  lines(c(0, fit_list[[i]]$ctrl$time / 30.44),
        c(0, 1 - fit_list[[i]]$ctrl$surv),
        col = colors[i], lty = 3, lwd = 2.5)
}

# # Legend
# legend_labels <- c()
# legend_cols <- c()
# legend_ltys <- c()

# for(i in seq_along(risk_labels)) {
#   legend_labels <- c(legend_labels,
#                      paste0(risk_labels[i], " - GLP-1 RAs"),
#                      paste0(risk_labels[i], " - Non-GLP-1 RAs"))
#   legend_cols <- c(legend_cols, colors[i], colors[i])
#   legend_ltys <- c(legend_ltys, 1, 3)
# }

# legend("topleft",
#        legend = legend_labels,
#        col = legend_cols,
#        lty = legend_ltys,
#        lwd = 2.5,
#        bty = "n",
#        cex = 1.4)

dev.off()

cat("Plot saved to:", image_path, "\n")

par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(4, 1, 0))

plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.2),
     xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
     las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

axis(2, at = seq(0, 0.20, by = 0.05),
     labels = paste0(seq(0, 20, by = 5), "%"),
     las = 1, cex.axis = 1.5)
abline(h = seq(0, 0.2, by = 0.05), col = "gray72", lwd = 0.5)

# Plot lines for each risk group
for(i in seq_along(risk_groups)) {
  lines(c(0, fit_list[[i]]$treat$time / 30.44),
        c(0, 1 - fit_list[[i]]$treat$surv),
        col = colors[i], lty = 1, lwd = 2.5)
  lines(c(0, fit_list[[i]]$ctrl$time / 30.44),
        c(0, 1 - fit_list[[i]]$ctrl$surv),
        col = colors[i], lty = 3, lwd = 2.5)
}

# legend("topleft",
#        legend = legend_labels,
#        col = legend_cols,
#        lty = legend_ltys,
#        lwd = 2.5,
#        bty = "n",
#        cex = 1.4)

## ETHNICITY

In [ ]:
setDTthreads(0)

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
perform_lasso_first_only <- function(imp_data, covariates, subgroup_var, treatment,
                                     sample_size = 50000) {

  cat("\n=== LASSO (First Imputation Only, Stratified Sampling) ===\n")

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment)
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(sample_size / n_groups)

  cat(sprintf("  Total groups (Si × Z): %d\n", n_groups))
  cat(sprintf("  Target per group: %d\n", sample_per_group))

  # Stratified sampling by Si × Z
  set.seed(12345)
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Actual sample: %d obs\n", nrow(dt_sample)))

  factor_cols <- names(dt_sample)[sapply(dt_sample, is.factor) | sapply(dt_sample, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt_sample[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  # Sparse matrix
  X_df <- dt_sample[, ..covariates]
  X <- sparse.model.matrix(~ . - 1, data = X_df)
  Si <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  n_subgroups <- ncol(Si)

  cat("  Creating interactions...\n")

  Xi_Si_list <- lapply(1:n_subgroups, function(k) {
    si_k <- as.vector(Si[, k])
    X_k <- X * si_k
    colnames(X_k) <- paste0(colnames(X), ":", colnames(Si)[k])
    X_k
  })

  Xi_Si <- do.call(cbind, Xi_Si_list)
  rm(Xi_Si_list); invisible(gc())

  design_matrix <- cbind(X, Si, Xi_Si)

  # Penalty factor
  penalty.factor <- c(
    rep(0, ncol(X) + ncol(Si)),
    rep(1, ncol(Xi_Si))
  )

  cat("  Running LASSO...\n")

  # LASSO
  lasso_fit <- cv.glmnet(
    design_matrix, dt_sample$Z,
    family = "binomial",
    alpha = 1,
    penalty.factor = penalty.factor,
    nfolds = 3,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 2000
  )

  selected_coef <- coef(lasso_fit, s = "lambda.min")
  selected_vars <- rownames(selected_coef)[selected_coef[, 1] != 0]
  selected_vars <- selected_vars[selected_vars != "(Intercept)"]
  selected_interactions <- selected_vars[grepl(":", selected_vars)]

  cat(sprintf("  Selected %d interactions\n", length(selected_interactions)))

  # Base variables + selected interactions
  union_vars <- c(colnames(X), colnames(Si), selected_interactions)

  return(list(
    union_vars = union_vars,
    selected_interactions = selected_interactions,
    base_vars = c(colnames(X), colnames(Si))
  ))
}

# ==============================================================================
# Propensity Score Estimation with Selected Variables
# ==============================================================================
analyze_with_selected_vars <- function(imp_data, imp_num,
                                       covariates, subgroup_var, treatment,
                                       union_vars, tau = 1095) {

  cat(sprintf("\n=== Processing Imputation %d ===\n", imp_num))

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(target_sample_size / n_groups)

  cat(sprintf("  Target sample: %d (stratified by Si × Z)\n", target_sample_size))

  # Stratified sampling
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]

  cat(sprintf("  Actual sample: %d\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample <- sparse.model.matrix(~ . - 1, data = X_df_sample)
  Si_sample <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  interaction_names <- union_vars[grepl(":", union_vars)]

  if(length(interaction_names) > 0 && length(interaction_names) < 800) {
    cat(sprintf("  Creating %d interactions (sample)...\n", length(interaction_names)))

    n_subgroups <- ncol(Si_sample)
    Xi_Si_list <- list()

    for(k in 1:n_subgroups) {
      si_k <- as.vector(Si_sample[, k])
      X_k <- X_sample * si_k
      colnames(X_k) <- paste0(colnames(X_sample), ":", colnames(Si_sample)[k])

      selected_cols <- intersect(colnames(X_k), interaction_names)
      if(length(selected_cols) > 0) {
        Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols, drop = FALSE]
      }
    }

    if(length(Xi_Si_list) > 0) {
      Xi_Si_sample <- do.call(cbind, Xi_Si_list)
      design_sample <- cbind(X_sample, Si_sample, Xi_Si_sample)
      rm(Xi_Si_list); invisible(gc())
    } else {
      design_sample <- cbind(X_sample, Si_sample)
    }
  } else {
    cat("  Too many interactions, using main effects only\n")
    design_sample <- cbind(X_sample, Si_sample)
  }

  all_vars <- colnames(design_sample)
  selected_cols <- intersect(union_vars, all_vars)
  design_selected_sample <- design_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d × %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model...\n")

  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family = "binomial",
    alpha = 0,
    lambda = 0.01,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 1000
  )

  cat("  Predicting on full data...\n")

  chunk_size <- 200000
  n_chunks <- ceiling(n_total / chunk_size)
  ps_all <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) {
      cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))
    }

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx <- min(chunk_i * chunk_size, n_total)

    dt_chunk <- dt[start_idx:end_idx]

    # Design matrix for chunk
    X_df_chunk <- dt_chunk[, ..covariates]
    X_chunk <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    Si_chunk <- sparse.model.matrix(~ Si - 1, data = dt_chunk)

    if(length(interaction_names) > 0 && length(interaction_names) < 800) {
      n_subgroups <- ncol(Si_chunk)
      Xi_Si_list <- list()

      for(k in 1:n_subgroups) {
        si_k <- as.vector(Si_chunk[, k])
        X_k <- X_chunk * si_k
        colnames(X_k) <- paste0(colnames(X_chunk), ":", colnames(Si_chunk)[k])

        selected_cols_chunk <- intersect(colnames(X_k), interaction_names)
        if(length(selected_cols_chunk) > 0) {
          Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols_chunk, drop = FALSE]
        }
      }

      if(length(Xi_Si_list) > 0) {
        Xi_Si_chunk <- do.call(cbind, Xi_Si_list)
        design_chunk <- cbind(X_chunk, Si_chunk, Xi_Si_chunk)
        rm(Xi_Si_list)
      } else {
        design_chunk <- cbind(X_chunk, Si_chunk)
      }
    } else {
      design_chunk <- cbind(X_chunk, Si_chunk)
    }

    design_chunk <- design_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, Si_chunk, design_chunk)
    invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  # Overlap weights
  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = .(Si, Z)]

  data <- dt[, .(Si, Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data, X = NULL, Si = NULL))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  data$Si <- factor(data$Si, levels = c("1065359", "1065401"))
  subgroup_levels <- levels(data$Si)

  cat(sprintf("  Cox models... (Subgroups: %s, Reference: %s)\n",
              paste(subgroup_levels, collapse = ", "),
              subgroup_levels[1]))

  # Subgroup main effect
  si_coef_names <- grep("^Si", rownames(main_summary$coefficients), value = TRUE)

  if(length(si_coef_names) > 0) {
    main_effects$subgroup_coef <- main_summary$coefficients[si_coef_names[1], "coef"]
    main_effects$subgroup_se <- main_summary$coefficients[si_coef_names[1], "robust se"]
    main_effects$subgroup_hr <- main_summary$coefficients[si_coef_names[1], "exp(coef)"]
    main_effects$subgroup_pvalue <- main_summary$coefficients[si_coef_names[1], "Pr(>|z|)"]
    main_effects$subgroup_var <- subgroup_var
    main_effects$subgroup_term <- si_coef_names[1]
    main_effects$subgroup_reference <- subgroup_levels[1]

    cat(sprintf("  Subgroup main effect: %s vs %s (Ref), HR = %.3f\n",
                si_coef_names[1], subgroup_levels[1], main_effects$subgroup_hr))
  } else {
    cat("  Warning: No subgroup main effect coefficient found\n")
    main_effects$subgroup_coef <- NA
    main_effects$subgroup_se <- NA
  }

  # Subgroup-Specific Models
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  # Interaction Model
  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)

  all_coef_names <- rownames(interaction_summary$coefficients)
  cat(sprintf("  All coefficients: %s\n", paste(all_coef_names, collapse = ", ")))

  interaction_coefs <- grep("Z:Si", all_coef_names, value = TRUE)
  cat(sprintf("  Interaction terms: %s\n", paste(interaction_coefs, collapse = ", ")))

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  data$Si <- factor(data$Si)
  subgroup_levels <- levels(data$Si)

  cat(sprintf("  Cox models... (Subgroups: %s)\n", paste(subgroup_levels, collapse = ", ")))

  cox_main <- coxph(
    Surv(Survival_Time, Outcome) ~ Z + Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  main_summary <- summary(cox_main)

  main_effects <- list(
    treatment_coef = main_summary$coefficients["Z", "coef"],
    treatment_se = main_summary$coefficients["Z", "robust se"],
    treatment_hr = main_summary$coefficients["Z", "exp(coef)"],
    treatment_pvalue = main_summary$coefficients["Z", "Pr(>|z|)"]
  )

  si_coef_names <- grep("^Si", rownames(main_summary$coefficients), value = TRUE)

  if(length(si_coef_names) > 0) {
    main_effects$subgroup_coef <- main_summary$coefficients[si_coef_names[1], "coef"]
    main_effects$subgroup_se <- main_summary$coefficients[si_coef_names[1], "robust se"]
    main_effects$subgroup_hr <- main_summary$coefficients[si_coef_names[1], "exp(coef)"]
    main_effects$subgroup_pvalue <- main_summary$coefficients[si_coef_names[1], "Pr(>|z|)"]
    main_effects$subgroup_var <- subgroup_var
    main_effects$subgroup_term <- si_coef_names[1]

    cat(sprintf("  Subgroup main effect found: %s (HR = %.3f)\n",
                si_coef_names[1], main_effects$subgroup_hr))
  } else {
    cat("  Warning: No subgroup main effect coefficient found\n")
    main_effects$subgroup_coef <- NA
    main_effects$subgroup_se <- NA
  }

  # Subgroup-Specific Models
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  # Interaction Model
  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)
  interaction_coefs <- grep("Z:Si", rownames(interaction_summary$coefficients), value = TRUE)

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
# ==============================================================================
# RMST and Risk Difference Calculation
# ==============================================================================

calculate_rmst_weighted <- function(time, status, weights, tau) {

  ord <- order(time)
  time <- time[ord]
  status <- status[ord]
  weights <- weights[ord]

  w_atrisk <- rev(cumsum(rev(weights)))
  w_event  <- status * weights

  w_surv <- cumprod(1 - w_event / pmax(w_atrisk, 1e-10))
  w_surv <- c(1, w_surv)
  time_ext <- c(0, time)

  idx <- time_ext <= tau
  time_use <- time_ext[idx]
  surv_use <- w_surv[idx]

  if (max(time_use) < tau) {
    time_use <- c(time_use, tau)
    surv_use <- c(surv_use, tail(surv_use, 1))
  }

  rmst <- sum(diff(time_use) * head(surv_use, -1))
  return(rmst)
}

calculate_rmst_point <- function(results, tau = 1095, verbose = TRUE) {

  if (verbose) cat("  RMST (weighted KM, point estimates only)...\n")

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  results_list <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if (nrow(data_sg) <= 10) return(NULL)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl  <- data_sg[data_sg$Z == 0, ]

    if (nrow(data_treat) <= 10 || nrow(data_ctrl) <= 10) return(NULL)

    # Weighted Kaplan–Meier
    fit_treat <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_treat,
      weights = data_treat$w
    )
    fit_ctrl <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_ctrl,
      weights = data_ctrl$w
    )

    # RMST
    rmst_treat <- tryCatch(
      calculate_rmst_weighted(
        time = data_treat$Survival_Time,
        status = data_treat$Outcome,
        weights = data_treat$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_ctrl <- tryCatch(
      calculate_rmst_weighted(
        time = data_ctrl$Survival_Time,
        status = data_ctrl$Outcome,
        weights = data_ctrl$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_diff <- rmst_treat - rmst_ctrl

    # Risk Differences
    time_points <- c(365, 730, 1095)
    surv_at_times <- lapply(time_points, function(t) {
      sum_treat <- summary(fit_treat, times = t, extend = TRUE)
      sum_ctrl  <- summary(fit_ctrl,  times = t, extend = TRUE)

      surv_t <- if (length(sum_treat$surv) > 0) sum_treat$surv else NA_real_
      surv_c <- if (length(sum_ctrl$surv) > 0) sum_ctrl$surv else NA_real_
      se_t   <- if (length(sum_treat$std.err) > 0) sum_treat$std.err else NA_real_
      se_c   <- if (length(sum_ctrl$std.err) > 0) sum_ctrl$std.err else NA_real_

      risk_t <- 1 - surv_t
      risk_c <- 1 - surv_c
      risk_diff <- risk_c - risk_t

      risk_diff_var <- if (!is.na(se_t) && !is.na(se_c)) {
        se_t^2 + se_c^2
      } else NA_real_

      baseline_risk_var <- if (!is.na(se_c)) {
        se_c^2
      } else NA_real_

      list(
        time = t,
        surv_treat = surv_t,
        se_treat = se_t,
        surv_ctrl = surv_c,
        se_ctrl = se_c,
        risk_treat = risk_t,
        risk_ctrl = risk_c,
        risk_diff = risk_diff,
        risk_diff_var = risk_diff_var,
        baseline_risk = risk_c,
        baseline_risk_var = baseline_risk_var
      )
    })

    list(
      rmst = list(
        subgroup = sg,
        rmst_treat = rmst_treat,
        rmst_ctrl = rmst_ctrl,
        rmst_diff = rmst_diff,
        tau = tau
      ),
      surv_prob = list(
        subgroup = sg,
        time_points = surv_at_times
      )
    )
  })

  results_list <- results_list[!sapply(results_list, is.null)]

  rmst_results <- lapply(results_list, function(x) x$rmst)
  names(rmst_results) <- sapply(rmst_results, function(x) x$subgroup)

  surv_prob_results <- lapply(results_list, function(x) x$surv_prob)
  names(surv_prob_results) <- sapply(surv_prob_results, function(x) x$subgroup)

  return(list(
    rmst_results = rmst_results,
    surv_prob_results = surv_prob_results
  ))
}

bootstrap_rmst_diff <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  set.seed(seed)

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  base_est <- calculate_rmst_point(results, tau = tau, verbose = TRUE)
  rmst_point <- base_est$rmst_results

  use_parallel <- !is.null(n_cores) && n_cores > 1

  if (use_parallel) {
    if (n_cores == "auto") {
      n_cores <- max(1, parallel::detectCores() - 1)
    }

    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac,
        ", cores =", n_cores, ")...\n")

    cl <- parallel::makeCluster(n_cores)

    parallel::clusterExport(cl,
                           c("calculate_rmst_point", "calculate_rmst_weighted",
                             "data_by_sg", "tau", "subsample_frac"),
                           envir = environment())

    parallel::clusterEvalQ(cl, library(survival))

    boot_results <- parallel::parLapply(cl, 1:B, function(b) {

      # Within-subgroup subsample bootstrap
      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- tryCatch(
        calculate_rmst_point(results_boot, tau = tau, verbose = FALSE),
        error = function(e) NULL
      )

      if (is.null(boot_est)) return(NULL)

      sapply(boot_est$rmst_results, function(x) x$rmst_diff)
    })

    parallel::stopCluster(cl)

    boot_results <- boot_results[!sapply(boot_results, is.null)]
    boot_mat_data <- do.call(rbind, boot_results)

    boot_mat <- lapply(seq_along(subgroup_levels), function(i) {
      boot_mat_data[, i]
    })
    names(boot_mat) <- subgroup_levels

  } else {
    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac, ")...\n")

    boot_mat <- lapply(subgroup_levels, function(sg) {
      numeric(B)
    })
    names(boot_mat) <- subgroup_levels

    for (b in seq_len(B)) {
      if (b %% 20 == 0) cat("    Iteration", b, "of", B, "\n")

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- calculate_rmst_point(results_boot, tau = tau, verbose = FALSE)
      rmst_boot <- boot_est$rmst_results

      for (sg in names(rmst_boot)) {
        boot_mat[[sg]][b] <- rmst_boot[[sg]]$rmst_diff
      }
    }
  }

  out <- lapply(rmst_point, function(x) {
    sg <- x$subgroup
    diffs <- boot_mat[[sg]]
    diffs <- diffs[is.finite(diffs)]

    if (length(diffs) < 10) {
      se <- NA_real_
      ci_lower <- NA_real_
      ci_upper <- NA_real_
    } else {
      se <- sd(diffs)
      # Percentile CI (더 robust)
      ci_lower <- quantile(diffs, 0.025, na.rm = TRUE)
      ci_upper <- quantile(diffs, 0.975, na.rm = TRUE)
    }

    c(x, list(
      se_rmst_diff = se,
      ci_lower = ci_lower,
      ci_upper = ci_upper,
      B = length(diffs),
      subsample_frac = subsample_frac
    ))
  })
  names(out) <- names(rmst_point)

  return(list(
    rmst_results = out,
    surv_prob_results = base_est$surv_prob_results
  ))
}

calculate_rmst_fast <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  cat("\n=== RMST with Subsample Bootstrap ===\n")
  cat("References:\n")
  cat("  - Zeng et al. (2022) Stat Med [bootstrap SE for weighted estimates]\n")
  cat("  - adjustedCurves::adjusted_rmst [RMST bootstrap implementation]\n\n")

  bootstrap_rmst_diff(
    results = results,
    tau = tau,
    B = B,
    subsample_frac = subsample_frac,
    seed = seed,
    n_cores = n_cores
  )
}

In [ ]:
# ==============================================================================
# Helper Function for Rubin's Pooling
# ==============================================================================
pool_variance <- function(coefs, ses, var_name = "parameter") {
  # Remove NA values
  valid_idx <- !is.na(coefs) & !is.na(ses)
  coefs <- coefs[valid_idx]
  ses <- ses[valid_idx]

  if(length(coefs) < 1) {
    warning(sprintf("No valid imputations for %s", var_name))
    return(NULL)
  }

  m_valid <- length(coefs)
  Q_bar <- mean(coefs)
  U_bar <- mean(ses^2)
  B <- var(coefs)

  # Handle NA or very small between-imputation variance
  if(is.na(B) || B < 1e-10) {
    T <- U_bar
    df <- Inf
  } else {
    T <- U_bar + (1 + 1/m_valid) * B
    df <- (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2
  }

  se_pooled <- sqrt(T)

  return(list(
    coef = Q_bar,
    se = se_pooled,
    df = df,
    n_valid = m_valid,
    within_var = U_bar,
    between_var = B
  ))
}

# ==============================================================================
# Main Pooling Function
# ==============================================================================
pool_results_part1 <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  m <- length(all_results)

  # ===== 1. Pool Main Effects (Treatment) =====
  treat_coefs <- sapply(all_results, function(x) x$main_effects$treatment_coef)
  treat_ses <- sapply(all_results, function(x) x$main_effects$treatment_se)

  treat_pooled <- pool_variance(treat_coefs, treat_ses, "treatment")

  pooled_main_effects <- list(
    treatment = list(
      coef = treat_pooled$coef,
      se = treat_pooled$se,
      hr = exp(treat_pooled$coef),
      ci_lower = exp(treat_pooled$coef - qt(0.975, treat_pooled$df) * treat_pooled$se),
      ci_upper = exp(treat_pooled$coef + qt(0.975, treat_pooled$df) * treat_pooled$se),
      pvalue = 2 * pt(-abs(treat_pooled$coef / treat_pooled$se), df = treat_pooled$df),
      df = treat_pooled$df
    )
  )

  # Pool Subgroup Main Effect (if exists)
  if(!is.null(all_results[[1]]$main_effects$subgroup_coef)) {
    subgroup_coefs <- sapply(all_results, function(x) {
      if(!is.null(x$main_effects$subgroup_coef)) {
        x$main_effects$subgroup_coef
      } else {
        NA
      }
    })

    subgroup_ses <- sapply(all_results, function(x) {
      if(!is.null(x$main_effects$subgroup_se)) {
        x$main_effects$subgroup_se
      } else {
        NA
      }
    })

    sg_pooled <- pool_variance(subgroup_coefs, subgroup_ses, "subgroup_main")

    if(!is.null(sg_pooled)) {
      pooled_main_effects$subgroup <- list(
        coef = sg_pooled$coef,
        se = sg_pooled$se,
        hr = exp(sg_pooled$coef),
        ci_lower = exp(sg_pooled$coef - qt(0.975, sg_pooled$df) * sg_pooled$se),
        ci_upper = exp(sg_pooled$coef + qt(0.975, sg_pooled$df) * sg_pooled$se),
        pvalue = 2 * pt(-abs(sg_pooled$coef / sg_pooled$se), df = sg_pooled$df),
        df = sg_pooled$df,
        var_name = all_results[[1]]$main_effects$subgroup_var
      )
    } else {
      cat("  Warning: Could not pool subgroup main effect (insufficient valid imputations)\n")
    }
  }

  # ===== 2. Pool Subgroup-Specific Results =====
  pooled_subgroups <- lapply(subgroup_names, function(sg) {
    coefs <- sapply(all_results, function(x) x$subgroup_results[[sg]]$coef)
    ses <- sapply(all_results, function(x) x$subgroup_results[[sg]]$se_robust)

    pooled <- pool_variance(coefs, ses, sprintf("subgroup_%s", sg))
    if(is.null(pooled)) return(NULL)

    list(
      subgroup = sg,
      coef = pooled$coef,
      se = pooled$se,
      hr = exp(pooled$coef),
      ci_lower = exp(pooled$coef - qt(0.975, pooled$df) * pooled$se),
      ci_upper = exp(pooled$coef + qt(0.975, pooled$df) * pooled$se),
      pvalue = 2 * pt(-abs(pooled$coef / pooled$se), df = pooled$df),
      df = pooled$df,
      n_valid_imputations = pooled$n_valid
    )
  })
  names(pooled_subgroups) <- subgroup_names
  pooled_subgroups <- pooled_subgroups[!sapply(pooled_subgroups, is.null)]

  interaction_terms <- sapply(all_results[[1]]$interaction_results, function(x) x$term)

  pooled_interactions <- lapply(interaction_terms, function(term) {
    coefs <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$coef else NA
    })

    ses <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$se_robust else NA
    })

    pooled <- pool_variance(coefs, ses, sprintf("interaction_%s", term))
    if(is.null(pooled)) return(NULL)

    list(
      term = term,
      coef = pooled$coef,
      se = pooled$se,
      pvalue = 2 * pt(-abs(pooled$coef / pooled$se), df = pooled$df),
      df = pooled$df,
      n_valid_imputations = pooled$n_valid
    )
  })
  pooled_interactions <- pooled_interactions[!sapply(pooled_interactions, is.null)]

  return(list(
    main_effects = pooled_main_effects,
    subgroup_results = pooled_subgroups,
    interaction_results = pooled_interactions
  ))
}

# ==============================================================================
# Main Pooling Function
# ==============================================================================
pool_results_part2 <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)

  pooled_rmst <- lapply(subgroup_names, function(sg) {
    rmst_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$rmst_diff)
    se_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$se_rmst_diff)

    pooled <- pool_variance(rmst_diffs, se_diffs, sprintf("rmst_%s", sg))
    if(is.null(pooled)) return(NULL)

    list(
      subgroup = sg,
      rmst_diff = pooled$coef,
      se = pooled$se,
      ci_lower = pooled$coef - qt(0.975, pooled$df) * pooled$se,
      ci_upper = pooled$coef + qt(0.975, pooled$df) * pooled$se,
      pvalue = 2 * pt(-abs(pooled$coef / pooled$se), df = pooled$df),
      tau = all_results[[1]]$rmst_results[[sg]]$tau,
      n_valid_imputations = pooled$n_valid
    )
  })
  names(pooled_rmst) <- subgroup_names
  pooled_rmst <- pooled_rmst[!sapply(pooled_rmst, is.null)]

  time_points <- c(365, 730, 1095)

  pooled_surv_probs <- lapply(subgroup_names, function(sg) {
    time_results <- lapply(seq_along(time_points), function(t_idx) {

      # Extract risk differences
      risk_diffs <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff else NA
      })

      risk_diff_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff_var else NA
      })

      # Extract baseline risks
      baseline_risks <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk else NA
      })

      baseline_risk_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$baseline_risk_var else NA
      })

      # Pool risk difference
      rd_ses <- sqrt(risk_diff_vars)
      pooled_rd <- pool_variance(risk_diffs, rd_ses,
                                  sprintf("risk_diff_%s_t%d", sg, time_points[t_idx]))
      if(is.null(pooled_rd)) return(NULL)

      # Pool baseline risk
      br_ses <- sqrt(baseline_risk_vars)
      pooled_br <- pool_variance(baseline_risks, br_ses,
                                  sprintf("baseline_risk_%s_t%d", sg, time_points[t_idx]))

      baseline_result <- if(!is.null(pooled_br)) {
        list(
          baseline_risk = pooled_br$coef,
          baseline_risk_se = pooled_br$se,
          baseline_risk_ci_lower = pooled_br$coef - qt(0.975, pooled_br$df) * pooled_br$se,
          baseline_risk_ci_upper = pooled_br$coef + qt(0.975, pooled_br$df) * pooled_br$se,
          baseline_risk_df = pooled_br$df
        )
      } else {
        list(
          baseline_risk = NA,
          baseline_risk_se = NA,
          baseline_risk_ci_lower = NA,
          baseline_risk_ci_upper = NA,
          baseline_risk_df = NA
        )
      }

      c(
        list(
          time = time_points[t_idx],
          risk_diff = pooled_rd$coef,
          se = pooled_rd$se,
          ci_lower = pooled_rd$coef - qt(0.975, pooled_rd$df) * pooled_rd$se,
          ci_upper = pooled_rd$coef + qt(0.975, pooled_rd$df) * pooled_rd$se,
          pvalue = 2 * pt(-abs(pooled_rd$coef / pooled_rd$se), df = pooled_rd$df),
          df = pooled_rd$df,
          within_var = pooled_rd$within_var,
          between_var = pooled_rd$between_var,
          fmi = (1 + 1/pooled_rd$n_valid) * pooled_rd$between_var /
                (pooled_rd$within_var + (1 + 1/pooled_rd$n_valid) * pooled_rd$between_var)
        ),
        baseline_result
      )
    })

    time_results <- time_results[!sapply(time_results, is.null)]
    list(subgroup = sg, time_results = time_results)
  })
  names(pooled_surv_probs) <- subgroup_names

  return(list(
    rmst_results = pooled_rmst,
    surv_prob_results = pooled_surv_probs
  ))
}

In [ ]:
# ==============================================================================
# Wrapper Function to Combine Parts
# ==============================================================================
pool_results <- function(all_results) {
  part1 <- pool_results_part1(all_results)
  part2 <- pool_results_part2(all_results)

  return(list(
    main_effects = part1$main_effects,
    subgroup_results = part1$subgroup_results,
    interaction_results = part1$interaction_results,
    rmst_results = part2$rmst_results,
    surv_prob_results = part2$surv_prob_results
  ))
}

In [ ]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
subgroup_var <- "Ethnicity"
covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
  "HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
  "HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
  "HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
  "HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant",
  "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone", "HasOtherAntiHypertensive",
  "HasHospitalization", "Race", "CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
  "NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
  "NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Sex", "Age", "CalendarYear",
  "AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HemoglobinValue",
  "LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
  "SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue", "HbA1cValue",
  "AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
  "AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)
covariates <- setdiff(covariates, subgroup_var)
treatment <- "FirstMedGroup"

start_time <- Sys.time()

# ==============================================================================
# FILTER OUT UNKNOWN ETHNICITY
# ==============================================================================
imputed_t2d_list <- lapply(imputed_t2d_list, function(dt) {
  dt <- as.data.frame(dt)
  dt$Race <- as.character(dt$Race)
  dt <- dt[dt$Ethnicity != "Unknown", ]
  return(dt)
})

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 1: LASSO (First Imputation Only, 50K Stratified)\n")
cat(rep("=", 80), "\n")

lasso_result <- perform_lasso_first_only(
  imp_data = imputed_t2d_list[[1]],
  covariates = covariates,
  subgroup_var = subgroup_var,
  treatment = treatment,
  sample_size = 50000
)

union_vars <- lasso_result$union_vars

cat("\n", rep("=", 80), "\n")
cat(sprintf("Selected variables: %d (including %d interactions)\n",
            length(union_vars), length(lasso_result$selected_interactions)))
cat(rep("=", 80), "\n")

# ==============================================================================
# Full Analysis
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 2: Full Analysis (All 5 Imputations - SEQUENTIAL)\n")
cat(rep("=", 80), "\n")

all_results <- list()

for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Processing Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_with_selected_vars(
    imp_data = imputed_t2d_list[[i]],
    imp_num = i,
    covariates = covariates,
    subgroup_var = subgroup_var,
    treatment = treatment,
    union_vars = union_vars,
    tau = 1095
  )

  cox_results <- complete_analysis_fast(
    data_list,
    imp_num = i,
    subgroup_var = subgroup_var,
    tau = 1095
  )

  rmst_results <- calculate_rmst_fast(
    cox_results,
    tau = 1095,
    B = 100,
    subsample_frac = 0.3,
    n_cores = 4
  )

  all_results[[i]] <- list(
    main_effects = cox_results$main_effects,
    subgroup_results = cox_results$subgroup_results,
    interaction_results = cox_results$interaction_results,
    data_by_sg = if(i == 1) cox_results$data_by_sg else NULL,
    rmst_results = rmst_results$rmst_results,
    surv_prob_results = rmst_results$surv_prob_results
  )

  cat("Imputation", i, "completed!\n")

  invisible(gc())
}

cat("\nAll imputations completed!\n")

# ==============================================================================
# Rubin's Pooling
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results(all_results)

end_time <- Sys.time()
elapsed <- as.numeric(end_time - start_time, units = "mins")

cat("\n", rep("=", 80), "\n")
cat(sprintf("Total Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))
cat(rep("=", 80), "\n")

In [ ]:
# ==============================================================================
# PRINT RESULTS
# ==============================================================================
cat("\n", rep("=", 120), "\n")
cat("FINAL POOLED RESULTS (3-Year Follow-up)\n")
cat(sprintf("Subgroup Analysis by: %s\n", subgroup_var))
cat(rep("=", 120), "\n")

# Subgroup-Specific Results
cat("\n2. SUBGROUP-SPECIFIC TREATMENT EFFECTS (Hazard Ratios)\n")
cat(rep("-", 120), "\n")

for(sg_name in names(pooled_results$subgroup_results)) {
  sg <- pooled_results$subgroup_results[[sg_name]]
  cat(sprintf("Subgroup %s: HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
              sg_name, sg$hr, sg$ci_lower, sg$ci_upper, sg$pvalue))
}

# Interaction Test
cat("\n3. INTERACTION TEST (Treatment × Ethnicity)\n")
cat(rep("-", 120), "\n")

if(length(pooled_results$interaction_results) > 0) {
  for(int in pooled_results$interaction_results) {
    cat(sprintf("%s: Coef = %.4f (SE = %.4f), p = %.4f\n",
                int$term, int$coef, int$se, int$pvalue))

    if(int$pvalue < 0.05) {
      cat("  → Significant heterogeneity on relative scale\n")
    } else {
      cat("  → No significant heterogeneity on relative scale\n")
    }
  }

  all_p <- sapply(pooled_results$interaction_results, function(x) x$pvalue)
  if(any(all_p < 0.05)) {
    cat("\nOVERALL: Significant treatment effect heterogeneity detected\n")
  } else {
    cat("\nOVERALL: No significant treatment effect heterogeneity\n")
  }
} else {
  cat("No interaction terms available\n")
}

# RMST
cat("\n4. RESTRICTED MEAN SURVIVAL TIME (3-year RMST)\n")
cat(rep("-", 120), "\n")

for(sg_name in names(pooled_results$rmst_results)) {
  rmst <- pooled_results$rmst_results[[sg_name]]
  cat(sprintf("Subgroup %s: RMST Diff = %.1f days (95%% CI: %.1f-%.1f), p = %.4f\n",
              sg_name, rmst$rmst_diff, rmst$ci_lower, rmst$ci_upper, rmst$pvalue))
  cat(sprintf("  → Treatment extends by %.1f days of event-free survival\n", rmst$rmst_diff))
}

# Absolute Risk Differences
cat("\n5. ABSOLUTE RISK DIFFERENCES\n")
cat(rep("-", 120), "\n\n")

for(sg_name in names(pooled_results$surv_prob_results)) {
  cat(sprintf("Subgroup %s:\n", sg_name))

  surv_res <- pooled_results$surv_prob_results[[sg_name]]$time_results

  for(tr in surv_res) {
    cat(sprintf("  At %.1f years: Risk Diff = %.4f (95%% CI: %.4f-%.4f)\n",
                tr$time / 365, tr$risk_diff, tr$ci_lower, tr$ci_upper))
    cat(sprintf("    SE = %.4f, p = %.4f (%.2f%% reduction)\n",
                tr$se, tr$pvalue, abs(tr$risk_diff) * 100))
    cat(sprintf("    FMI = %.3f (%.1f%% of variance from imputation uncertainty)\n",
                tr$fmi, tr$fmi * 100))
  }
  cat("\n")
}

cat("\n", rep("=", 120), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 120), "\n")

In [ ]:
# ==============================================================================
# SUBGROUP SAMPLE STATISTICS
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("6. SUBGROUP SAMPLE STATISTICS\n")
cat(rep("=", 80), "\n")

cat("\n6-1. ORIGINAL DATA (Before Overlap Weighting)\n")
cat(rep("-", 80), "\n\n")

cat("A. Overall Dataset Statistics\n")
cat(rep("-", 40), "\n")

overall_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := factor(get(subgroup_var), levels = c("1065359", "1065401"))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  total_n <- nrow(dt)

  gender_counts <- dt[, .N, by = Si]

  list(
    total_n = total_n,
    gender_counts = gender_counts
  )
})

avg_total_n <- round(mean(sapply(overall_stats, function(x) x$total_n)))

avg_gender_counts <- lapply(c("1065359", "1065401"), function(sg) {
  counts <- sapply(overall_stats, function(x) {
    x$gender_counts[x$gender_counts$Si == sg, ]$N
  })
  list(
    subgroup = sg,
    count = round(mean(counts))
  )
})

cat(sprintf("Total Sample Size: n = %d\n\n", avg_total_n))
cat("Ethnicity Distribution:\n")
for(gc in avg_gender_counts) {
  cat(sprintf("  %s: n = %d (%.1f%%)\n",
              gc$subgroup, gc$count, gc$count/avg_total_n*100))
}
cat("\n")

cat("\nB. Statistics by Ethnicity Subgroup\n")
cat(rep("-", 40), "\n\n")

original_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := factor(get(subgroup_var), levels = c("1065359", "1065401"))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  stats_by_sg <- dt[, .(
    n_total = .N,
    n_treat = sum(Z == 1),
    n_ctrl = sum(Z == 0),
    events_total = sum(Outcome),
    events_treat = sum(Outcome[Z == 1]),
    events_ctrl = sum(Outcome[Z == 0])
  ), by = Si]

  setDF(stats_by_sg)
  stats_by_sg
})

avg_original_stats <- lapply(unique(original_stats[[1]]$Si), function(sg) {
  n_total_vec <- sapply(original_stats, function(x) x$n_total[x$Si == sg])
  n_treat_vec <- sapply(original_stats, function(x) x$n_treat[x$Si == sg])
  n_ctrl_vec <- sapply(original_stats, function(x) x$n_ctrl[x$Si == sg])
  events_total_vec <- sapply(original_stats, function(x) x$events_total[x$Si == sg])
  events_treat_vec <- sapply(original_stats, function(x) x$events_treat[x$Si == sg])
  events_ctrl_vec <- sapply(original_stats, function(x) x$events_ctrl[x$Si == sg])

  list(
    subgroup = as.character(sg),
    n_total = round(mean(n_total_vec)),
    n_treat = round(mean(n_treat_vec)),
    n_ctrl = round(mean(n_ctrl_vec)),
    events_total = round(mean(events_total_vec)),
    events_treat = round(mean(events_treat_vec)),
    events_ctrl = round(mean(events_ctrl_vec))
  )
})

for(stats in avg_original_stats) {
  cat(sprintf("Subgroup: %s\n", stats$subgroup))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size:\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat("\n6-2. AFTER OVERLAP WEIGHTING (Pseudo Population)\n")
cat(rep("-", 80), "\n\n")

subgroup_levels <- names(all_results[[1]]$data_by_sg)

avg_weighted_stats <- lapply(subgroup_levels, function(sg) {

  stats_list <- lapply(1:5, function(i) {
    data_sg <- all_results[[i]]$data_by_sg[[sg]]

    if(is.null(data_sg)) return(NULL)

    n_total <- nrow(data_sg)
    n_events <- sum(data_sg$Outcome)
    ess_total <- sum(data_sg$w)^2 / sum(data_sg$w^2)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl <- data_sg[data_sg$Z == 0, ]

    n_treat <- nrow(data_treat)
    n_ctrl <- nrow(data_ctrl)
    events_treat <- sum(data_treat$Outcome)
    events_ctrl <- sum(data_ctrl$Outcome)
    ess_treat <- if(n_treat > 0) sum(data_treat$w)^2 / sum(data_treat$w^2) else 0
    ess_ctrl <- if(n_ctrl > 0) sum(data_ctrl$w)^2 / sum(data_ctrl$w^2) else 0

    list(
      n_total = n_total,
      n_treat = n_treat,
      n_ctrl = n_ctrl,
      events_total = n_events,
      events_treat = events_treat,
      events_ctrl = events_ctrl,
      ess_total = ess_total,
      ess_treat = ess_treat,
      ess_ctrl = ess_ctrl
    )
  })

  stats_list <- stats_list[!sapply(stats_list, is.null)]

  list(
    subgroup = sg,
    n_total = round(mean(sapply(stats_list, function(x) x$n_total))),
    n_treat = round(mean(sapply(stats_list, function(x) x$n_treat))),
    n_ctrl = round(mean(sapply(stats_list, function(x) x$n_ctrl))),
    events_total = round(mean(sapply(stats_list, function(x) x$events_total))),
    events_treat = round(mean(sapply(stats_list, function(x) x$events_treat))),
    events_ctrl = round(mean(sapply(stats_list, function(x) x$events_ctrl))),
    ess_total = round(mean(sapply(stats_list, function(x) x$ess_total)), 1),
    ess_treat = round(mean(sapply(stats_list, function(x) x$ess_treat)), 1),
    ess_ctrl = round(mean(sapply(stats_list, function(x) x$ess_ctrl)), 1)
  )
})
names(avg_weighted_stats) <- subgroup_levels

for(sg in names(avg_weighted_stats)) {
  stats <- avg_weighted_stats[[sg]]

  cat(sprintf("Subgroup: %s\n", sg))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size (Original):\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Effective Sample Size (ESS):\n"))
  cat(sprintf("    Total:     ESS = %.1f\n", stats$ess_total))
  cat(sprintf("    Treatment: ESS = %.1f\n", stats$ess_treat))
  cat(sprintf("    Control:   ESS = %.1f\n", stats$ess_ctrl))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat(rep("=", 80), "\n")
cat("Note:\n")
cat("  - Original n = Actual number of patients before weighting\n")
cat("  - ESS = Effective Sample Size after overlap weighting\n")
cat("  - ESS = (sum of weights)^2 / sum of squared weights\n")
cat("  - ESS represents the 'pseudo population' size for inference\n")
cat(rep("=", 80), "\n")

In [ ]:
data_by_sg <- all_results[[1]]$data_by_sg

data_hispanic <- data_by_sg[["1065359"]]
data_h_treat <- data_hispanic[data_hispanic$Z == 1, ]
data_h_ctrl <- data_hispanic[data_hispanic$Z == 0, ]

fit_h_treat <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_h_treat, weights = w)
fit_h_ctrl <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_h_ctrl, weights = w)

data_nonhispanic <- data_by_sg[["1065401"]]
data_nh_treat <- data_nonhispanic[data_nonhispanic$Z == 1, ]
data_nh_ctrl <- data_nonhispanic[data_nonhispanic$Z == 0, ]

fit_nh_treat <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_nh_treat, weights = w)
fit_nh_ctrl <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_nh_ctrl, weights = w)

image_path <- file.path(artifacts_images_dir, "t2d_secondary_primary_endpoint_ethnicity.png")

draw_plot <- function() {
  par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

  plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.20),
       xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
       las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

  axis(2, at = seq(0, 0.20, by = 0.05),
       labels = paste0(seq(0, 20, by = 5), "%"),
       las = 1, cex.axis = 1.5)

  abline(h = seq(0, 0.20, by = 0.05), col = "gray50", lwd = 0.5)

  lines(c(0, fit_h_treat$time / 30.44), c(0, 1 - fit_h_treat$surv),
        col = "#E41A1C", lty = 1, lwd = 2.5)
  lines(c(0, fit_h_ctrl$time / 30.44), c(0, 1 - fit_h_ctrl$surv),
        col = "#E41A1C", lty = 3, lwd = 2.5)
  lines(c(0, fit_nh_treat$time / 30.44), c(0, 1 - fit_nh_treat$surv),
        col = "#377EB8", lty = 1, lwd = 2.5)
  lines(c(0, fit_nh_ctrl$time / 30.44), c(0, 1 - fit_nh_ctrl$surv),
        col = "#377EB8", lty = 3, lwd = 2.5)

#   legend("topleft",
#          legend = c("Hispanic - GLP-1 RAs", "Hispanic - Non-GLP-1 RAs",
#                     "Non-Hispanic - GLP-1 RAs", "Non-Hispanic - Non-GLP-1 RAs"),
#          col = c("#E41A1C", "#E41A1C", "#377EB8", "#377EB8"),
#          lty = c(1, 3, 1, 3),
#          lwd = 2.5,
#          bty = "n",
#          cex = 1.4)
}

png(filename = image_path, width = 9.5, height = 7, units = "in", res = 300)
draw_plot()
dev.off()

cat("Plot saved to:", image_path, "\n")
draw_plot()

## DCSI Score

In [ ]:
setDTthreads(0)

# ==============================================================================
# LASSO Variable Selection
# ==============================================================================
perform_lasso_first_only <- function(imp_data, covariates, subgroup_var, treatment,
                                     sample_size = 50000) {

  cat("\n=== LASSO (First Imputation Only, Stratified Sampling) ===\n")

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment)
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(sample_size / n_groups)

  cat(sprintf("  Total groups (Si × Z): %d\n", n_groups))
  cat(sprintf("  Target per group: %d\n", sample_per_group))

  # Stratified sampling by Si × Z
  set.seed(12345)
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]
  cat(sprintf("  Actual sample: %d obs\n", nrow(dt_sample)))

  factor_cols <- names(dt_sample)[sapply(dt_sample, is.factor) | sapply(dt_sample, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt_sample[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  # Sparse matrix
  X_df <- dt_sample[, ..covariates]
  X <- sparse.model.matrix(~ . - 1, data = X_df)
  Si <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  n_subgroups <- ncol(Si)

  cat("  Creating interactions...\n")

  Xi_Si_list <- lapply(1:n_subgroups, function(k) {
    si_k <- as.vector(Si[, k])
    X_k <- X * si_k
    colnames(X_k) <- paste0(colnames(X), ":", colnames(Si)[k])
    X_k
  })

  Xi_Si <- do.call(cbind, Xi_Si_list)
  rm(Xi_Si_list); invisible(gc())

  design_matrix <- cbind(X, Si, Xi_Si)

  # Penalty factor
  penalty.factor <- c(
    rep(0, ncol(X) + ncol(Si)),
    rep(1, ncol(Xi_Si))
  )

  cat("  Running LASSO...\n")

  # LASSO
  lasso_fit <- cv.glmnet(
    design_matrix, dt_sample$Z,
    family = "binomial",
    alpha = 1,
    penalty.factor = penalty.factor,
    nfolds = 3,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 2000
  )

  selected_coef <- coef(lasso_fit, s = "lambda.min")
  selected_vars <- rownames(selected_coef)[selected_coef[, 1] != 0]
  selected_vars <- selected_vars[selected_vars != "(Intercept)"]
  selected_interactions <- selected_vars[grepl(":", selected_vars)]

  cat(sprintf("  Selected %d interactions\n", length(selected_interactions)))

  # Base variables + selected interactions
  union_vars <- c(colnames(X), colnames(Si), selected_interactions)

  return(list(
    union_vars = union_vars,
    selected_interactions = selected_interactions,
    base_vars = c(colnames(X), colnames(Si))
  ))
}

# ==============================================================================
# Propensity Score Estimation with Selected Variables
# ==============================================================================
analyze_with_selected_vars <- function(imp_data, imp_num,
                                       covariates, subgroup_var, treatment,
                                       union_vars, tau = 1095) {

  cat(sprintf("\n=== Processing Imputation %d ===\n", imp_num))

  dt <- as.data.table(imp_data)
  needed_cols <- c(covariates, subgroup_var, treatment, "Survival_Time", "Outcome")
  dt <- dt[, ..needed_cols]

  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  n_total <- nrow(dt)
  cat(sprintf("  Dataset: %d obs\n", n_total))

  factor_cols <- names(dt)[sapply(dt, is.factor) | sapply(dt, is.character)]
  factor_cols <- setdiff(factor_cols, c("Si", "Z"))

  if(length(factor_cols) > 0) {
    dt[, (factor_cols) := lapply(.SD, as.factor), .SDcols = factor_cols]
  }

  set.seed(456 + imp_num)
  target_sample_size <- min(300000, n_total)

  n_groups <- uniqueN(dt, by = c("Si", "Z"))
  sample_per_group <- ceiling(target_sample_size / n_groups)

  cat(sprintf("  Target sample: %d (stratified by Si × Z)\n", target_sample_size))

  # Stratified sampling
  sample_idx <- dt[, {
    n <- min(.N, sample_per_group)
    .I[sample.int(.N, n)]
  }, by = .(Si, Z)]$V1

  dt_sample <- dt[sample_idx]

  cat(sprintf("  Actual sample: %d\n", nrow(dt_sample)))

  X_df_sample <- dt_sample[, ..covariates]
  X_sample <- sparse.model.matrix(~ . - 1, data = X_df_sample)
  Si_sample <- sparse.model.matrix(~ Si - 1, data = dt_sample)

  interaction_names <- union_vars[grepl(":", union_vars)]

  if(length(interaction_names) > 0 && length(interaction_names) < 800) {
    cat(sprintf("  Creating %d interactions (sample)...\n", length(interaction_names)))

    n_subgroups <- ncol(Si_sample)
    Xi_Si_list <- list()

    for(k in 1:n_subgroups) {
      si_k <- as.vector(Si_sample[, k])
      X_k <- X_sample * si_k
      colnames(X_k) <- paste0(colnames(X_sample), ":", colnames(Si_sample)[k])

      selected_cols <- intersect(colnames(X_k), interaction_names)
      if(length(selected_cols) > 0) {
        Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols, drop = FALSE]
      }
    }

    if(length(Xi_Si_list) > 0) {
      Xi_Si_sample <- do.call(cbind, Xi_Si_list)
      design_sample <- cbind(X_sample, Si_sample, Xi_Si_sample)
      rm(Xi_Si_list); invisible(gc())
    } else {
      design_sample <- cbind(X_sample, Si_sample)
    }
  } else {
    cat("  Too many interactions, using main effects only\n")
    design_sample <- cbind(X_sample, Si_sample)
  }

  all_vars <- colnames(design_sample)
  selected_cols <- intersect(union_vars, all_vars)
  design_selected_sample <- design_sample[, selected_cols, drop = FALSE]

  cat(sprintf("  Design matrix (sample): %d × %d\n",
              nrow(design_selected_sample), ncol(design_selected_sample)))

  cat("  Fitting PS model...\n")

  ps_model <- glmnet(
    design_selected_sample, dt_sample$Z,
    family = "binomial",
    alpha = 0,
    lambda = 0.01,
    standardize = TRUE,
    thresh = 1e-2,
    maxit = 1000
  )

  cat("  Predicting on full data...\n")

  chunk_size <- 200000
  n_chunks <- ceiling(n_total / chunk_size)
  ps_all <- numeric(n_total)

  for(chunk_i in 1:n_chunks) {
    if(chunk_i %% 2 == 0) {
      cat(sprintf("\r  Chunk %d/%d", chunk_i, n_chunks))
    }

    start_idx <- (chunk_i - 1) * chunk_size + 1
    end_idx <- min(chunk_i * chunk_size, n_total)

    dt_chunk <- dt[start_idx:end_idx]

    # Design matrix for chunk
    X_df_chunk <- dt_chunk[, ..covariates]
    X_chunk <- sparse.model.matrix(~ . - 1, data = X_df_chunk)
    Si_chunk <- sparse.model.matrix(~ Si - 1, data = dt_chunk)

    if(length(interaction_names) > 0 && length(interaction_names) < 800) {
      n_subgroups <- ncol(Si_chunk)
      Xi_Si_list <- list()

      for(k in 1:n_subgroups) {
        si_k <- as.vector(Si_chunk[, k])
        X_k <- X_chunk * si_k
        colnames(X_k) <- paste0(colnames(X_chunk), ":", colnames(Si_chunk)[k])

        selected_cols_chunk <- intersect(colnames(X_k), interaction_names)
        if(length(selected_cols_chunk) > 0) {
          Xi_Si_list[[length(Xi_Si_list) + 1]] <- X_k[, selected_cols_chunk, drop = FALSE]
        }
      }

      if(length(Xi_Si_list) > 0) {
        Xi_Si_chunk <- do.call(cbind, Xi_Si_list)
        design_chunk <- cbind(X_chunk, Si_chunk, Xi_Si_chunk)
        rm(Xi_Si_list)
      } else {
        design_chunk <- cbind(X_chunk, Si_chunk)
      }
    } else {
      design_chunk <- cbind(X_chunk, Si_chunk)
    }

    design_chunk <- design_chunk[, selected_cols, drop = FALSE]

    ps_all[start_idx:end_idx] <- as.vector(
      predict(ps_model, newx = design_chunk, type = "response", s = 0.01)
    )

    rm(X_chunk, Si_chunk, design_chunk)
    invisible(gc())
  }
  cat("\n")

  dt[, ps := ps_all]

  # Overlap weights
  dt[, w := fifelse(Z == 1L, 1 - ps, ps)]
  dt[, w := w / sum(w), by = .(Si, Z)]

  data <- dt[, .(Si, Z, w, Survival_Time, Outcome)]
  setDF(data)

  return(list(data = data, X = NULL, Si = NULL))
}

In [ ]:
# ==============================================================================
# Cox Regression Analysis
# ==============================================================================
complete_analysis_fast <- function(data_list, imp_num, subgroup_var, tau = 1095) {

  data <- data_list$data
  data <- data[data$w > 1e-8, ]

  data$Si <- factor(data$Si, levels = c("0", "+1"))
  subgroup_levels <- levels(data$Si)

  cat(sprintf("  Cox models... (Subgroups: %s, Reference: %s)\n",
              paste(subgroup_levels, collapse = ", "),
              subgroup_levels[1]))

  cox_main <- coxph(
    Surv(Survival_Time, Outcome) ~ Z + Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  main_summary <- summary(cox_main)

  main_effects <- list(
    treatment_coef = main_summary$coefficients["Z", "coef"],
    treatment_se = main_summary$coefficients["Z", "robust se"],
    treatment_hr = main_summary$coefficients["Z", "exp(coef)"],
    treatment_pvalue = main_summary$coefficients["Z", "Pr(>|z|)"]
  )

  si_coef_names <- grep("^Si", rownames(main_summary$coefficients), value = TRUE)

  if(length(si_coef_names) > 0) {
    main_effects$subgroup_coef <- main_summary$coefficients[si_coef_names[1], "coef"]
    main_effects$subgroup_se <- main_summary$coefficients[si_coef_names[1], "robust se"]
    main_effects$subgroup_hr <- main_summary$coefficients[si_coef_names[1], "exp(coef)"]
    main_effects$subgroup_pvalue <- main_summary$coefficients[si_coef_names[1], "Pr(>|z|)"]
    main_effects$subgroup_var <- subgroup_var
    main_effects$subgroup_term <- si_coef_names[1]
    main_effects$subgroup_reference <- subgroup_levels[1]

    cat(sprintf("  Subgroup main effect: %s vs %s (Ref), HR = %.3f\n",
                si_coef_names[1], subgroup_levels[1], main_effects$subgroup_hr))
  } else {
    cat("  Warning: No subgroup main effect coefficient found\n")
    main_effects$subgroup_coef <- NA
    main_effects$subgroup_se <- NA
  }

  # Subgroup-Specific Models
  data_by_sg <- split(data, data$Si)

  subgroup_results <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if(nrow(data_sg) > 10) {
      cox_fit <- coxph(
        Surv(Survival_Time, Outcome) ~ Z,
        data = data_sg,
        weights = w,
        robust = TRUE,
        method = "efron",
        model = FALSE, x = FALSE, y = FALSE,
        control = coxph.control(iter.max = 15)
      )

      coef_summary <- summary(cox_fit)

      list(
        subgroup = sg,
        coef = coef_summary$coefficients["Z", "coef"],
        se_robust = coef_summary$coefficients["Z", "robust se"],
        hr = coef_summary$coefficients["Z", "exp(coef)"],
        pvalue_robust = coef_summary$coefficients["Z", "Pr(>|z|)"]
      )
    } else {
      NULL
    }
  })

  subgroup_results <- subgroup_results[!sapply(subgroup_results, is.null)]
  names(subgroup_results) <- sapply(subgroup_results, function(x) x$subgroup)

  # Interaction Model
  cox_interaction <- coxph(
    Surv(Survival_Time, Outcome) ~ Z * Si,
    data = data,
    weights = w,
    robust = TRUE,
    method = "efron",
    model = FALSE, x = FALSE, y = FALSE,
    control = coxph.control(iter.max = 15)
  )

  interaction_summary <- summary(cox_interaction)

  all_coef_names <- rownames(interaction_summary$coefficients)
  cat(sprintf("  All coefficients: %s\n", paste(all_coef_names, collapse = ", ")))

  interaction_coefs <- grep("Z:Si", all_coef_names, value = TRUE)
  cat(sprintf("  Interaction terms: %s\n", paste(interaction_coefs, collapse = ", ")))

  interaction_results <- lapply(interaction_coefs, function(coef_name) {
    list(
      term = coef_name,
      coef = interaction_summary$coefficients[coef_name, "coef"],
      se_robust = interaction_summary$coefficients[coef_name, "robust se"],
      pvalue_robust = interaction_summary$coefficients[coef_name, "Pr(>|z|)"]
    )
  })

  return(list(
    main_effects = main_effects,
    subgroup_results = subgroup_results,
    interaction_results = interaction_results,
    data = data,
    data_by_sg = data_by_sg
  ))
}

In [ ]:
# ==============================================================================
# RMST and Risk Difference Calculation
# ==============================================================================

calculate_rmst_weighted <- function(time, status, weights, tau) {

  ord <- order(time)
  time <- time[ord]
  status <- status[ord]
  weights <- weights[ord]

  w_atrisk <- rev(cumsum(rev(weights)))
  w_event  <- status * weights

  w_surv <- cumprod(1 - w_event / pmax(w_atrisk, 1e-10))
  w_surv <- c(1, w_surv)
  time_ext <- c(0, time)

  idx <- time_ext <= tau
  time_use <- time_ext[idx]
  surv_use <- w_surv[idx]

  if (max(time_use) < tau) {
    time_use <- c(time_use, tau)
    surv_use <- c(surv_use, tail(surv_use, 1))
  }

  rmst <- sum(diff(time_use) * head(surv_use, -1))
  return(rmst)
}

calculate_rmst_point <- function(results, tau = 1095, verbose = TRUE) {

  if (verbose) cat("  RMST (weighted KM, point estimates only)...\n")

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  results_list <- lapply(subgroup_levels, function(sg) {
    data_sg <- data_by_sg[[sg]]

    if (nrow(data_sg) <= 10) return(NULL)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl  <- data_sg[data_sg$Z == 0, ]

    if (nrow(data_treat) <= 10 || nrow(data_ctrl) <= 10) return(NULL)

    # Weighted Kaplan–Meier
    fit_treat <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_treat,
      weights = data_treat$w
    )
    fit_ctrl <- survfit(
      Surv(Survival_Time, Outcome) ~ 1,
      data = data_ctrl,
      weights = data_ctrl$w
    )

    # RMST
    rmst_treat <- tryCatch(
      calculate_rmst_weighted(
        time = data_treat$Survival_Time,
        status = data_treat$Outcome,
        weights = data_treat$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_ctrl <- tryCatch(
      calculate_rmst_weighted(
        time = data_ctrl$Survival_Time,
        status = data_ctrl$Outcome,
        weights = data_ctrl$w,
        tau = tau
      ),
      error = function(e) NA_real_
    )

    rmst_diff <- rmst_treat - rmst_ctrl

    # Risk Differences
    time_points <- c(365, 730, 1095)
    surv_at_times <- lapply(time_points, function(t) {
      sum_treat <- summary(fit_treat, times = t, extend = TRUE)
      sum_ctrl  <- summary(fit_ctrl,  times = t, extend = TRUE)

      surv_t <- if (length(sum_treat$surv) > 0) sum_treat$surv else NA_real_
      surv_c <- if (length(sum_ctrl$surv) > 0) sum_ctrl$surv else NA_real_
      se_t   <- if (length(sum_treat$std.err) > 0) sum_treat$std.err else NA_real_
      se_c   <- if (length(sum_ctrl$std.err) > 0) sum_ctrl$std.err else NA_real_

      risk_t <- 1 - surv_t
      risk_c <- 1 - surv_c
      risk_diff <- risk_c - risk_t

      risk_diff_var <- if (!is.na(se_t) && !is.na(se_c)) {
        se_t^2 + se_c^2
      } else NA_real_

      baseline_risk_var <- if (!is.na(se_c)) {
        se_c^2
      } else NA_real_

      list(
        time = t,
        surv_treat = surv_t,
        se_treat = se_t,
        surv_ctrl = surv_c,
        se_ctrl = se_c,
        risk_treat = risk_t,
        risk_ctrl = risk_c,
        risk_diff = risk_diff,
        risk_diff_var = risk_diff_var,
        baseline_risk = risk_c,
        baseline_risk_var = baseline_risk_var
      )
    })

    list(
      rmst = list(
        subgroup = sg,
        rmst_treat = rmst_treat,
        rmst_ctrl = rmst_ctrl,
        rmst_diff = rmst_diff,
        tau = tau
      ),
      surv_prob = list(
        subgroup = sg,
        time_points = surv_at_times
      )
    )
  })

  results_list <- results_list[!sapply(results_list, is.null)]

  rmst_results <- lapply(results_list, function(x) x$rmst)
  names(rmst_results) <- sapply(rmst_results, function(x) x$subgroup)

  surv_prob_results <- lapply(results_list, function(x) x$surv_prob)
  names(surv_prob_results) <- sapply(surv_prob_results, function(x) x$subgroup)

  return(list(
    rmst_results = rmst_results,
    surv_prob_results = surv_prob_results
  ))
}

bootstrap_rmst_diff <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  set.seed(seed)

  data_by_sg <- results$data_by_sg
  subgroup_levels <- names(data_by_sg)

  base_est <- calculate_rmst_point(results, tau = tau, verbose = TRUE)
  rmst_point <- base_est$rmst_results

  use_parallel <- !is.null(n_cores) && n_cores > 1

  if (use_parallel) {
    if (n_cores == "auto") {
      n_cores <- max(1, parallel::detectCores() - 1)
    }

    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac,
        ", cores =", n_cores, ")...\n")

    cl <- parallel::makeCluster(n_cores)

    parallel::clusterExport(cl,
                           c("calculate_rmst_point", "calculate_rmst_weighted",
                             "data_by_sg", "tau", "subsample_frac"),
                           envir = environment())

    parallel::clusterEvalQ(cl, library(survival))

    boot_results <- parallel::parLapply(cl, 1:B, function(b) {

      # Within-subgroup subsample bootstrap
      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- tryCatch(
        calculate_rmst_point(results_boot, tau = tau, verbose = FALSE),
        error = function(e) NULL
      )

      if (is.null(boot_est)) return(NULL)

      sapply(boot_est$rmst_results, function(x) x$rmst_diff)
    })

    parallel::stopCluster(cl)

    boot_results <- boot_results[!sapply(boot_results, is.null)]
    boot_mat_data <- do.call(rbind, boot_results)

    boot_mat <- lapply(seq_along(subgroup_levels), function(i) {
      boot_mat_data[, i]
    })
    names(boot_mat) <- subgroup_levels

  } else {
    cat("  Bootstrap RMST diff (B =", B,
        ", subsample frac =", subsample_frac, ")...\n")

    boot_mat <- lapply(subgroup_levels, function(sg) {
      numeric(B)
    })
    names(boot_mat) <- subgroup_levels

    for (b in seq_len(B)) {
      if (b %% 20 == 0) cat("    Iteration", b, "of", B, "\n")

      data_by_sg_boot <- lapply(data_by_sg, function(df) {
        n  <- nrow(df)
        nb <- max(10, round(n * subsample_frac))
        idx <- sample.int(n, size = nb, replace = TRUE)
        df[idx, , drop = FALSE]
      })

      results_boot <- list(data_by_sg = data_by_sg_boot)
      boot_est <- calculate_rmst_point(results_boot, tau = tau, verbose = FALSE)
      rmst_boot <- boot_est$rmst_results

      for (sg in names(rmst_boot)) {
        boot_mat[[sg]][b] <- rmst_boot[[sg]]$rmst_diff
      }
    }
  }

  out <- lapply(rmst_point, function(x) {
    sg <- x$subgroup
    diffs <- boot_mat[[sg]]
    diffs <- diffs[is.finite(diffs)]

    if (length(diffs) < 10) {
      se <- NA_real_
      ci_lower <- NA_real_
      ci_upper <- NA_real_
    } else {
      se <- sd(diffs)
      # Percentile CI
      ci_lower <- quantile(diffs, 0.025, na.rm = TRUE)
      ci_upper <- quantile(diffs, 0.975, na.rm = TRUE)
    }

    c(x, list(
      se_rmst_diff = se,
      ci_lower = ci_lower,
      ci_upper = ci_upper,
      B = length(diffs),
      subsample_frac = subsample_frac
    ))
  })
  names(out) <- names(rmst_point)

  return(list(
    rmst_results = out,
    surv_prob_results = base_est$surv_prob_results
  ))
}

calculate_rmst_fast <- function(results,
                                tau = 1095,
                                B = 100,
                                subsample_frac = 0.3,
                                seed = 12345,
                                n_cores = NULL) {

  bootstrap_rmst_diff(
    results = results,
    tau = tau,
    B = B,
    subsample_frac = subsample_frac,
    seed = seed,
    n_cores = n_cores
  )
}

In [ ]:
# ==============================================================================
# Helper Function for Rubin's Pooling
# ==============================================================================
pool_variance <- function(coefs, ses, var_name = "parameter") {
  # Remove NA values
  valid_idx <- !is.na(coefs) & !is.na(ses)
  coefs <- coefs[valid_idx]
  ses <- ses[valid_idx]

  if(length(coefs) < 1) {
    warning(sprintf("No valid imputations for %s", var_name))
    return(NULL)
  }

  m_valid <- length(coefs)
  Q_bar <- mean(coefs)
  U_bar <- mean(ses^2)
  B <- var(coefs)

  # Handle NA or very small between-imputation variance
  if(is.na(B) || B < 1e-10) {
    T <- U_bar
    df <- Inf
  } else {
    T <- U_bar + (1 + 1/m_valid) * B
    df <- (m_valid - 1) * (1 + U_bar / ((1 + 1/m_valid) * B))^2
  }

  se_pooled <- sqrt(T)

  return(list(
    coef = Q_bar,
    se = se_pooled,
    df = df,
    n_valid = m_valid,
    within_var = U_bar,
    between_var = B
  ))
}

# ==============================================================================
# Main Pooling Function
# ==============================================================================
pool_results_part1 <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)
  m <- length(all_results)

  treat_coefs <- sapply(all_results, function(x) x$main_effects$treatment_coef)
  treat_ses <- sapply(all_results, function(x) x$main_effects$treatment_se)

  treat_pooled <- pool_variance(treat_coefs, treat_ses, "treatment")

  pooled_main_effects <- list(
    treatment = list(
      coef = treat_pooled$coef,
      se = treat_pooled$se,
      hr = exp(treat_pooled$coef),
      ci_lower = exp(treat_pooled$coef - qt(0.975, treat_pooled$df) * treat_pooled$se),
      ci_upper = exp(treat_pooled$coef + qt(0.975, treat_pooled$df) * treat_pooled$se),
      pvalue = 2 * pt(-abs(treat_pooled$coef / treat_pooled$se), df = treat_pooled$df),
      df = treat_pooled$df
    )
  )

  # Pool Subgroup Main Effect (if exists)
  if(!is.null(all_results[[1]]$main_effects$subgroup_coef)) {
    subgroup_coefs <- sapply(all_results, function(x) {
      if(!is.null(x$main_effects$subgroup_coef)) {
        x$main_effects$subgroup_coef
      } else {
        NA
      }
    })

    subgroup_ses <- sapply(all_results, function(x) {
      if(!is.null(x$main_effects$subgroup_se)) {
        x$main_effects$subgroup_se
      } else {
        NA
      }
    })

    sg_pooled <- pool_variance(subgroup_coefs, subgroup_ses, "subgroup_main")

    if(!is.null(sg_pooled)) {
      pooled_main_effects$subgroup <- list(
        coef = sg_pooled$coef,
        se = sg_pooled$se,
        hr = exp(sg_pooled$coef),
        ci_lower = exp(sg_pooled$coef - qt(0.975, sg_pooled$df) * sg_pooled$se),
        ci_upper = exp(sg_pooled$coef + qt(0.975, sg_pooled$df) * sg_pooled$se),
        pvalue = 2 * pt(-abs(sg_pooled$coef / sg_pooled$se), df = sg_pooled$df),
        df = sg_pooled$df,
        var_name = all_results[[1]]$main_effects$subgroup_var
      )
    } else {
      cat("  Warning: Could not pool subgroup main effect (insufficient valid imputations)\n")
    }
  }

  pooled_subgroups <- lapply(subgroup_names, function(sg) {
    coefs <- sapply(all_results, function(x) x$subgroup_results[[sg]]$coef)
    ses <- sapply(all_results, function(x) x$subgroup_results[[sg]]$se_robust)

    pooled <- pool_variance(coefs, ses, sprintf("subgroup_%s", sg))
    if(is.null(pooled)) return(NULL)

    list(
      subgroup = sg,
      coef = pooled$coef,
      se = pooled$se,
      hr = exp(pooled$coef),
      ci_lower = exp(pooled$coef - qt(0.975, pooled$df) * pooled$se),
      ci_upper = exp(pooled$coef + qt(0.975, pooled$df) * pooled$se),
      pvalue = 2 * pt(-abs(pooled$coef / pooled$se), df = pooled$df),
      df = pooled$df,
      n_valid_imputations = pooled$n_valid
    )
  })
  names(pooled_subgroups) <- subgroup_names
  pooled_subgroups <- pooled_subgroups[!sapply(pooled_subgroups, is.null)]

  interaction_terms <- sapply(all_results[[1]]$interaction_results, function(x) x$term)

  pooled_interactions <- lapply(interaction_terms, function(term) {
    coefs <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$coef else NA
    })

    ses <- sapply(all_results, function(x) {
      res <- x$interaction_results[sapply(x$interaction_results, function(y) y$term == term)]
      if(length(res) > 0) res[[1]]$se_robust else NA
    })

    pooled <- pool_variance(coefs, ses, sprintf("interaction_%s", term))
    if(is.null(pooled)) return(NULL)

    list(
      term = term,
      coef = pooled$coef,
      se = pooled$se,
      pvalue = 2 * pt(-abs(pooled$coef / pooled$se), df = pooled$df),
      df = pooled$df,
      n_valid_imputations = pooled$n_valid
    )
  })
  pooled_interactions <- pooled_interactions[!sapply(pooled_interactions, is.null)]

  return(list(
    main_effects = pooled_main_effects,
    subgroup_results = pooled_subgroups,
    interaction_results = pooled_interactions
  ))
}

# ==============================================================================
# Main Pooling Function
# ==============================================================================
pool_results_part2 <- function(all_results) {

  subgroup_names <- names(all_results[[1]]$subgroup_results)

  # ===== 4. Pool RMST Differences =====
  pooled_rmst <- lapply(subgroup_names, function(sg) {
    rmst_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$rmst_diff)
    se_diffs <- sapply(all_results, function(x) x$rmst_results[[sg]]$se_rmst_diff)

    pooled <- pool_variance(rmst_diffs, se_diffs, sprintf("rmst_%s", sg))
    if(is.null(pooled)) return(NULL)

    list(
      subgroup = sg,
      rmst_diff = pooled$coef,
      se = pooled$se,
      ci_lower = pooled$coef - qt(0.975, pooled$df) * pooled$se,
      ci_upper = pooled$coef + qt(0.975, pooled$df) * pooled$se,
      pvalue = 2 * pt(-abs(pooled$coef / pooled$se), df = pooled$df),
      tau = all_results[[1]]$rmst_results[[sg]]$tau,
      n_valid_imputations = pooled$n_valid
    )
  })
  names(pooled_rmst) <- subgroup_names
  pooled_rmst <- pooled_rmst[!sapply(pooled_rmst, is.null)]

  time_points <- c(365, 730, 1095)

  pooled_surv_probs <- lapply(subgroup_names, function(sg) {
    time_results <- lapply(seq_along(time_points), function(t_idx) {

      # Extract risk differences
      risk_diffs <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff else NA
      })

      risk_diff_vars <- sapply(all_results, function(x) {
        tp <- x$surv_prob_results[[sg]]$time_points
        if(!is.null(tp) && length(tp) >= t_idx) tp[[t_idx]]$risk_diff_var else NA
      })

      # Pool risk difference
      rd_ses <- sqrt(risk_diff_vars)
      pooled_rd <- pool_variance(risk_diffs, rd_ses,
                                  sprintf("risk_diff_%s_t%d", sg, time_points[t_idx]))
      if(is.null(pooled_rd)) return(NULL)

      # Pool baseline risk
      br_ses <- sqrt(baseline_risk_vars)
      pooled_br <- pool_variance(baseline_risks, br_ses,
                                  sprintf("baseline_risk_%s_t%d", sg, time_points[t_idx]))

      baseline_result <- if(!is.null(pooled_br)) {
        list(
          baseline_risk = pooled_br$coef,
          baseline_risk_se = pooled_br$se,
          baseline_risk_ci_lower = pooled_br$coef - qt(0.975, pooled_br$df) * pooled_br$se,
          baseline_risk_ci_upper = pooled_br$coef + qt(0.975, pooled_br$df) * pooled_br$se,
          baseline_risk_df = pooled_br$df
        )
      } else {
        list(
          baseline_risk = NA,
          baseline_risk_se = NA,
          baseline_risk_ci_lower = NA,
          baseline_risk_ci_upper = NA,
          baseline_risk_df = NA
        )
      }

      c(
        list(
          time = time_points[t_idx],
          risk_diff = pooled_rd$coef,
          se = pooled_rd$se,
          ci_lower = pooled_rd$coef - qt(0.975, pooled_rd$df) * pooled_rd$se,
          ci_upper = pooled_rd$coef + qt(0.975, pooled_rd$df) * pooled_rd$se,
          pvalue = 2 * pt(-abs(pooled_rd$coef / pooled_rd$se), df = pooled_rd$df),
          df = pooled_rd$df,
          within_var = pooled_rd$within_var,
          between_var = pooled_rd$between_var,
          fmi = (1 + 1/pooled_rd$n_valid) * pooled_rd$between_var /
                (pooled_rd$within_var + (1 + 1/pooled_rd$n_valid) * pooled_rd$between_var)
        ),
        baseline_result
      )
    })

    time_results <- time_results[!sapply(time_results, is.null)]
    list(subgroup = sg, time_results = time_results)
  })
  names(pooled_surv_probs) <- subgroup_names

  return(list(
    rmst_results = pooled_rmst,
    surv_prob_results = pooled_surv_probs
  ))
}

In [ ]:
# ==============================================================================
# Wrapper Function to Combine Parts
# ==============================================================================
pool_results <- function(all_results) {
  part1 <- pool_results_part1(all_results)
  part2 <- pool_results_part2(all_results)

  return(list(
    main_effects = part1$main_effects,
    subgroup_results = part1$subgroup_results,
    interaction_results = part1$interaction_results,
    rmst_results = part2$rmst_results,
    surv_prob_results = part2$surv_prob_results
  ))
}

In [ ]:
# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
subgroup_var <- "DcsiScoreGroup"
covariates <- c(
  "HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
  "HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
  "HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
  "HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
  "HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
  "HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
  "HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
  "HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
  "HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
  "HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
  "HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant",
  "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone", "HasOtherAntiHypertensive",
  "HasHospitalization", "Race", "Ethnicity", "CensusDivision",
  "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
  "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
  "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
  "NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
  "NumAntidiabeticMed", "ElixhauserComorbidityScore", "Sex", "Age", "CalendarYear",
  "AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HemoglobinValue",
  "LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
  "SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue", "HbA1cValue",
  "AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
  "AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
  "CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
  "HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)
covariates <- setdiff(covariates, subgroup_var)
treatment <- "FirstMedGroup"

start_time <- Sys.time()

# ==============================================================================
# LASSO
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 1: LASSO\n")
cat(rep("=", 80), "\n")

lasso_result <- perform_lasso_first_only(
  imp_data = imputed_t2d_list[[1]],
  covariates = covariates,
  subgroup_var = subgroup_var,
  treatment = treatment,
  sample_size = 50000
)

union_vars <- lasso_result$union_vars

# ==============================================================================
# Full Analysis
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 2: Full Analysis (All 5 Imputations - SEQUENTIAL)\n")
cat(rep("=", 80), "\n")

all_results <- list()

for(i in 1:5) {

  cat("\n", rep("-", 80), "\n")
  cat("Processing Imputation", i, "of 5\n")
  cat(rep("-", 80), "\n")

  data_list <- analyze_with_selected_vars(
    imp_data = imputed_t2d_list[[i]],
    imp_num = i,
    covariates = covariates,
    subgroup_var = subgroup_var,
    treatment = treatment,
    union_vars = union_vars,
    tau = 1095
  )

  cox_results <- complete_analysis_fast(
    data_list,
    imp_num = i,
    subgroup_var = subgroup_var,
    tau = 1095
  )

  rmst_results <- calculate_rmst_fast(
    cox_results,
    tau = 1095,
    B = 100,
    subsample_frac = 0.3,
    n_cores = 4
  )

  all_results[[i]] <- list(
    main_effects = cox_results$main_effects,
    subgroup_results = cox_results$subgroup_results,
    interaction_results = cox_results$interaction_results,
    data_by_sg = if(i == 1) cox_results$data_by_sg else NULL,
    rmst_results = rmst_results$rmst_results,
    surv_prob_results = rmst_results$surv_prob_results
  )

  cat("Imputation", i, "completed!\n")

  invisible(gc())
}

cat("\nAll imputations completed!\n")

# ==============================================================================
# Pooling
# ==============================================================================
cat("\n", rep("=", 80), "\n")
cat("PHASE 3: Pooling\n")
cat(rep("=", 80), "\n")

pooled_results <- pool_results(all_results)

end_time <- Sys.time()
elapsed <- as.numeric(end_time - start_time, units = "mins")

cat("\n", rep("=", 80), "\n")
cat(sprintf("Total Time: %.2f minutes (%.2f hours)\n", elapsed, elapsed/60))
cat(rep("=", 80), "\n")


In [ ]:
# ==============================================================================
# PRINT RESULTS
# ==============================================================================
cat("\n", rep("=", 120), "\n")
cat("FINAL POOLED RESULTS (3-Year Follow-up)\n")
cat(sprintf("Subgroup Analysis by: %s\n", subgroup_var))
cat(rep("=", 120), "\n")

# 1. Subgroup-Specific Results
cat("\n2. SUBGROUP-SPECIFIC TREATMENT EFFECTS (Hazard Ratios)\n")
cat(rep("-", 120), "\n")

for(sg_name in names(pooled_results$subgroup_results)) {
  sg <- pooled_results$subgroup_results[[sg_name]]
  cat(sprintf("Subgroup %s: HR = %.3f (95%% CI: %.3f-%.3f), p = %.4f\n",
              sg_name, sg$hr, sg$ci_lower, sg$ci_upper, sg$pvalue))
}

# 2. Interaction Test
cat("\n3. INTERACTION TEST (Treatment × Dcsi)\n")
cat(rep("-", 120), "\n")

if(length(pooled_results$interaction_results) > 0) {
  for(int in pooled_results$interaction_results) {
    cat(sprintf("%s: Coef = %.4f (SE = %.4f), p = %.4f\n",
                int$term, int$coef, int$se, int$pvalue))

    if(int$pvalue < 0.05) {
      cat("  → Significant heterogeneity on relative scale\n")
    } else {
      cat("  → No significant heterogeneity on relative scale\n")
    }
  }

  all_p <- sapply(pooled_results$interaction_results, function(x) x$pvalue)
  if(any(all_p < 0.05)) {
    cat("\nOVERALL: Significant treatment effect heterogeneity detected\n")
  } else {
    cat("\nOVERALL: No significant treatment effect heterogeneity\n")
  }
} else {
  cat("No interaction terms available\n")
}

# 3. RMST
cat("\n4. RESTRICTED MEAN SURVIVAL TIME (3-year RMST)\n")
cat(rep("-", 120), "\n")

for(sg_name in names(pooled_results$rmst_results)) {
  rmst <- pooled_results$rmst_results[[sg_name]]
  cat(sprintf("Subgroup %s: RMST Diff = %.1f days (95%% CI: %.1f-%.1f), p = %.4f\n",
              sg_name, rmst$rmst_diff, rmst$ci_lower, rmst$ci_upper, rmst$pvalue))
  cat(sprintf("  → Treatment extends by %.1f days of event-free survival\n", rmst$rmst_diff))
}

# 4. Absolute Risk Differences
cat("\n5. ABSOLUTE RISK DIFFERENCES\n")
cat(rep("-", 120), "\n\n")

for(sg_name in names(pooled_results$surv_prob_results)) {
  cat(sprintf("Subgroup %s:\n", sg_name))

  surv_res <- pooled_results$surv_prob_results[[sg_name]]$time_results

  for(tr in surv_res) {
    cat(sprintf("  At %.1f years: Risk Diff = %.4f (95%% CI: %.4f-%.4f)\n",
                tr$time / 365, tr$risk_diff, tr$ci_lower, tr$ci_upper))
    cat(sprintf("    SE = %.4f, p = %.4f (%.2f%% reduction)\n",
                tr$se, tr$pvalue, abs(tr$risk_diff) * 100))
    cat(sprintf("    FMI = %.3f (%.1f%% of variance from imputation uncertainty)\n",
                tr$fmi, tr$fmi * 100))
  }
  cat("\n")
}

cat("\n", rep("=", 120), "\n")
cat("Analysis Complete!\n")
cat(rep("=", 120), "\n")

In [ ]:
cat("\n", rep("=", 80), "\n")
cat("6. SUBGROUP SAMPLE STATISTICS\n")
cat(rep("=", 80), "\n")

cat("\n6-1. ORIGINAL DATA (Before Overlap Weighting)\n")
cat(rep("-", 80), "\n\n")

cat("A. Overall Dataset Statistics\n")
cat(rep("-", 40), "\n")

overall_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  total_n <- nrow(dt)

  age_counts <- dt[, .N, by = Si]

  list(
    total_n = total_n,
    age_counts = age_counts
  )
})

avg_total_n <- round(mean(sapply(overall_stats, function(x) x$total_n)))

age_group_levels <- unique(overall_stats[[1]]$age_counts$Si)

avg_age_counts <- lapply(age_group_levels, function(sg) {
  counts <- sapply(overall_stats, function(x) {
    x$age_counts[x$age_counts$Si == sg, ]$N
  })
  list(
    subgroup = as.character(sg),
    count = round(mean(counts))
  )
})

cat(sprintf("Total Sample Size: n = %d\n\n", avg_total_n))
cat("Age Group Distribution:\n")
for(ac in avg_age_counts) {
  cat(sprintf("  %s: n = %d (%.1f%%)\n",
              ac$subgroup, ac$count, ac$count/avg_total_n*100))
}
cat("\n")

cat("\nB. Statistics by Age Subgroup\n")
cat(rep("-", 40), "\n\n")

original_stats <- lapply(1:5, function(i) {
  dt <- as.data.table(imputed_t2d_list[[i]])
  dt[, Si := as.factor(get(subgroup_var))]
  dt[, Z := as.integer(get(treatment) == "glp")]

  stats_by_sg <- dt[, .(
    n_total = .N,
    n_treat = sum(Z == 1),
    n_ctrl = sum(Z == 0),
    events_total = sum(Outcome),
    events_treat = sum(Outcome[Z == 1]),
    events_ctrl = sum(Outcome[Z == 0])
  ), by = Si]

  setDF(stats_by_sg)
  stats_by_sg
})

avg_original_stats <- lapply(unique(original_stats[[1]]$Si), function(sg) {
  n_total_vec <- sapply(original_stats, function(x) x$n_total[x$Si == sg])
  n_treat_vec <- sapply(original_stats, function(x) x$n_treat[x$Si == sg])
  n_ctrl_vec <- sapply(original_stats, function(x) x$n_ctrl[x$Si == sg])
  events_total_vec <- sapply(original_stats, function(x) x$events_total[x$Si == sg])
  events_treat_vec <- sapply(original_stats, function(x) x$events_treat[x$Si == sg])
  events_ctrl_vec <- sapply(original_stats, function(x) x$events_ctrl[x$Si == sg])

  list(
    subgroup = as.character(sg),
    n_total = round(mean(n_total_vec)),
    n_treat = round(mean(n_treat_vec)),
    n_ctrl = round(mean(n_ctrl_vec)),
    events_total = round(mean(events_total_vec)),
    events_treat = round(mean(events_treat_vec)),
    events_ctrl = round(mean(events_ctrl_vec))
  )
})

for(stats in avg_original_stats) {
  cat(sprintf("Age Group: %s\n", stats$subgroup))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size:\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat("\n6-2. AFTER OVERLAP WEIGHTING (Pseudo Population)\n")
cat(rep("-", 80), "\n\n")

subgroup_levels <- names(all_results[[1]]$data_by_sg)

avg_weighted_stats <- lapply(subgroup_levels, function(sg) {

  stats_list <- lapply(1:5, function(i) {
    data_sg <- all_results[[i]]$data_by_sg[[sg]]

    if(is.null(data_sg)) return(NULL)

    n_total <- nrow(data_sg)
    n_events <- sum(data_sg$Outcome)
    ess_total <- sum(data_sg$w)^2 / sum(data_sg$w^2)

    data_treat <- data_sg[data_sg$Z == 1, ]
    data_ctrl <- data_sg[data_sg$Z == 0, ]

    n_treat <- nrow(data_treat)
    n_ctrl <- nrow(data_ctrl)
    events_treat <- sum(data_treat$Outcome)
    events_ctrl <- sum(data_ctrl$Outcome)
    ess_treat <- if(n_treat > 0) sum(data_treat$w)^2 / sum(data_treat$w^2) else 0
    ess_ctrl <- if(n_ctrl > 0) sum(data_ctrl$w)^2 / sum(data_ctrl$w^2) else 0

    list(
      n_total = n_total,
      n_treat = n_treat,
      n_ctrl = n_ctrl,
      events_total = n_events,
      events_treat = events_treat,
      events_ctrl = events_ctrl,
      ess_total = ess_total,
      ess_treat = ess_treat,
      ess_ctrl = ess_ctrl
    )
  })

  stats_list <- stats_list[!sapply(stats_list, is.null)]

  list(
    subgroup = sg,
    n_total = round(mean(sapply(stats_list, function(x) x$n_total))),
    n_treat = round(mean(sapply(stats_list, function(x) x$n_treat))),
    n_ctrl = round(mean(sapply(stats_list, function(x) x$n_ctrl))),
    events_total = round(mean(sapply(stats_list, function(x) x$events_total))),
    events_treat = round(mean(sapply(stats_list, function(x) x$events_treat))),
    events_ctrl = round(mean(sapply(stats_list, function(x) x$events_ctrl))),
    ess_total = round(mean(sapply(stats_list, function(x) x$ess_total)), 1),
    ess_treat = round(mean(sapply(stats_list, function(x) x$ess_treat)), 1),
    ess_ctrl = round(mean(sapply(stats_list, function(x) x$ess_ctrl)), 1)
  )
})
names(avg_weighted_stats) <- subgroup_levels

for(sg in names(avg_weighted_stats)) {
  stats <- avg_weighted_stats[[sg]]

  cat(sprintf("Age Group: %s\n", sg))
  cat(rep("-", 40), "\n")
  cat(sprintf("  Sample Size (Original):\n"))
  cat(sprintf("    Total:     n = %d\n", stats$n_total))
  cat(sprintf("    Treatment: n = %d (%.1f%%)\n",
              stats$n_treat, stats$n_treat/stats$n_total*100))
  cat(sprintf("    Control:   n = %d (%.1f%%)\n",
              stats$n_ctrl, stats$n_ctrl/stats$n_total*100))
  cat(sprintf("\n  Effective Sample Size (ESS):\n"))
  cat(sprintf("    Total:     ESS = %.1f\n", stats$ess_total))
  cat(sprintf("    Treatment: ESS = %.1f\n", stats$ess_treat))
  cat(sprintf("    Control:   ESS = %.1f\n", stats$ess_ctrl))
  cat(sprintf("\n  Events:\n"))
  cat(sprintf("    Total:     %d (%.1f%%)\n",
              stats$events_total, stats$events_total/stats$n_total*100))
  cat(sprintf("    Treatment: %d (%.1f%%)\n",
              stats$events_treat, stats$events_treat/stats$n_treat*100))
  cat(sprintf("    Control:   %d (%.1f%%)\n",
              stats$events_ctrl, stats$events_ctrl/stats$n_ctrl*100))
  cat("\n")
}

cat(rep("=", 80), "\n")
cat("Note:\n")
cat("  - Original n = Actual number of patients before weighting\n")
cat("  - ESS = Effective Sample Size after overlap weighting\n")
cat("  - ESS = (sum of weights)^2 / sum of squared weights\n")
cat("  - ESS represents the 'pseudo population' size for inference\n")
cat(rep("=", 80), "\n")

In [ ]:
data_by_sg <- all_results[[1]]$data_by_sg

data_any_complications <- data_by_sg[["+1"]]
data_c_treat <- data_any_complications[data_any_complications$Z == 1, ]
data_c_ctrl <- data_any_complications[data_any_complications$Z == 0, ]

fit_c_treat <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_c_treat, weights = w)
fit_c_ctrl <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_c_ctrl, weights = w)

data_no_complications <- data_by_sg[["0"]]
data_nc_treat <- data_no_complications[data_no_complications$Z == 1, ]
data_nc_ctrl <- data_no_complications[data_no_complications$Z == 0, ]

fit_nc_treat <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_nc_treat, weights = w)
fit_nc_ctrl <- survfit(Surv(Survival_Time, Outcome) ~ 1, data = data_nc_ctrl, weights = w)

image_path <- file.path(artifacts_images_dir, "t2d_secondary_primary_endpoint_dcsi_score.png")

draw_plot <- function() {
  par(mfrow = c(1, 1), mar = c(6, 7, 4, 2), mgp = c(5, 1, 0))

  plot(0, 0, type = "n", xlim = c(0, 36), ylim = c(0, 0.20),
       xlab = "Follow-up, mo", ylab = "Cumulative incidence, %",
       las = 1, cex.lab = 1.6, yaxt = "n", cex.axis = 1.5)

  axis(2, at = seq(0, 0.20, by = 0.05),
       labels = paste0(seq(0, 20, by = 5), "%"),
       las = 1, cex.axis = 1.5)

  abline(h = seq(0, 0.20, by = 0.05), col = "gray50", lwd = 0.5)

  lines(c(0, fit_c_treat$time / 30.44), c(0, 1 - fit_c_treat$surv),
        col = "#E41A1C", lty = 1, lwd = 2.5)
  lines(c(0, fit_c_ctrl$time / 30.44), c(0, 1 - fit_c_ctrl$surv),
        col = "#E41A1C", lty = 3, lwd = 2.5)
  lines(c(0, fit_nc_treat$time / 30.44), c(0, 1 - fit_nc_treat$surv),
        col = "#377EB8", lty = 1, lwd = 2.5)
  lines(c(0, fit_nc_ctrl$time / 30.44), c(0, 1 - fit_nc_ctrl$surv),
        col = "#377EB8", lty = 3, lwd = 2.5)

#   legend("topleft",
#          legend = c("Any complications - GLP-1 RAs", "Any complications - Non-GLP-1 RAs",
#                     "No complications - GLP-1 RAs", "No complications - Non-GLP-1 RAs"),
#          col = c("#E41A1C", "#E41A1C", "#377EB8", "#377EB8"),
#          lty = c(1, 3, 1, 3),
#          lwd = 2.5,
#          bty = "n",
#          cex = 1.4)
}

png(filename = image_path, width = 9.5, height = 7, units = "in", res = 300)
draw_plot()
dev.off()

cat("Plot saved to:", image_path, "\n")
draw_plot()